# CS-781 Phase 7 (Prior Case Retrieval) -- E3_partial_ft

Partial / layer-wise fine-tuning: top 6 blocks + final norm, layer-wise LR decay 0.85 (lower layers frozen). Every model is a shared-weight bi-encoder (query and candidate through the same Phase-1 trunk, mean-pooled chunk vectors, identity-initialised projection, learned temperature), trained with InfoNCE over in-batch + BM25 hard negatives (GradCache); the final ranking fuses the dense score with BM25 (weight tuned on Dev, >= 0.2).

**Accelerator: GPU T4 x2. Internet: ON if the data comes from the HF secret.** Expected runtime: <= 220 min (time budget 3.5 h; every job plans its epochs / data fraction / inference chunks from measured throughput and stops cleanly at its share of the budget). Two models train at the same time, one per GPU (no DDP).

Run it with **Save Version -> Save & Run All (Commit)** so the output persists.

### Inputs to attach (Add Input)
1. **The four Phase-1 datasets** (`01-train-gpt2style`, `02-train-llamastyle`,
   `03-train-mistral-sliding`, `04-train-bloom-alibi`: `final/model.pt` + `config.json`) --
   the SAME weights as every phase since Phase 2. Models are recognised by the `arch` field of
   `config.json`, so dataset names do not matter.
2. **The `bpe_model` dataset** (`bpe_model.json`; found anywhere under /kaggle/input).
3. **Data (one of):** (a) Add-ons -> Secrets -> `HF` (tick it for THIS notebook) **and**
   Settings -> Internet **On** -- the six Phase_7_PCR files are downloaded; or (b) a Kaggle
   dataset holding `Phase_7_PCR/{Train,Dev,Test}/{split}.json` + `{split}_candidates.json`
   (then Internet can stay Off: nothing else is downloaded in Phase 7).
4. *Optional:* the output of an earlier session of THIS notebook (finished models are skipped,
   an unfinished one resumes from `last.pt`). For **E6** you may also attach a finished
   **E2_full_ft** output: its M4 run is reused as the KD teacher (saves ~40% of E6).

### What a healthy run prints in the first minutes
* `all 25 modules verified`
* `[phase1] M1 gpt2style -> ...` (all four found), `[bpe] using ...`
* `[data] train: N queries ..., pool M candidates; ... 0 relevant ids not in the pool` (same for
  dev / test), `[cache] ...`, `[bm25] tuned on dev: ... Micro-F1@7 0.xxxx`, the `[audit]` line
* per job: `[check] ... pass` (4 lines), `[preflight] train ... s/step, encode ... tok/s`,
  `[zero-shot] ...`, `[plan] CHOSEN: epochs=... train_fraction=...`, then `[eval] epoch 0.50: dev
  proxy MAP ...` rising above the zero-shot value
* at the end of a job: `DONE dev Micro-F1@7 0.xxxx ... dense only ..., BM25 only ...; submission
  validated`


In [ ]:
import os, sys, json, hashlib, subprocess, glob, time, traceback
CODE = "/kaggle/working/phase7"; OUT = "/kaggle/working/phase7_out"
os.makedirs(CODE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print(sys.version.split()[0], time.strftime('%Y-%m-%d %H:%M:%S'))
try:
    import torch
    print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count(),
          [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
except Exception as e:
    print('torch not importable:', e)
print('inputs:', sorted(os.listdir('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'none')
EXPECTED_MD5 = {
 "bpe_tokenizer.py": "906d8a37d55bf0081dd5e8d7944c0b4e",
 "phase1_models.py": "1ae9785176b01f67a305e1504b0f7ae6",
 "ner_common.py": "020d134c97ddb4ef8fc97810501a107c",
 "ner_models.py": "6e3e619bf13a1248284f5d75184923e0",
 "ner_peft.py": "d418dc954f786d538ba288e4f34ca454",
 "cjpe_common.py": "b811813f00f7be904ae7f06b0b17c6de",
 "cjpe_data.py": "c6b24be5b0242d1ec5b49c31b775ea4a",
 "cjpe_tok.py": "8502c252f6cd039160ad2ac45d2fa55b",
 "cjpe_models.py": "275de149af23054f9dabc22804669279",
 "cjpe_peft.py": "1a2675371e02330c7de371609715686a",
 "cjpe_eval.py": "4786294a9c58dcb56e871ba01eff8ba9",
 "cjpe_kd.py": "96e8e80de8ddc909254ddc178a45b9c7",
 "lsi_ml.py": "693934d09f3df14d13586f1e1f03a2e1",
 "cjpe_predict.py": "ad3f7efab7c537387ed98b3a6215b79e",
 "cjpe_train.py": "e8c11b7d4edcbf2a0c1d274f43e41248",
 "summ_model.py": "a7073f4aafb82447c8044a7303cd1c38",
 "pcr_ml.py": "955c6f262e01eaa03a70a91d960337d0",
 "pcr_data.py": "bbaf244b1690dc43f77a4ccc620a265c",
 "pcr_model.py": "e57d135df61b2e19484acc8de1bf094b",
 "pcr_train.py": "670fe6b1ef34f8b56af9a71a93e6166a",
 "validate_submission.py": "c3e9a820b282233e92b9660537aadc72",
 "select_best.py": "adddee9083832272eac834a2fb3423a6",
 "aggregate.py": "b865a3f19dc3b8e5e6959cd2409cd628",
 "run_experiment.py": "6997db3715b7a5a5139130c7f6c91ad7",
 "selftest.py": "06e563e7b31a34fc6e53898e40d57b38"
}


In [ ]:
%%writefile /kaggle/working/phase7/bpe_tokenizer.py
"""
bpe_tokenizer.py -- from-scratch byte-level BPE (pure Python, stdlib + optional numpy/torch).

No `tokenizers`, `transformers`, `sentencepiece`, `tiktoken` or any other tokenizer library.
Only: re, heapq, json, collections, random, time, gc.

Token-id layout (matches the original notebook's special-token ids):
    0 <|endoftext|>   1 <|pad|>   2 <|unk|>   3..258 = raw bytes 0..255   259.. = merges
Internally training works with "internal ids" (byte b -> b, merge k -> 256+k);
the public API adds NUM_SPECIAL (=3) to every id.
"""
import gc
import heapq
import json
import random
import re
import time
from collections import Counter
from pathlib import Path

SPECIALS = ["<|endoftext|>", "<|pad|>", "<|unk|>"]
NUM_SPECIAL = len(SPECIALS)

# GPT-2 style pre-tokenisation, expressed with the stdlib `re` module
# (\p{L} -> [^\W\d_],  \p{N} -> \d,  punctuation/symbols -> [^\s\w] or '_').
PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)"      # English contractions
    r"| ?[^\W\d_]+"               # optional space + letters
    r"| ?\d+"                     # optional space + digits
    r"| ?(?:[^\s\w]|_)+"          # optional space + punctuation / symbols
    r"|\s+(?!\S)"                 # whitespace run not followed by a non-space
    r"|\s+"                       # remaining whitespace
)

MAX_PRETOKEN_CHARS = 512          # pathological runs (e.g. 50k dashes) are cut into pieces


# =============================================================================
# 1. TRAINING
# =============================================================================
def count_pretokens(texts, max_unique=6_000_000, max_word_chars=64, log_every=20000,
                    log=print):
    """Stream documents -> Counter of pre-tokens, with a hard memory ceiling.

    When the Counter grows past `max_unique`, singletons (then rarer and rarer words) are
    pruned, so RAM stays bounded no matter how large the corpus is.
    """
    counter = Counter()
    findall = PRETOKEN_RE.findall
    n_docs = n_chars = 0
    t0 = time.time()
    for text in texts:
        counter.update(findall(text))
        n_docs += 1
        n_chars += len(text)
        if len(counter) > max_unique:
            thr = 2
            while len(counter) > int(max_unique * 0.7):
                counter = Counter({k: v for k, v in counter.items() if v >= thr})
                thr += 1
            gc.collect()
            log(f"   [count] pruned to {len(counter):,} unique pre-tokens (min count {thr - 1})")
        if n_docs % log_every == 0:
            log(f"   [count] {n_docs:,} docs | {n_chars/1e6:,.0f}M chars | "
                f"{len(counter):,} unique | {time.time()-t0:.0f}s")
    # drop absurdly long pre-tokens (junk); they carry no useful merge statistics
    if max_word_chars:
        for k in [k for k in counter if len(k) > max_word_chars]:
            del counter[k]
    return counter


def train_bpe(word_counts, vocab_size, max_words=2_000_000, min_word_count=2,
              min_pair_count=2, n_verify=5000, seed=0, log_every=1000, log=print):
    """Learn BPE merges from a Counter of pre-tokens.

    Classic incremental algorithm: exact pair counts + inverted index (pair -> words that
    contain it) + lazy max-heap, so each merge only touches the words that contain the pair.

    Returns (merges, verify) where
        merges : list[(a, b)] in rank order, in INTERNAL ids (byte b -> b, merge k -> 256+k)
        verify : list[(pretoken_str, final_internal_ids)] used to cross-check the encoder
    """
    num_merges = vocab_size - NUM_SPECIAL - 256
    assert num_merges > 0, "vocab_size too small"

    # ---- 1. pick the training vocabulary of words ---------------------------------
    items = [(w, c) for w, c in word_counts.items() if c >= min_word_count]
    if len(items) > max_words:
        items.sort(key=lambda kv: kv[1], reverse=True)
        items = items[:max_words]
    words, freqs, strs = [], [], []
    for w, c in items:
        ids = list(w.encode("utf-8", "replace"))
        if len(ids) > 1:
            words.append(ids); freqs.append(c); strs.append(w)
    del items
    log(f"   [bpe] training on {len(words):,} unique words "
        f"({sum(freqs)/1e6:,.1f}M word occurrences)")

    # ---- 2. initial pair statistics ------------------------------------------------
    pc = {}        # pair -> total count
    where = {}     # pair -> set(word index)
    for idx, ids in enumerate(words):
        f = freqs[idx]
        for p in zip(ids, ids[1:]):
            if p in pc:
                pc[p] += f
                where[p].add(idx)
            else:
                pc[p] = f
                where[p] = {idx}
    heap = [(-c, p) for p, c in pc.items()]
    heapq.heapify(heap)

    merges = []
    t0 = time.time()
    next_id = 256
    while next_id < 256 + num_merges:
        # ---- pick the most frequent pair (lazy heap: skip stale entries) -----------
        best = None
        while heap:
            negc, p = heapq.heappop(heap)
            if pc.get(p, 0) == -negc:
                best = (-negc, p)
                break
        if best is None:
            log("   [bpe] no more pairs to merge -- stopping early")
            break
        count, (a, b) = best
        if count < min_pair_count:
            log(f"   [bpe] best pair count {count} < {min_pair_count} -- stopping early")
            break

        merges.append((a, b))
        new = next_id
        next_id += 1

        # ---- apply the merge to every word that contains it ------------------------
        idxs = where.pop((a, b), ())
        changed = {}                      # pair -> count before this merge
        for idx in idxs:
            ids = words[idx]
            n = len(ids)
            out = []
            i = 0
            hit = False
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new); i += 2; hit = True
                else:
                    out.append(ids[i]); i += 1
            if not hit:
                continue
            f = freqs[idx]
            old_pairs = list(zip(ids, ids[1:]))
            new_pairs = list(zip(out, out[1:]))
            for q in old_pairs:
                if q not in changed:
                    changed[q] = pc.get(q, 0)
                pc[q] -= f
            for q in new_pairs:
                if q not in changed:
                    changed[q] = pc.get(q, 0)
                pc[q] = pc.get(q, 0) + f
            new_set = set(new_pairs)
            for q in set(old_pairs) - new_set:
                s = where.get(q)
                if s is not None:
                    s.discard(idx)
            for q in new_set:
                s = where.get(q)
                if s is None:
                    where[q] = {idx}
                else:
                    s.add(idx)
            words[idx] = out

        # ---- publish the changed pair counts ---------------------------------------
        for q, orig in changed.items():
            c = pc.get(q, 0)
            if c <= 0:
                pc.pop(q, None)
                where.pop(q, None)
            elif c != orig:
                heapq.heappush(heap, (-c, q))
        # keep the heap from growing without bound
        if len(heap) > max(4_000_000, 8 * len(pc)):
            heap = [(-c, q) for q, c in pc.items()]
            heapq.heapify(heap)

        k = len(merges)
        if k % log_every == 0 or k == num_merges:
            log(f"   [bpe] merge {k:,}/{num_merges:,} | last pair count {count:,} | "
                f"{len(pc):,} live pairs | {time.time()-t0:.0f}s")

    # ---- 3. sample of final segmentations (to verify the encoder) ----------------
    rng = random.Random(seed)
    pick = rng.sample(range(len(words)), min(n_verify, len(words)))
    verify = [(strs[i], list(words[i])) for i in pick]
    del words, freqs, strs, pc, where, heap
    gc.collect()
    return merges, verify


# =============================================================================
# 2. TOKENIZER (encode / decode / save / load)
# =============================================================================
class BPETokenizer:
    """Byte-level BPE tokenizer with a small HuggingFace-like surface:
    tok(text, add_special_tokens=False)["input_ids"], tok.encode, tok.decode,
    len(tok), eos_token_id / pad_token_id, padding_side / truncation_side.
    """

    def __init__(self, merges, cache_max=2_000_000):
        self.merges = [tuple(m) for m in merges]
        self.vocab_size = NUM_SPECIAL + 256 + len(self.merges)
        self._merge_map = {m: 256 + r for r, m in enumerate(self.merges)}   # pair -> internal id
        # id -> bytes (internal ids)
        self._bytes = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            self._bytes.append(self._bytes[a] + self._bytes[b])
        self._special_bytes = [s.encode("utf-8") for s in SPECIALS]

        self._cache = {}
        self._cache_max = cache_max

        self.eos_token, self.bos_token = SPECIALS[0], SPECIALS[0]
        self.pad_token, self.unk_token = SPECIALS[1], SPECIALS[2]
        self.eos_token_id = self.bos_token_id = 0
        self.pad_token_id = 1
        self.unk_token_id = 2
        self.padding_side = "left"        # decoder-only batched generation
        self.truncation_side = "left"     # an over-long prompt loses its beginning

    # ------------------------------------------------------------------ basics
    def __len__(self):
        return self.vocab_size

    def get_vocab(self):
        v = {s: i for i, s in enumerate(SPECIALS)}
        for i, bs in enumerate(self._bytes):
            v[bs.decode("latin-1") + f"#{i}"] = i + NUM_SPECIAL   # unique readable key
        return v

    # ------------------------------------------------------------------ encoding
    def _bpe_word(self, b):
        """BPE-merge one pre-token given as bytes -> tuple of PUBLIC ids."""
        ids = list(b)
        mm = self._merge_map
        while len(ids) > 1:
            best = None
            bp = None
            for p in zip(ids, ids[1:]):
                m = mm.get(p)
                if m is not None and (best is None or m < best):
                    best, bp = m, p
            if best is None:
                break
            a, c = bp
            out = []
            i, n = 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == c:
                    out.append(best); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        return tuple(i + NUM_SPECIAL for i in ids)

    def _encode_pretoken(self, w):
        if len(w) > MAX_PRETOKEN_CHARS:
            res = ()
            for s in range(0, len(w), MAX_PRETOKEN_CHARS):
                res += self._bpe_word(w[s:s + MAX_PRETOKEN_CHARS].encode("utf-8", "replace"))
            return res
        return self._bpe_word(w.encode("utf-8", "replace"))

    def encode(self, text, add_special_tokens=False):
        cache = self._cache
        out = []
        ext = out.extend
        for w in PRETOKEN_RE.findall(text):
            t = cache.get(w)
            if t is None:
                t = self._encode_pretoken(w)
                if len(cache) >= self._cache_max:
                    cache.clear()                 # bounded RAM
                cache[w] = t
            ext(t)
        return out

    def __call__(self, text, add_special_tokens=False, padding=False, truncation=False,
                 max_length=None, return_tensors=None, **_ignored):
        single = isinstance(text, str)
        batch = [text] if single else list(text)
        enc = [self.encode(t) for t in batch]
        if truncation and max_length is not None:
            if self.truncation_side == "left":
                enc = [e[-max_length:] if len(e) > max_length else e for e in enc]
            else:
                enc = [e[:max_length] for e in enc]
        masks = [[1] * len(e) for e in enc]
        if padding:
            target = max_length if (padding == "max_length" and max_length) else max(len(e) for e in enc)
            for k, e in enumerate(enc):
                gap = target - len(e)
                if gap > 0:
                    if self.padding_side == "left":
                        enc[k] = [self.pad_token_id] * gap + e
                        masks[k] = [0] * gap + masks[k]
                    else:
                        enc[k] = e + [self.pad_token_id] * gap
                        masks[k] = masks[k] + [0] * gap
        if return_tensors in ("pt", "np"):
            if return_tensors == "pt":
                import torch
                mk = lambda x: torch.tensor(x, dtype=torch.long)
            else:
                import numpy as np
                mk = lambda x: np.asarray(x, dtype=np.int64)
            return {"input_ids": mk(enc), "attention_mask": mk(masks)}
        if single:
            return {"input_ids": enc[0], "attention_mask": masks[0]}
        return {"input_ids": enc, "attention_mask": masks}

    # ------------------------------------------------------------------ decoding
    def decode(self, ids, skip_special_tokens=False):
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        nb = len(self._bytes)
        for i in ids:
            i = int(i)
            if i < NUM_SPECIAL:
                if not skip_special_tokens and i >= 0:
                    parts.append(self._special_bytes[i])
            elif i - NUM_SPECIAL < nb:
                parts.append(self._bytes[i - NUM_SPECIAL])
        return b"".join(parts).decode("utf-8", errors="replace")

    def batch_decode(self, seqs, skip_special_tokens=False):
        return [self.decode(s, skip_special_tokens) for s in seqs]

    # ------------------------------------------------------------------ persistence
    def save(self, path):
        obj = {"type": "byte_level_bpe", "version": 1, "specials": SPECIALS,
               "num_special": NUM_SPECIAL, "vocab_size": self.vocab_size,
               "pretoken_regex": PRETOKEN_RE.pattern, "merges": self.merges}
        with open(path, "w") as f:
            json.dump(obj, f)

    def save_pretrained(self, out_dir):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        self.save(out_dir / "bpe_model.json")
        with open(out_dir / "tokenizer_config.json", "w") as f:
            json.dump({"eos_token": self.eos_token, "pad_token": self.pad_token,
                       "unk_token": self.unk_token, "eos_token_id": 0, "pad_token_id": 1,
                       "unk_token_id": 2, "vocab_size": self.vocab_size,
                       "padding_side": self.padding_side,
                       "truncation_side": self.truncation_side}, f, indent=2)

    @classmethod
    def load(cls, path, **kw):
        with open(path) as f:
            obj = json.load(f)
        assert obj.get("type") == "byte_level_bpe", "not a byte_level_bpe file"
        assert obj["pretoken_regex"] == PRETOKEN_RE.pattern, "pre-tokenizer regex mismatch"
        return cls(obj["merges"], **kw)

    @classmethod
    def from_pretrained(cls, directory, **kw):
        return cls.load(Path(directory) / "bpe_model.json", **kw)


In [ ]:
%%writefile /kaggle/working/phase7/phase1_models.py
"""
phase1_models.py -- the Phase-1 model code, copied VERBATIM from the Phase-1 training notebooks.

  * cell 13 of every notebook (identical in all four): shared core
    (CausalLMOutput, positions_from_mask, causal_allowed, build_attn_mask, init_weights,
     decoding helpers, CausalLMBase). The only change: its trailing print() was removed so
     that importing this module is silent.
  * cell 15, class definitions only (everything before `MODEL_CLASS, CONFIG_CLASS = ...`):
      - 01_train_gpt2style       -> GPT2StyleConfig, GPT2Attention, GELUMLP, GPT2Block, GPT2StyleLM
      - 02_train_llamastyle      -> LlamaStyleConfig, RMSNorm, RotaryEmbedding, GQAttention, SwiGLU,
                                    LlamaBlock, LlamaStyleLM
        (03_train_mistral_sliding defines the SAME classes; it differs only by a module-level
         SLIDING_WINDOW constant and one comment. The window comes from config.json, so one
         copy serves both M2 and M3.)
      - 04_train_bloom_alibi     -> BloomStyleConfig, alibi_slopes, alibi_bias, BloomAttention,
                                    BloomMLP, BloomBlock, BloomStyleLM

DO NOT EDIT the copied code: state_dict keys of the Phase-1 `model.pt` files depend on it.
Everything NER-specific lives in ner_models.py.
"""

# =============================================================================================
# notebook cell 13 (shared core)
# =============================================================================================
import math
from dataclasses import dataclass, asdict, field
from typing import Optional, List, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.lib.stride_tricks import sliding_window_view


@dataclass
class CausalLMOutput:
    loss: Optional[torch.Tensor] = None
    logits: Optional[torch.Tensor] = None
    past_key_values: Optional[list] = None      # per layer: (k, v), each (B, n_kv_heads, S, head_dim)


def positions_from_mask(attention_mask, past_len, T, B, device):
    """Position ids for the T new tokens. attention_mask (if given) covers ALL keys: (B, past_len+T)."""
    if attention_mask is None:
        return torch.arange(past_len, past_len + T, device=device).unsqueeze(0).expand(B, T)
    pos = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)
    return pos[:, -T:]


def causal_allowed(T, past_len, attention_mask, window, device):
    """Bool (B|1, 1, T, S): True where query t may attend key s. Always materialised."""
    S = past_len + T
    q = torch.arange(past_len, S, device=device)[:, None]      # absolute index of each query
    k = torch.arange(S, device=device)[None, :]
    allowed = k <= q                                             # causal
    if window is not None:
        allowed = allowed & (q - k < window)                     # sliding window: last `window` keys incl. self
    allowed = allowed[None, None]
    if attention_mask is not None:
        allowed = allowed & attention_mask[:, None, None, :].bool()   # drop padding keys
        allowed = allowed | (q == k)[None, None]                 # a pad query would see nothing -> NaN; let it see itself
    return allowed


def build_attn_mask(T, past_len, attention_mask, window, device):
    """Returns (mask, is_causal) for F.scaled_dot_product_attention."""
    S = past_len + T
    window_active = window is not None and window < S
    no_pad = attention_mask is None or bool(attention_mask.all())
    if no_pad and not window_active:
        if past_len == 0:
            return None, True        # fast fused causal kernel (training path)
        if T == 1:
            return None, False       # single cached query sees every key
    return causal_allowed(T, past_len, None if no_pad else attention_mask,
                          window if window_active else None, device), False


def init_weights(model, n_layer, std=0.02):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=std)
    for name, p in model.named_parameters():       # residual-branch output projections
        if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
            nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * n_layer))


# ------------------------------------------------------------------ decoding helpers
def _sample(logits, temperature, top_p, top_k):
    logits = logits / max(float(temperature), 1e-5)
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(int(top_k), logits.size(-1)), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p is not None and top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        probs = sl.softmax(-1)
        remove = probs.cumsum(-1) - probs > top_p           # keep the smallest set with mass >= top_p
        sl = sl.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(1, si, sl)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def _banned_ngram_tokens(row, n):
    """Tokens that would complete an n-gram already present in `row` (list of ints)."""
    if n < 2 or len(row) < n:
        return []
    arr = np.asarray(row)
    win = sliding_window_view(arr[:-1], n - 1)           # window i = arr[i:i+n-1], followed by arr[i+n-1]
    hit = np.nonzero((win == arr[-(n - 1):]).all(1))[0]
    return arr[hit + n - 1].tolist()


class CausalLMBase(nn.Module):
    """Loss + generation shared by all four architectures."""

    @staticmethod
    def lm_loss(logits, labels):
        return F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)).float(),
                               labels[:, 1:].reshape(-1), ignore_index=-100)

    def _process_logits(self, logits, seq, rows, repetition_penalty, no_repeat_ngram_size, suppress):
        if repetition_penalty and repetition_penalty != 1.0:
            sc = torch.gather(logits, 1, seq)
            sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
            logits = logits.scatter(1, seq, sc)
        if no_repeat_ngram_size and no_repeat_ngram_size > 1:
            for b, row in enumerate(rows):
                banned = _banned_ngram_tokens(row, no_repeat_ngram_size)
                if banned:
                    logits[b, banned] = float("-inf")
        if suppress:
            logits[:, list(suppress)] = float("-inf")
        return logits

    @torch.inference_mode()
    def generate(self, input_ids, attention_mask=None, max_new_tokens=256, do_sample=False,
                 temperature=1.0, top_p=1.0, top_k=0, repetition_penalty=1.0,
                 no_repeat_ngram_size=0, num_beams=1, suppress_token_ids=(),
                 stop_fn=None, stop_check_every=16, **_ignored):
        """Returns ONLY the new tokens, shape (B, n_new). stop_fn(new_tokens) -> True ends early."""
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        attention_mask = attention_mask.long()
        rows = [ids[m.bool()].tolist() for ids, m in zip(input_ids, attention_mask)]   # real tokens only
        if num_beams > 1:
            return self._beam_search(input_ids, attention_mask, rows, max_new_tokens, num_beams,
                                     repetition_penalty, no_repeat_ngram_size, suppress_token_ids,
                                     stop_fn, stop_check_every)
        out = self(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out.past_key_values, out.logits[:, -1].float()
        seq, new = input_ids, []
        for step in range(max_new_tokens):
            logits = self._process_logits(logits, seq, rows, repetition_penalty,
                                          no_repeat_ngram_size, suppress_token_ids)
            nxt = _sample(logits, temperature, top_p, top_k) if do_sample else logits.argmax(-1)
            new.append(nxt)
            seq = torch.cat([seq, nxt[:, None]], 1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(nxt)[:, None]], 1)
            for row, t in zip(rows, nxt.tolist()):
                row.append(t)
            if step == max_new_tokens - 1:
                break
            if stop_fn is not None and (step + 1) % stop_check_every == 0 and stop_fn(torch.stack(new, 1)):
                break
            out = self(nxt[:, None], attention_mask=attention_mask, past_key_values=past, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1].float()
        return torch.stack(new, 1)

    def _beam_search(self, input_ids, attention_mask, rows, max_new_tokens, K, repetition_penalty,
                     no_repeat_ngram_size, suppress, stop_fn, stop_check_every):
        B = input_ids.size(0)
        out = self(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past = [(k.repeat_interleave(K, 0), v.repeat_interleave(K, 0)) for k, v in out.past_key_values]
        logits = out.logits[:, -1].float().repeat_interleave(K, 0)
        seq = input_ids.repeat_interleave(K, 0)
        am = attention_mask.repeat_interleave(K, 0)
        rows = [list(r) for r in rows for _ in range(K)]
        scores = torch.zeros(B, K, device=input_ids.device)
        scores[:, 1:] = -1e9                       # all beams start identical: expand from beam 0 only
        gen = seq[:, :0]
        for step in range(max_new_tokens):
            logits = self._process_logits(logits, seq, rows, repetition_penalty, no_repeat_ngram_size, suppress)
            logp = logits.log_softmax(-1)
            V = logp.size(-1)
            cand = (scores.view(-1, 1) + logp).view(B, K * V)
            top, idx = cand.topk(K, dim=1)
            src = (torch.arange(B, device=idx.device)[:, None] * K + idx // V).view(-1)
            tok = (idx % V).view(-1)
            scores = top
            gen = torch.cat([gen[src], tok[:, None]], 1)
            seq = torch.cat([seq[src], tok[:, None]], 1)
            am = torch.cat([am[src], torch.ones_like(tok)[:, None]], 1)
            rows = [rows[i] + [t] for i, t in zip(src.tolist(), tok.tolist())]
            past = [(k[src], v[src]) for k, v in past]
            if step == max_new_tokens - 1:
                break
            if stop_fn is not None and (step + 1) % stop_check_every == 0 and stop_fn(gen):
                break
            out = self(tok[:, None], attention_mask=am, past_key_values=past, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1].float()
        best = scores.argmax(1)
        return gen.view(B, K, -1)[torch.arange(B, device=best.device), best]


# =============================================================================================
# 01_train_gpt2style, cell 15 (classes)
# =============================================================================================
@dataclass
class GPT2StyleConfig:
    vocab_size: int
    n_positions: int = 2048
    n_embd: int = 768
    n_layer: int = 14
    n_head: int = 12
    mlp_ratio: int = 4
    layer_norm_eps: float = 1e-5
    initializer_range: float = 0.02


class GPT2Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.n_embd % c.n_head == 0
        self.n_head, self.head_dim, self.d = c.n_head, c.n_embd // c.n_head, c.n_embd
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd)
        self.o_proj = nn.Linear(c.n_embd, c.n_embd)

    def forward(self, x, mask, is_causal, past=None):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).split(self.d, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)      # (B, H, T, hd)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal)
        y = y.transpose(1, 2).reshape(B, T, self.d)                        # merge heads
        return self.o_proj(y), (k, v)


class GELUMLP(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.up_proj = nn.Linear(d, hidden)
        self.down_proj = nn.Linear(hidden, d)

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class GPT2Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln_1 = nn.LayerNorm(c.n_embd, eps=c.layer_norm_eps)
        self.attn = GPT2Attention(c)
        self.ln_2 = nn.LayerNorm(c.n_embd, eps=c.layer_norm_eps)
        self.mlp = GELUMLP(c.n_embd, c.mlp_ratio * c.n_embd)

    def forward(self, x, mask, is_causal, past=None):
        a, present = self.attn(self.ln_1(x), mask, is_causal, past)       # pre-LN residual
        x = x + a
        x = x + self.mlp(self.ln_2(x))
        return x, present


class GPT2StyleLM(CausalLMBase):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.wte = nn.Embedding(c.vocab_size, c.n_embd)
        self.wpe = nn.Embedding(c.n_positions, c.n_embd)
        self.blocks = nn.ModuleList([GPT2Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd, eps=c.layer_norm_eps)
        self.lm_head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        init_weights(self, c.n_layer, c.initializer_range)
        self.lm_head.weight = self.wte.weight                              # tied input/output embeddings

    def get_input_embeddings(self):
        return self.wte

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None,
                use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        pos = positions_from_mask(attention_mask, past_len, T, B, input_ids.device)
        pos = pos.clamp(max=self.config.n_positions - 1)
        mask, is_causal = build_attn_mask(T, past_len, attention_mask, None, input_ids.device)
        x = self.wte(input_ids) + self.wpe(pos)
        presents = []
        for i, blk in enumerate(self.blocks):
            x, pr = blk(x, mask, is_causal, past_key_values[i] if past_key_values else None)
            presents.append(pr)
        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = self.lm_loss(logits, labels) if labels is not None else None
        return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents if use_cache else None)


# =============================================================================================
# 02_train_llamastyle, cell 15 (classes) -- also used for 03_train_mistral_sliding
# =============================================================================================
@dataclass
class LlamaStyleConfig:
    vocab_size: int
    max_position_embeddings: int = 2048
    hidden_size: int = 768
    intermediate_size: int = 3072
    num_hidden_layers: int = 12
    num_attention_heads: int = 12
    num_key_value_heads: int = 4
    rope_theta: float = 10000.0
    rms_norm_eps: float = 1e-5
    sliding_window: Optional[int] = None      # None = full causal attention
    initializer_range: float = 0.02


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, pos):                                   # pos: (B, T) int
        freqs = pos[..., None].float() * self.inv_freq        # (B, T, hd/2)
        emb = torch.cat([freqs, freqs], dim=-1)               # (B, T, hd): angle for dim i and i+hd/2
        return emb.cos()[:, None], emb.sin()[:, None]         # (B, 1, T, hd) -> broadcast over heads


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    return (x * cos + rotate_half(x) * sin).to(x.dtype)


class GQAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.hidden_size % c.num_attention_heads == 0
        assert c.num_attention_heads % c.num_key_value_heads == 0, "KV heads must divide query heads"
        self.h, self.kv = c.num_attention_heads, c.num_key_value_heads
        self.hd = c.hidden_size // c.num_attention_heads
        self.n_rep = self.h // self.kv
        self.q_proj = nn.Linear(c.hidden_size, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(c.hidden_size, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(c.hidden_size, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, c.hidden_size, bias=False)

    def forward(self, x, cos, sin, mask, is_causal, past=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)    # (B, 12, T, hd)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)   # (B,  4, T, hd)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)           # rotate Q and K only
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        present = (k, v)                                                  # cache stays at 4 heads
        if self.n_rep > 1:                                                # kv head j serves q heads j*3..j*3+2
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal)
        return self.o_proj(y.transpose(1, 2).reshape(B, T, self.h * self.hd)), present


class SwiGLU(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.gate_proj = nn.Linear(d, hidden, bias=False)
        self.up_proj = nn.Linear(d, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, d, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class LlamaBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.input_norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.attn = GQAttention(c)
        self.post_attn_norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.mlp = SwiGLU(c.hidden_size, c.intermediate_size)

    def forward(self, x, cos, sin, mask, is_causal, past=None):
        a, present = self.attn(self.input_norm(x), cos, sin, mask, is_causal, past)
        x = x + a
        x = x + self.mlp(self.post_attn_norm(x))
        return x, present


class LlamaStyleLM(CausalLMBase):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.embed_tokens = nn.Embedding(c.vocab_size, c.hidden_size)
        self.rotary = RotaryEmbedding(c.hidden_size // c.num_attention_heads, c.rope_theta)
        self.layers = nn.ModuleList([LlamaBlock(c) for _ in range(c.num_hidden_layers)])
        self.norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.lm_head = nn.Linear(c.hidden_size, c.vocab_size, bias=False)
        init_weights(self, c.num_hidden_layers, c.initializer_range)
        self.lm_head.weight = self.embed_tokens.weight

    def get_input_embeddings(self):
        return self.embed_tokens

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None,
                use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        pos = positions_from_mask(attention_mask, past_len, T, B, input_ids.device)
        cos, sin = self.rotary(pos)
        mask, is_causal = build_attn_mask(T, past_len, attention_mask,
                                          self.config.sliding_window, input_ids.device)
        x = self.embed_tokens(input_ids)
        presents = []
        for i, layer in enumerate(self.layers):
            x, pr = layer(x, cos, sin, mask, is_causal, past_key_values[i] if past_key_values else None)
            presents.append(pr)
        x = self.norm(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = self.lm_loss(logits, labels) if labels is not None else None
        return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents if use_cache else None)


# =============================================================================================
# 04_train_bloom_alibi, cell 15 (classes)
# =============================================================================================
@dataclass
class BloomStyleConfig:
    vocab_size: int
    hidden_size: int = 768
    n_layer: int = 14
    n_head: int = 12
    mlp_ratio: int = 4
    layer_norm_eps: float = 1e-5
    initializer_range: float = 0.02


def alibi_slopes(n_head):
    def pow2_slopes(n):
        start = 2.0 ** (-(2.0 ** -(math.log2(n) - 3)))
        return [start * start ** i for i in range(n)]
    if math.log2(n_head).is_integer():
        return pow2_slopes(n_head)
    closest = 2 ** math.floor(math.log2(n_head))
    return pow2_slopes(closest) + alibi_slopes(2 * closest)[0::2][: n_head - closest]


def alibi_bias(pos_q, pos_k, slopes, allowed):
    """pos_q (B,T), pos_k (B,S), slopes (H,), allowed bool (B|1,1,T,S) -> float bias (B,H,T,S)."""
    dist = (pos_q[:, None, :, None] - pos_k[:, None, None, :]).clamp(min=0).float()
    bias = -slopes.view(1, -1, 1, 1) * dist
    return bias.masked_fill(~allowed, float("-inf"))


class BloomAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.hidden_size % c.n_head == 0
        self.h, self.hd, self.d = c.n_head, c.hidden_size // c.n_head, c.hidden_size
        self.query_key_value = nn.Linear(c.hidden_size, 3 * c.hidden_size)
        self.o_proj = nn.Linear(c.hidden_size, c.hidden_size)

    def forward(self, x, bias, past=None):
        B, T, _ = x.shape
        q, k, v = self.query_key_value(x).split(self.d, dim=-1)
        q = q.view(B, T, self.h, self.hd).transpose(1, 2)
        k = k.view(B, T, self.h, self.hd).transpose(1, 2)
        v = v.view(B, T, self.h, self.hd).transpose(1, 2)
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.to(q.dtype))   # additive ALiBi bias
        return self.o_proj(y.transpose(1, 2).reshape(B, T, self.d)), (k, v)


class BloomMLP(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.up_proj = nn.Linear(d, hidden)
        self.down_proj = nn.Linear(hidden, d)

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class BloomBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.input_layernorm = nn.LayerNorm(c.hidden_size, eps=c.layer_norm_eps)
        self.self_attention = BloomAttention(c)
        self.post_attention_layernorm = nn.LayerNorm(c.hidden_size, eps=c.layer_norm_eps)
        self.mlp = BloomMLP(c.hidden_size, c.mlp_ratio * c.hidden_size)

    def forward(self, x, bias, past=None):
        a, present = self.self_attention(self.input_layernorm(x), bias, past)
        x = x + a
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x, present


class BloomStyleLM(CausalLMBase):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.word_embeddings = nn.Embedding(c.vocab_size, c.hidden_size)
        self.blocks = nn.ModuleList([BloomBlock(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.hidden_size, eps=c.layer_norm_eps)
        self.lm_head = nn.Linear(c.hidden_size, c.vocab_size, bias=False)
        self.register_buffer("slopes", torch.tensor(alibi_slopes(c.n_head), dtype=torch.float32),
                             persistent=False)
        init_weights(self, c.n_layer, c.initializer_range)
        self.lm_head.weight = self.word_embeddings.weight

    def get_input_embeddings(self):
        return self.word_embeddings

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None,
                use_cache=False, last_only=False):
        B, T = input_ids.shape
        dev = input_ids.device
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        S = past_len + T
        if attention_mask is None:                          # all rows identical -> batch dim 1
            pos_k = torch.arange(S, device=dev)[None]
            allowed = causal_allowed(T, past_len, None, None, dev)
        else:
            pos_k = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)
            allowed = causal_allowed(T, past_len, attention_mask, None, dev)
        bias = alibi_bias(pos_k[:, -T:], pos_k, self.slopes, allowed)    # computed once, shared by all layers
        x = self.word_embeddings(input_ids)                  # no position embedding, no embedding LayerNorm (see note)
        presents = []
        for i, blk in enumerate(self.blocks):
            x, pr = blk(x, bias, past_key_values[i] if past_key_values else None)
            presents.append(pr)
        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = self.lm_loss(logits, labels) if labels is not None else None
        return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents if use_cache else None)


# ---------------------------------------------------------------------------------------------
# Registry (added for Phase 2; not part of the copied code)
# ---------------------------------------------------------------------------------------------
MODEL_CLASSES = {
    "GPT2StyleLM": (GPT2StyleLM, GPT2StyleConfig),
    "LlamaStyleLM": (LlamaStyleLM, LlamaStyleConfig),
    "BloomStyleLM": (BloomStyleLM, BloomStyleConfig),
}


In [ ]:
%%writefile /kaggle/working/phase7/ner_common.py
"""
ner_common.py -- small torch-free helpers shared by every Phase-2 module.

  * registries: the four Phase-1 models (M1..M4) and the experiments (E1..E6, extras)
  * file discovery under /kaggle/input (or any root), atomic JSON writes
  * disk guard and output-size accounting (no Hugging Face Hub anywhere in this project)
"""
import glob
import hashlib
import json
import os
import shutil
import time

# ---------------------------------------------------------------------------------------------
# Registries
# ---------------------------------------------------------------------------------------------
# `arch` must equal the "arch" field the Phase-1 notebooks wrote into final/config.json.
MODELS = {
    "M1": {"arch": "gpt2style", "folder": "M1_gpt2style", "kind": "gpt2"},
    "M2": {"arch": "llamastyle", "folder": "M2_llamastyle", "kind": "llama"},
    "M3": {"arch": "mistral-sliding", "folder": "M3_mistral_sliding", "kind": "llama"},
    "M4": {"arch": "bloom-alibi", "folder": "M4_bloom_alibi", "kind": "bloom"},
}
DEFAULT_MODEL_ORDER = ["M4", "M2", "M3", "M1"]          # strongest Phase-1 LM first

# Layer subsets for the layer-pruned KD students (every second layer; section 8 of the brief).
STUDENT_LAYERS = {
    "M1": [0, 2, 4, 6, 8, 10, 12],      # 7 of 14
    "M2": [0, 2, 4, 6, 8, 10],          # 6 of 12
    "M3": [0, 2, 4, 6, 8, 10],          # 6 of 12
    "M4": [0, 2, 4, 6, 8, 10, 12],      # 7 of 14
}

# The six required experiments + extras. `technique` selects the code path in ner_peft.py.
EXPERIMENTS = {
    "E1": {"folder": "E1_full_ft", "technique": "full_ft", "title": "Full fine-tuning"},
    "E2": {"folder": "E2_lorma", "technique": "lorma", "title": "LoRMA+ (compulsory)"},
    "E3": {"folder": "E3_lora", "technique": "lora", "title": "LoRA"},
    "E4": {"folder": "E4_adapters", "technique": "adapters", "title": "Houlsby adapters"},
    "E5": {"folder": "E5_partial_ft", "technique": "partial_ft", "title": "Partial / layer-wise FT"},
    "E6": {"folder": "E6_kd", "technique": "kd", "title": "Knowledge distillation"},
    # extras (not counted among the six)
    "X1": {"folder": "X1_bitfit", "technique": "bitfit", "title": "BitFit (M1/M4 only)",
           "models": ["M4", "M1"]},
    "X2": {"folder": "X2_student_no_kd", "technique": "student_hard",
           "title": "Pruned student, hard labels only (KD control)"},
}
REQUIRED_EXPS = ["E1", "E2", "E3", "E4", "E5", "E6"]

# Default learning rates (section 9.3). The LR probe multiplies `lr` by {1/3, 1, 3}.
DEFAULT_LR = {
    "full_ft": 5e-5, "lorma": 1e-3, "lora": 1e-3, "adapters": 1e-3, "bitfit": 2e-3,
    "partial_ft": 1e-4, "kd": 1e-4, "student_hard": 1e-4,
}
DEFAULT_EPOCHS = {
    "full_ft": 6, "kd": 6, "student_hard": 6, "partial_ft": 8,
    "lorma": 10, "lora": 10, "adapters": 10, "bitfit": 10,
}


# ---------------------------------------------------------------------------------------------
# JSON / files
# ---------------------------------------------------------------------------------------------
def write_json(path, obj, indent=2):
    """Atomic JSON write: a crash never leaves a half-written file behind."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = f"{path}.tmp{os.getpid()}"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=indent, ensure_ascii=False, default=_json_default)
    os.replace(tmp, path)


def _json_default(o):
    try:
        import numpy as np
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
    except ImportError:
        pass
    raise TypeError(f"not JSON serialisable: {type(o)}")


def read_json(path, default=None):
    if not os.path.exists(path):
        return default
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def find_files(name, roots, max_hits=50):
    """All files called `name` below any of `roots` (recursive), sorted for determinism."""
    hits = []
    for r in roots:
        if r and os.path.isdir(r):
            hits.extend(glob.glob(os.path.join(r, "**", name), recursive=True))
    return sorted(set(hits))[:max_hits]


def default_input_roots():
    """Where attached Kaggle datasets live; PHASE2_INPUT_ROOTS (os.pathsep-separated) overrides."""
    env = os.environ.get("PHASE2_INPUT_ROOTS")
    if env:
        return [p for p in env.split(os.pathsep) if p]
    return ["/kaggle/input"]


def iter_finished_runs(out_root, input_roots=()):
    """Yield (exp_folder, model_folder, run_dir) for every finished run (DONE.json):
    directly under `out_root` (<exp>/<model>/DONE.json, whatever out_root is called) and under
    any `phase2_out` folder inside the attached input datasets. First occurrence wins."""
    seen = set()
    out_root = os.path.abspath(out_root) if out_root else None
    if out_root and os.path.isdir(out_root):
        for d in sorted(glob.glob(os.path.join(out_root, "*", "*", "DONE.json"))):
            exp_f, mod_f = d.split(os.sep)[-3], d.split(os.sep)[-2]
            if mod_f != "BEST" and (exp_f, mod_f) not in seen:
                seen.add((exp_f, mod_f))
                yield exp_f, mod_f, os.path.dirname(d)
    for d in find_files("DONE.json", list(input_roots), max_hits=5000):
        parts = d.split(os.sep)
        if "phase2_out" not in parts:
            continue
        i = len(parts) - 1 - parts[::-1].index("phase2_out")
        if len(parts) - i != 4 or parts[-2] == "BEST":
            continue
        if (parts[i + 1], parts[-2]) not in seen:
            seen.add((parts[i + 1], parts[-2]))
            yield parts[i + 1], parts[-2], os.path.dirname(d)


def find_phase1_models(roots=None):
    """Map M1..M4 -> directory holding that model's Phase-1 `model.pt` + `config.json`.

    Identified by the "arch" field the Phase-1 notebooks wrote into config.json, so the
    dataset/folder names do not matter.
    """
    roots = roots or default_input_roots()
    by_arch = {}
    for cfg_path in find_files("config.json", roots, max_hits=10000):
        d = os.path.dirname(cfg_path)
        if not os.path.exists(os.path.join(d, "model.pt")):
            continue
        try:
            cfg = read_json(cfg_path)
        except Exception:
            continue
        if isinstance(cfg, dict) and "arch" in cfg and "model_class" in cfg:
            by_arch.setdefault(cfg["arch"], []).append(d)
    out = {}
    for mid, m in MODELS.items():
        cands = by_arch.get(m["arch"], [])
        if cands:
            # prefer a directory literally called "final" (what the Phase-1 notebooks saved)
            cands = sorted(cands, key=lambda p: (os.path.basename(p) != "final", p))
            out[mid] = cands[0]
    return out


# ---------------------------------------------------------------------------------------------
# Disk guard (section 9.5)
# ---------------------------------------------------------------------------------------------
def free_gb(path="/kaggle/working"):
    p = path
    while p and not os.path.exists(p):
        p = os.path.dirname(p)
    return shutil.disk_usage(p or "/").free / 1e9


def dir_size_gb(path):
    tot = 0
    for dp, _dn, fns in os.walk(path):
        for fn in fns:
            fp = os.path.join(dp, fn)
            if not os.path.islink(fp):
                try:
                    tot += os.path.getsize(fp)
                except OSError:
                    pass
    return tot / 1e9


class DiskFull(RuntimeError):
    pass


def disk_guard(out_root, min_free_gb=4.0, log=print, keep=None):
    """Make sure at least `min_free_gb` is free under `out_root`.

    Deletion order: leftover last.pt / last.tmp of models that are finished (DONE.json) or not
    the one currently running (`keep` = its checkpoint dir), then tmp files. Never touches
    best.pt, submissions or reports. Raises DiskFull if still short (callers stop cleanly).
    """
    if free_gb(out_root) >= min_free_gb:
        return
    removed = []
    for pat in ("**/checkpoints/last.pt", "**/checkpoints/last.tmp", "**/*.tmp*"):
        for f in glob.glob(os.path.join(out_root, pat), recursive=True):
            if keep and os.path.abspath(f).startswith(os.path.abspath(keep)):
                continue
            try:
                sz = os.path.getsize(f) / 1e9
                os.remove(f)
                removed.append((f, sz))
            except OSError:
                pass
        if free_gb(out_root) >= min_free_gb:
            break
    for f, sz in removed:
        log(f"   [disk] removed {f} ({sz:.2f} GB)")
    if free_gb(out_root) < min_free_gb:
        raise DiskFull(f"only {free_gb(out_root):.1f} GB free under {out_root} "
                       f"(need {min_free_gb} GB) even after cleanup")


# ---------------------------------------------------------------------------------------------
# misc
# ---------------------------------------------------------------------------------------------
def now():
    return time.time()


def fmt_h(seconds):
    return f"{seconds / 3600:.2f} h"


def log_line(path, msg, also_print=True):
    """Append a timestamped line to a text log (and print it)."""
    line = time.strftime("%Y-%m-%d %H:%M:%S ") + msg
    if also_print:
        print(line, flush=True)
    if path:
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        with open(path, "a", encoding="utf-8") as f:
            f.write(line + "\n")


In [ ]:
%%writefile /kaggle/working/phase7/ner_models.py
"""
ner_models.py -- Phase-1 trunk wrappers + NER heads.

The Phase-1 models' forward() only returns LM logits and is always causal. For NER we need the
final hidden states, optionally with bidirectional attention. `Trunk` re-implements only the
tiny "embed -> blocks -> final norm" loop and REUSES the original submodule objects (no copies;
lm_head is dropped, which also avoids a (B, T, 32000) logits tensor).

Attention modes (section 5.4):
  causal : exactly the Phase-1 masks (verified equal to the original forward in tests)
  bidir  : M1/M2 key-padding mask only; M3 symmetric sliding window |q-k| < window;
           M4 ALiBi with symmetric distance -slope*|q-k|. Padding is on the right.
"""
import json
import os

import torch
import torch.nn as nn
import torch.nn.functional as F

from ner_common import MODELS
from phase1_models import (MODEL_CLASSES, RMSNorm, alibi_bias, build_attn_mask, causal_allowed,
                           positions_from_mask)

MAX_POSITIONS = 1024        # every Phase-1 model was trained on 1024-token blocks


def torch_load(path, map_location="cpu"):
    """torch.load that works on old and new torch (weights_only appeared in 1.13)."""
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)
    except Exception:
        # weights_only=True refuses non-tensor pickles (our resumable checkpoints)
        try:
            return torch.load(path, map_location=map_location, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=map_location)


def load_phase1_lm(phase1_dir):
    """Rebuild a Phase-1 model from final/{config.json, model.pt}; strict key match."""
    with open(os.path.join(phase1_dir, "config.json")) as f:
        cfg = json.load(f)
    cls, cfg_cls = MODEL_CLASSES[cfg["model_class"]]
    lm = cls(cfg_cls(**cfg["config"]))
    sd = torch_load(os.path.join(phase1_dir, "model.pt"))
    lm.load_state_dict(sd, strict=True)
    return lm, cfg


def kind_of(lm):
    name = type(lm).__name__
    return {"GPT2StyleLM": "gpt2", "LlamaStyleLM": "llama", "BloomStyleLM": "bloom"}[name]


class Trunk(nn.Module):
    """embed -> selected blocks -> final norm, reusing the Phase-1 submodules."""

    def __init__(self, lm, attn_mode="bidir", layer_ids=None):
        super().__init__()
        assert attn_mode in ("causal", "bidir")
        self.kind = kind_of(lm)
        self.attn_mode = attn_mode
        c = lm.config
        self.window = None
        if self.kind == "gpt2":
            self.embed, self.wpe, blocks, self.norm = lm.wte, lm.wpe, lm.blocks, lm.ln_f
            self.n_positions = c.n_positions
            self.hidden_size = c.n_embd
        elif self.kind == "llama":
            self.embed, self.rotary, blocks, self.norm = lm.embed_tokens, lm.rotary, lm.layers, lm.norm
            self.window = c.sliding_window
            self.hidden_size = c.hidden_size
        else:
            self.embed, blocks, self.norm = lm.word_embeddings, lm.blocks, lm.ln_f
            self.register_buffer("slopes", lm.slopes.detach().clone(), persistent=False)
            self.hidden_size = c.hidden_size
        self.n_layers_original = len(blocks)
        self.layer_ids = list(range(len(blocks))) if layer_ids is None else list(layer_ids)
        assert all(0 <= i < len(blocks) for i in self.layer_ids)
        self.blocks = nn.ModuleList([blocks[i] for i in self.layer_ids])

    # ---------------------------------------------------------------- masks
    def _sdpa_mask(self, T, attention_mask, dev):
        """(mask, is_causal) for M1/M2/M3 blocks."""
        if self.attn_mode == "causal":
            return build_attn_mask(T, 0, attention_mask, self.window, dev)
        no_pad = bool(attention_mask.all())
        win = self.window if (self.window is not None and self.window < T) else None
        if no_pad and win is None:
            return None, False
        q = torch.arange(T, device=dev)[:, None]
        k = torch.arange(T, device=dev)[None, :]
        allowed = torch.ones(1, 1, T, T, dtype=torch.bool, device=dev)
        if win is not None:
            allowed = allowed & ((q - k).abs() < win)[None, None]
        if not no_pad:
            allowed = allowed & attention_mask[:, None, None, :].bool()
            allowed = allowed | (q == k)[None, None]          # no all-masked rows (pad queries)
        B = attention_mask.size(0)
        return allowed.expand(B, 1, T, T).contiguous(), False

    def _alibi(self, T, attention_mask, dev):
        pos_k = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)
        if self.attn_mode == "causal":                          # identical to BloomStyleLM.forward
            allowed = causal_allowed(T, 0, attention_mask, None, dev)
            return alibi_bias(pos_k[:, -T:], pos_k, self.slopes, allowed)
        dist = (pos_k[:, None, :, None] - pos_k[:, None, None, :]).abs().float()
        bias = -self.slopes.view(1, -1, 1, 1) * dist
        q = torch.arange(T, device=dev)[:, None]
        k = torch.arange(T, device=dev)[None, :]
        allowed = attention_mask[:, None, None, :].bool() | (q == k)[None, None]
        return bias.masked_fill(~allowed, float("-inf"))

    # ---------------------------------------------------------------- forward
    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        assert T <= MAX_POSITIONS, f"sequence of {T} tokens > {MAX_POSITIONS} (Phase-1 context)"
        dev = input_ids.device
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        pos = positions_from_mask(attention_mask, 0, T, B, dev)
        if self.kind == "gpt2":
            pos = pos.clamp(max=self.n_positions - 1)
            mask, is_causal = self._sdpa_mask(T, attention_mask, dev)
            x = self.embed(input_ids) + self.wpe(pos)
            for blk in self.blocks:
                x, _ = blk(x, mask, is_causal, None)
        elif self.kind == "llama":
            cos, sin = self.rotary(pos)
            mask, is_causal = self._sdpa_mask(T, attention_mask, dev)
            x = self.embed(input_ids)
            for blk in self.blocks:
                x, _ = blk(x, cos, sin, mask, is_causal, None)
        else:
            bias = self._alibi(T, attention_mask, dev)
            x = self.embed(input_ids)
            for blk in self.blocks:
                x, _ = blk(x, bias, None)
        return self.norm(x)

    def norm_modules(self):
        return [m for m in self.modules() if isinstance(m, (nn.LayerNorm, RMSNorm))]


class NERModel(nn.Module):
    """Trunk + dropout + Linear(H, C) [+ CRF]."""

    def __init__(self, trunk, n_labels, head="linear", dropout=0.1):
        super().__init__()
        self.trunk = trunk
        self.n_labels = n_labels
        self.num_tags = 1 + 2 * n_labels
        self.head_type = head
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(trunk.hidden_size, self.num_tags)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
        self.crf = None
        if head == "crf":
            from ner_crf import CRF
            self.crf = CRF(n_labels)

    def forward(self, input_ids, attention_mask=None, labels=None, class_weights=None):
        h = self.trunk(input_ids, attention_mask)
        logits = self.classifier(self.dropout(h)).float()
        loss = None
        if labels is not None:
            if self.crf is None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                                       weight=class_weights, ignore_index=-100)
            else:                                   # drop the EOS position, mask = real labels
                lab = labels[:, 1:]
                loss = self.crf.neg_log_likelihood(logits[:, 1:], lab, lab != -100)
        return logits, loss


def build_ner_model(phase1_dir, n_labels, attn_mode="bidir", head="linear", layer_ids=None,
                    dropout=0.1):
    lm, cfg = load_phase1_lm(phase1_dir)
    trunk = Trunk(lm, attn_mode, layer_ids)
    del lm                                          # unused blocks / lm_head are freed here
    return NERModel(trunk, n_labels, head, dropout), cfg


def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only))


# ---------------------------------------------------------------------------------------------
# trained-model reconstruction (teacher for E6, prediction, ensembles)
# ---------------------------------------------------------------------------------------------
def trainable_state_dict(model, half=False):
    sd = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
    if half:
        sd = {n: (t.half() if t.is_floating_point() else t) for n, t in sd.items()}
    return sd


def load_trainable_state(model, sd):
    """Load a trainable-only state dict; every key must exist and match a trainable param."""
    params = dict(model.named_parameters())
    missing = [n for n in sd if n not in params]
    if missing:
        raise KeyError(f"checkpoint has unknown parameters: {missing[:5]}")
    with torch.no_grad():
        for n, t in sd.items():
            params[n].copy_(t.to(params[n].dtype))
    expected = {n for n, p in params.items() if p.requires_grad}
    absent = expected - set(sd)
    if absent:
        raise KeyError(f"checkpoint lacks trainable parameters: {sorted(absent)[:5]}")


def rebuild_trained_model(run_config, best_pt, phase1_dirs, device="cpu"):
    """Recreate the exact module structure of a finished run and load its best.pt."""
    from ner_peft import apply_technique
    mid = run_config["model"]
    p1 = phase1_dirs.get(mid) or run_config.get("phase1_dir")
    if not p1 or not os.path.exists(os.path.join(p1, "model.pt")):
        raise FileNotFoundError(f"Phase-1 weights for {mid} ({MODELS[mid]['arch']}) not found")
    model, _ = build_ner_model(p1, run_config["n_labels"], run_config["attn_mode"],
                               run_config["head"], run_config.get("layer_ids"))
    apply_technique(model, run_config["technique"], run_config, log=lambda *a: None)
    load_trainable_state(model, torch_load(best_pt))
    return model.to(device).eval()


In [ ]:
%%writefile /kaggle/working/phase7/ner_peft.py
"""
ner_peft.py -- fine-tuning / PEFT techniques, implemented from scratch (no `peft` library).

  full_ft      every trunk parameter + head            (E1; also KD students E6 / X2)
  lorma        LoRMA+  h = (I + s*B*A) W0 x  (+ bias)   (E2, compulsory)
  lora         LoRA    h = W0 x + s*B*A x   (+ bias)    (E3)
  adapters     Houlsby bottleneck after attention-out and MLP-out, + norms (E4)
  partial_ft   top-6 blocks + final norm + head, layer-wise LR decay (E5)
  bitfit       bias terms only (M1/M4)                  (extra X1)

`apply_technique(model, technique, cfg)` freezes everything, inserts the modules, unfreezes
exactly what the method trains, ASSERTS that the set of trainable parameter names equals the
expected set, and returns optimizer parameter groups (no weight decay on 1-D tensors).

LoRMA reference: Bihany, Patel & Modi (2025), "LoRMA: Low-Rank Multiplicative Adaptation for
LLMs", arXiv:2506.07621. We implement the additive-identity rank inflation (LoRMA+): the
frozen output y0 = W0 x is multiplied by (I + s*B*A) with B (d_out x r) = 0 at init, so the
layer starts as the exact frozen layer. The d_out x d_out product is never materialised:
y = y0 + s * (y0 @ A^T) @ B^T. For fused QKV projections each of q, k, v gets its own B, A.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from phase1_models import RMSNorm

# Linear layers adapted by LoRA / LoRMA, per architecture family (paths inside each block).
TARGETS = {
    "gpt2": ["attn.qkv", "attn.o_proj", "mlp.up_proj", "mlp.down_proj"],
    "llama": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj",
              "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"],
    "bloom": ["self_attention.query_key_value", "self_attention.o_proj",
              "mlp.up_proj", "mlp.down_proj"],
}
FUSED_QKV = {"gpt2": "attn.qkv", "bloom": "self_attention.query_key_value"}
ADAPTER_SITES = {
    "gpt2": ["attn.o_proj", "mlp.down_proj"],
    "llama": ["attn.o_proj", "mlp.down_proj"],
    "bloom": ["self_attention.o_proj", "mlp.down_proj"],
}


def _get(mod, path):
    for p in path.split("."):
        mod = getattr(mod, p)
    return mod


def _set(mod, path, new):
    parts = path.split(".")
    for p in parts[:-1]:
        mod = getattr(mod, p)
    setattr(mod, parts[-1], new)


# =============================================================================================
# modules
# =============================================================================================
class LoRALinear(nn.Module):
    """y = W0 x + b + (alpha/r) * B(A(dropout(x)))."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(self.drop(x), self.lora_A.to(x.dtype)),
                                         self.lora_B.to(x.dtype))

    @torch.no_grad()
    def merged_weight(self):
        return self.base.weight + self.scale * (self.lora_B @ self.lora_A).to(self.base.weight.dtype)


class LoRMALinear(nn.Module):
    """LoRMA+ : y = (I + s*B*A) (W0 x) + b, applied separately to each output slice."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, slices=None):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        d_out = base.out_features
        self.slices = slices or [(0, d_out)]
        assert self.slices[0][0] == 0 and self.slices[-1][1] == d_out
        assert all(a[1] == b[0] for a, b in zip(self.slices, self.slices[1:])), "slices must tile"
        self.lorma_A = nn.ParameterList()
        self.lorma_B = nn.ParameterList()
        for (s, e) in self.slices:
            A = nn.Parameter(torch.empty(r, e - s))
            nn.init.kaiming_uniform_(A, a=math.sqrt(5))
            self.lorma_A.append(A)
            self.lorma_B.append(nn.Parameter(torch.zeros(e - s, r)))

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)                  # frozen W0 x, bias added after
        parts = []
        for (s, e), A, B in zip(self.slices, self.lorma_A, self.lorma_B):
            ys = y0[..., s:e]
            parts.append(ys + self.scale * ((ys @ A.to(ys.dtype).t()) @ B.to(ys.dtype).t()))
        y = parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)
        if self.base.bias is not None:
            y = y + self.base.bias
        return y

    @torch.no_grad()
    def merged_weight(self):
        W0 = self.base.weight
        W = W0.clone()
        for (s, e), A, B in zip(self.slices, self.lorma_A, self.lorma_B):
            W[s:e] = W0[s:e] + self.scale * (B @ (A @ W0[s:e]))
        return W


class Adapter(nn.Module):
    """Houlsby bottleneck: h + up(gelu(down(h))); up ~ N(0, 1e-3) so it starts near identity."""

    def __init__(self, d, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.down.bias)
        nn.init.normal_(self.up.weight, std=1e-3)
        nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class AdapterWrapped(nn.Module):
    """Wraps a sub-layer's output projection: adapter(base(x)), i.e. BEFORE the residual add."""

    def __init__(self, base: nn.Linear, bottleneck=64):
        super().__init__()
        self.base = base
        self.adapter = Adapter(base.out_features, bottleneck)

    def forward(self, x):
        return self.adapter(self.base(x))


# =============================================================================================
# applying a technique
# =============================================================================================
def _groups(named, lr, wd, tag):
    dec = [p for n, p in named if p.ndim >= 2]
    nodec = [p for n, p in named if p.ndim < 2]
    out = []
    if dec:
        out.append({"params": dec, "lr": lr, "weight_decay": wd, "name": f"{tag}/decay"})
    if nodec:
        out.append({"params": nodec, "lr": lr, "weight_decay": 0.0, "name": f"{tag}/no_decay"})
    return out


def apply_technique(model, technique, cfg, log=print):
    """Mutates `model` in place; returns optimizer param groups."""
    trunk = model.trunk
    kind = trunk.kind
    lr = float(cfg.get("lr", 1e-3))
    head_lr = float(cfg.get("head_lr", 1e-3))
    wd = float(cfg.get("weight_decay", 0.01))
    for p in model.parameters():
        p.requires_grad = False

    head_named = [(n, p) for n, p in model.named_parameters()
                  if n.startswith("classifier.") or n.startswith("crf.")]
    expected = {n for n, _ in head_named}
    groups = []

    if technique in ("full_ft", "kd", "student_hard"):
        named = [(n, p) for n, p in model.named_parameters() if n.startswith("trunk.")]
        expected |= {n for n, _ in named}
        for _, p in named:
            p.requires_grad = True
        groups += _groups(named, lr, wd, "trunk")

    elif technique in ("lora", "lorma"):
        r = int(cfg.get(f"{technique}_r", 16))
        alpha = float(cfg.get(f"{technique}_alpha", 32))
        for bi, blk in enumerate(trunk.blocks):
            for path in TARGETS[kind]:
                base = _get(blk, path)
                assert isinstance(base, nn.Linear), f"{path} is {type(base)}"
                if technique == "lora":
                    new = LoRALinear(base, r, alpha, float(cfg.get("lora_dropout", 0.05)))
                else:
                    slices = None
                    if FUSED_QKV.get(kind) == path:
                        d = trunk.hidden_size
                        assert base.out_features == 3 * d
                        slices = [(0, d), (d, 2 * d), (2 * d, 3 * d)]
                    new = LoRMALinear(base, r, alpha, slices)
                _set(blk, path, new)
        named = [(n, p) for n, p in model.named_parameters()
                 if ".lora_A" in n or ".lora_B" in n or ".lorma_A" in n or ".lorma_B" in n]
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, technique)

    elif technique == "adapters":
        bott = int(cfg.get("adapter_bottleneck", 64))
        for blk in trunk.blocks:
            for path in ADAPTER_SITES[kind]:
                _set(blk, path, AdapterWrapped(_get(blk, path), bott))
        named = [(n, p) for n, p in model.named_parameters() if ".adapter." in n]
        norm_ids = {id(p) for m in trunk.norm_modules() for p in m.parameters()}
        named += [(n, p) for n, p in model.named_parameters() if id(p) in norm_ids]
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, "adapters")

    elif technique == "bitfit":
        named = [(n, p) for n, p in model.named_parameters()
                 if n.startswith("trunk.") and n.endswith(".bias")]
        if not named:
            raise ValueError(f"BitFit needs bias terms; the {kind} architecture has none")
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, "bitfit")

    elif technique == "partial_ft":
        k = int(cfg.get("partial_top_k", 6))
        decay = float(cfg.get("layer_decay", 0.85))
        L = len(trunk.blocks)
        k = min(k, L)
        for i in range(L - k, L):
            named = [(f"trunk.blocks.{i}.{n}", p) for n, p in trunk.blocks[i].named_parameters()]
            for _, p in named:
                p.requires_grad = True
            expected |= {n for n, _ in named}
            groups += _groups(named, lr * decay ** (L - 1 - i), wd, f"block{i}")
        named = [(f"trunk.norm.{n}", p) for n, p in trunk.norm.named_parameters()]
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, "final_norm")
    else:
        raise ValueError(f"unknown technique {technique!r}")

    for _, p in head_named:
        p.requires_grad = True
    groups += _groups(head_named, head_lr, wd, "head")

    actual = {n for n, p in model.named_parameters() if p.requires_grad}
    if actual != expected:
        raise AssertionError(f"trainable-parameter mismatch: unexpected "
                             f"{sorted(actual - expected)[:5]}, missing {sorted(expected - actual)[:5]}")
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    log(f"   [{technique}] trainable {n_tr:,} / {n_all:,} params ({100 * n_tr / n_all:.3f}%) "
        f"in {len(actual)} tensors")
    return groups


def summarize_trainable(model, max_lines=40):
    rows = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
    lines = [f"{n}  {k:,}" for n, k in rows[:max_lines]]
    if len(rows) > max_lines:
        lines.append(f"... and {len(rows) - max_lines} more tensors")
    return lines


In [ ]:
%%writefile /kaggle/working/phase7/cjpe_common.py
"""
cjpe_common.py -- Phase-7 (Prior Case Retrieval) spec, registries and small helpers.

Torch-free. Every other cjpe_* module imports its constants from here.

Reused byte-identical from Phases 2/3 (never edited): ner_common.py (file discovery, atomic
JSON, disk guard, Phase-1 model discovery by the `arch` field), ner_models.py (Trunk),
ner_peft.py (LoRA, LoRMA+, adapters, BitFit, partial FT), phase1_models.py, bpe_tokenizer.py.
"""
import csv
import glob
import io
import json
import math
import os
import time
from typing import Dict, Iterator, List, Optional, Tuple

from ner_common import (MODELS, STUDENT_LAYERS, DiskFull, dir_size_gb, disk_guard, file_md5,
                        find_files, find_phase1_models, free_gb, read_json, write_json)

__all__ = ["SPEC", "MODELS", "STUDENT_LAYERS", "EXPERIMENTS", "DEFAULT_LR", "DEFAULT_EPOCHS",
           "MAX_TRAINABLE_PCT", "DEFAULT_MODEL_ORDER", "OUT_ROOT", "CACHE_DIR", "RAW_DIR",
           "CODE_DIR", "EXIT", "JOB_EXIT", "CSV_COLUMNS", "EPOCH_COLUMNS", "default_input_roots",
           "find_files", "find_phase1_models", "read_json", "write_json", "file_md5",
           "disk_guard", "DiskFull", "dir_size_gb", "free_gb", "module_md5s", "log_to",
           "write_csv", "read_csv", "now_str", "student_layers", "iter_out_dirs", "fmt",
           "MODULE_FILES"]

# =============================================================================================
# The brief (CS-781 Capstone, Phase 7 -- Prior Case Retrieval). Edit HERE only.
# =============================================================================================
SPEC: Dict = {
    "phase": 7,
    "task": "prior_case_retrieval",
    "hf_repo": "Exploration-Lab/CS-781-Capstone",
    "hf_folder": "Phase_7_PCR",
    "files": {"train": "Train/train.json", "train_candidates": "Train/train_candidates.json",
              "dev": "Dev/dev.json", "dev_candidates": "Dev/dev_candidates.json",
              "test": "Test/test.json", "test_candidates": "Test/test_candidates.json"},
    "metric": "micro_f1@7",                  # primary; Micro-P@7 / Micro-R@7 secondary
    "submission_entry": "prediction.json",   # {"query_id": [7 candidate ids]}, zip root
    "submission_zip": "submission.zip",
    "submissions": {"min": 6, "max": 40, "max_per_day": 10},
    "deadline": "2026-11-07 23:59 IST",
    "rules": {"min_experiments": 6, "min_peft_techniques": 5, "compulsory": ["lorma"],
              "min_kd_experiments": 1, "all_models_required": True,
              "foundation_params": [50e6, 300e6]},
}

DEFAULT_MODEL_ORDER = ["M4", "M2", "M3", "M1"]

# The six counted experiments (PDF 13.2.1 list; the technique is the ONLY difference) + extras.
EXPERIMENTS: Dict[str, Dict] = {
    "E1": {"folder": "E1_lorma", "technique": "lorma", "counts": True,
           "title": "LoRMA+ (compulsory)"},
    "E2": {"folder": "E2_full_ft", "technique": "full_ft", "counts": True,
           "title": "Full fine-tuning"},
    "E3": {"folder": "E3_partial_ft", "technique": "partial_ft", "counts": True,
           "title": "Partial / layer-wise fine-tuning (top 6 blocks, layer-wise LR decay)"},
    "E4": {"folder": "E4_qlora", "technique": "qlora", "counts": True,
           "title": "QLoRA (NF4 4-bit frozen base + LoRA)"},
    "E5": {"folder": "E5_prefix", "technique": "prefix", "counts": True,
           "title": "Prefix tuning (per-layer virtual activations)"},
    "E6": {"folder": "E6_kd", "technique": "kd", "counts": True,
           "title": "Knowledge distillation (full-FT teacher -> layer-pruned students, online "
                    "score-distribution KD)"},
    "E7": {"folder": "E7_extras", "technique": "extras", "counts": False,
           "title": "Extras: IA3, prompt tuning, zero-shot Phase-1, pooling / attention / "
                    "long-document / hard-negative ablations"},
    "E8": {"folder": "E8_report", "technique": "report", "counts": False,
           "title": "Report: ALL_RESULTS, upload plan, score-fusion ensemble, analysis (CPU)"},
}
COUNTED = [e for e, s in EXPERIMENTS.items() if s["counts"]]

DEFAULT_LR: Dict[str, float] = {
    "full_ft": 3e-5, "kd": 5e-5, "student_hard": 5e-5, "partial_ft": 1e-4, "lorma": 5e-4,
    "lora": 5e-4, "qlora": 5e-4, "adalora": 1e-3, "adapters": 5e-4, "prefix": 1e-3,
    "prompt": 5e-3, "ia3": 3e-3, "bitfit": 2e-3,
}
HEAD_LR = 1e-3
DEFAULT_EPOCHS: Dict[str, int] = {
    "full_ft": 4, "kd": 4, "student_hard": 4, "partial_ft": 4, "lorma": 5, "lora": 5,
    "qlora": 5, "adalora": 5, "adapters": 5, "prefix": 6, "prompt": 8, "ia3": 6, "bitfit": 5,
}
MAX_TRAINABLE_PCT: Dict[str, float] = {"lorma": 5.0, "lora": 5.0, "qlora": 5.0}

# =============================================================================================
# Paths / exit codes
# =============================================================================================
CODE_DIR = os.path.dirname(os.path.abspath(__file__))
OUT_ROOT = "/kaggle/working/phase7_out"
CACHE_DIR = "/kaggle/working/phase7_cache"
RAW_DIR = "/kaggle/working/phase7_raw"
OUT_MARKER = "phase7_out"
EXIT = {"ok": 0, "error": 1, "missing_dependency": 2, "gate_failure": 3,
        "validation_failure": 4}
# exit codes of one training job (cjpe_train.py)
JOB_EXIT = {0: "ok", 1: "error", 2: "does not fit the time budget", 4: "too many NaN steps",
            5: "early-fail watchdog", 6: "critical self-check failed", 7: "OOM at micro-batch 1",
            137: "killed"}

MODULE_FILES = ["bpe_tokenizer.py", "phase1_models.py", "ner_common.py", "ner_models.py",
                "ner_peft.py", "cjpe_common.py", "cjpe_data.py", "cjpe_tok.py", "cjpe_models.py",
                "cjpe_peft.py", "cjpe_eval.py", "cjpe_kd.py", "lsi_ml.py", "cjpe_predict.py",
                "cjpe_train.py", "summ_model.py", "pcr_ml.py", "pcr_data.py", "pcr_model.py",
                "pcr_train.py", "validate_submission.py", "select_best.py", "aggregate.py",
                "run_experiment.py", "selftest.py"]

CSV_COLUMNS = [
    "experiment", "technique", "model_id", "arch", "attn_mode", "pooling", "doc_repr",
    "chunk_tokens", "k_train", "k_infer", "hard_negs", "status", "fail_reason", "selfcheck",
    "train_queries", "train_pairs", "train_fraction", "dev_queries", "test_queries",
    "epochs_planned", "epochs_run", "best_epoch", "lr", "batch_pairs", "score_metric",
    "dev_score", "dev_micro_f1", "dev_micro_p", "dev_micro_r", "dev_dense_f1",
    "dev_dense_mean_f1", "dev_dense_max_f1", "dev_bm25_f1", "dense_weight", "doc_agg",
    "dev_dense_recall_at_100", "dev_proxy_map", "dev_proxy_map_init", "trainable_params",
    "total_params", "trainable_pct", "params_ratio_vs_teacher", "projected_minutes",
    "train_minutes", "tokens_per_s", "infer_tokens_per_s", "ms_per_doc_infer", "peak_mem_gb",
    "gpu_slot", "attempts", "leak_guard", "seed", "job", "submission_zip", "started_at",
    "finished_at"]
EPOCH_COLUMNS = ["experiment", "model_id", "job", "epoch", "step", "train_loss",
                 "dev_proxy_map", "dev_proxy_f1", "lr", "minutes", "skipped_steps"]


def default_input_roots() -> List[str]:
    """Where attached Kaggle datasets live; PHASE7_INPUT_ROOTS (os.pathsep-separated) overrides."""
    env = os.environ.get("PHASE7_INPUT_ROOTS")
    if env is not None:
        return [p for p in env.split(os.pathsep) if p]
    return ["/kaggle/input"]


def module_md5s(code_dir: str = CODE_DIR) -> Dict[str, Optional[str]]:
    out = {}
    for f in MODULE_FILES:
        p = os.path.join(code_dir, f)
        out[f] = file_md5(p) if os.path.exists(p) else None
    return out


def student_layers(mid: str, phase1_dir: str) -> List[int]:
    """Every second layer (== STUDENT_LAYERS for the real 12/14-layer models)."""
    cfg = read_json(os.path.join(phase1_dir, "config.json"))["config"]
    n = cfg.get("n_layer", cfg.get("num_hidden_layers"))
    ids = list(range(0, n, 2))
    if n in (12, 14):
        assert ids == STUDENT_LAYERS[mid], (ids, STUDENT_LAYERS[mid])
    return ids


def iter_out_dirs(out_root: Optional[str], input_roots=()) -> List[str]:
    """Every phase5_out folder: the working one first, then any inside attached inputs."""
    outs = []
    if out_root and os.path.isdir(out_root):
        outs.append(os.path.abspath(out_root))
    for r in input_roots:
        if r and os.path.isdir(r):
            for d in glob.glob(os.path.join(r, "**", OUT_MARKER), recursive=True):
                if os.path.isdir(d) and os.path.abspath(d) not in outs:
                    outs.append(os.path.abspath(d))
    return outs


def now_str() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")


def log_to(path: Optional[str], msg: str, prefix: str = "") -> None:
    line = f"{prefix}{msg}"
    print(line, flush=True)
    if path:
        try:
            os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
            with open(path, "a", encoding="utf-8") as f:
                f.write(time.strftime("%H:%M:%S ") + line + "\n")
        except OSError:
            pass


def fmt(v) -> str:
    """CSV cell: empty string for missing values, never None / nan."""
    if v is None:
        return ""
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, float):
        if math.isnan(v) or math.isinf(v):
            return ""
        return f"{v:.6g}" if abs(v) < 1e5 else f"{v:.1f}"
    return str(v)


def write_csv(path: str, rows: List[Dict], columns: List[str]) -> None:
    """Atomic CSV write (tmp + os.replace); unknown keys are ignored, missing ones are ''."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    buf = io.StringIO()
    w = csv.writer(buf)
    w.writerow(columns)
    for r in rows:
        w.writerow([fmt(r.get(c)) for c in columns])
    tmp = f"{path}.tmp{os.getpid()}"
    with open(tmp, "w", encoding="utf-8", newline="") as f:
        f.write(buf.getvalue())
    os.replace(tmp, path)


def read_csv(path: str) -> List[Dict]:
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


In [ ]:
%%writefile /kaggle/working/phase7/cjpe_data.py
"""
cjpe_data.py -- Phase-5 data: discovery, Hugging Face fallback download, schema-tolerant loading,
data audit and the leakage guard (sections 2, 3, 4.2, 4.3 of the brief).

Data integrity rules implemented here:
  * training uses train.json ONLY; dev.json only for selection; test.json text only for
    inference (enforced again by cjpe_train at start-up);
  * any train case whose id or text hash also occurs in test is DROPPED from training, any dev
    case that occurs in test is dropped from dev; every count is written to data_audit.json;
  * PROVENANCE.json records md5s of the three files and what each one was used for.
"""
import hashlib
import json
import os
import random
import re
import time
import urllib.error
import urllib.request
from collections import Counter
from typing import Dict, List, Optional, Tuple

from cjpe_common import RAW_DIR, SPEC, file_md5, find_files, log_to, write_json

ID_KEYS = ("id", "case_id", "doc_id", "Id", "ID", "uid", "name")
TEXT_KEYS = ("text", "case_text", "document", "doc", "content", "judgment_text", "raw_text")
LABEL_KEYS = ("labels", "label", "statutes", "statute_ids", "label_ids", "pred", "y", "target")


class DataError(RuntimeError):
    """The data cannot be loaded unambiguously: the pipeline stops and asks (section 13)."""


# =============================================================================================
# Hugging Face token (Kaggle secret) + download
# =============================================================================================
def get_hf_token() -> Optional[str]:
    """Kaggle secret `HF` (or HF_TOKEN / HUGGINGFACE_TOKEN), else the HF_TOKEN env var.
    The token is never printed, logged or written anywhere."""
    try:
        from kaggle_secrets import UserSecretsClient
        us = UserSecretsClient()
        for name in ("HF", "HF_TOKEN", "HUGGINGFACE_TOKEN"):
            try:
                tok = us.get_secret(name)
                if tok:
                    return tok.strip()
            except Exception:
                pass
    except Exception:
        pass
    return (os.environ.get("HF_TOKEN") or "").strip() or None


def _redact(msg: str, token: Optional[str]) -> str:
    return msg.replace(token, "<redacted>") if token else msg


def hf_download(dest_root: str = RAW_DIR, log=print) -> Optional[Dict[str, str]]:
    """Download the three files with urllib (no huggingface_hub). Returns {split: path} or None."""
    token = get_hf_token()
    log(f"   [data] HF token found: {'yes' if token else 'no'}"
        + (f" (length {len(token)})" if token else ""))
    if not token:
        return None
    out = {}
    for split, rel in SPEC["files"].items():
        url = (f"https://huggingface.co/datasets/{SPEC['hf_repo']}/resolve/main/"
               f"{SPEC['hf_folder']}/{rel}")
        dest = os.path.join(dest_root, SPEC["hf_folder"], rel)
        if os.path.exists(dest) and os.path.getsize(dest) > 0:
            try:
                with open(dest, encoding="utf-8") as f:
                    json.load(f)
                out[split] = dest
                continue
            except Exception:
                pass
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        last = None
        for attempt in range(3):
            try:
                req = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}",
                                                           "User-Agent": "cs781-phase7"})
                with urllib.request.urlopen(req, timeout=300) as r, open(dest + ".tmp", "wb") as f:
                    while True:
                        b = r.read(1 << 20)
                        if not b:
                            break
                        f.write(b)
                with open(dest + ".tmp", encoding="utf-8") as f:
                    json.load(f)                       # validate before accepting
                os.replace(dest + ".tmp", dest)
                out[split] = dest
                log(f"   [data] downloaded {rel} ({os.path.getsize(dest) / 1e6:.1f} MB)")
                break
            except urllib.error.HTTPError as e:
                last = f"HTTP {e.code}"
                if e.code in (401, 403):
                    log("   [data] HTTP %d: token missing, or this Hugging Face account has not "
                        "accepted the dataset terms" % e.code)
                    return None
            except Exception as e:                      # network / JSON errors: retry
                last = _redact(f"{type(e).__name__}: {e}", token)
            time.sleep(5 * (attempt + 1))
        if split not in out:
            log(f"   [data] download of {rel} failed after 3 attempts ({last})")
            return None
    return out


# =============================================================================================
# discovery
# =============================================================================================
def _peek_records(path: str) -> Optional[Tuple[str, Optional[str], Optional[str]]]:
    """(id_key, text_key, label_key) if `path` looks like a CJPE split, else None."""
    try:
        with open(path, encoding="utf-8") as f:
            obj = json.load(f)
        recs, keyed = _records(obj)
        if not recs:
            return None
        return _detect_keys(recs[:50], keyed)
    except Exception:
        return None


def discover_files(input_roots: List[str], log=print) -> Dict:
    """Find train/dev/test among the attached datasets; fall back to the HF download."""
    found, cands = {}, {}
    for split, rel in SPEC["files"].items():
        name = os.path.basename(rel)
        hits = find_files(name, input_roots, max_hits=200)
        good = []
        for h in hits:
            low = h.lower()
            score = (("phase_4" in low) + ("cjpe" in low) * 2 + (("/test/" in low) if split == "test"
                     else ("/train/" in low)))
            keys = _peek_records(h)
            if keys is not None:
                good.append((score, h))
        good.sort(key=lambda t: (-t[0], t[1]))
        cands[split] = [g[1] for g in good]
        if good:
            found[split] = good[0][1]
    for split, c in cands.items():
        if len(c) > 1:
            log(f"   [data] {len(c)} candidates for {split}: {c[:5]} -> using {found[split]}")
    if len(found) == len(SPEC["files"]):
        return {"source": "kaggle_dataset", "paths": found, "candidates": cands}
    log(f"   [data] attached dataset incomplete (found {sorted(found)}); trying the HF download")
    dl = hf_download(log=log)
    if dl and len(dl) == len(SPEC["files"]):
        return {"source": "huggingface", "paths": dl, "candidates": cands}
    return {"source": None, "paths": found, "candidates": cands}


# =============================================================================================
# schema-tolerant parsing
# =============================================================================================
def _records(obj) -> Tuple[List, bool]:
    """Return (records, keyed). keyed=True when the file was a dict keyed by id."""
    if isinstance(obj, list):
        return obj, False
    if isinstance(obj, dict):
        for k in ("data", "examples", "records", "cases", "rows"):
            if isinstance(obj.get(k), list):
                return obj[k], False
        # dict keyed by id: values are dicts or plain strings
        vals = list(obj.values())
        if vals and all(isinstance(v, (dict, str)) for v in vals[:50]):
            recs = []
            for k, v in obj.items():
                if isinstance(v, str):
                    recs.append({"id": k, "text": v})
                else:
                    r = dict(v)
                    r.setdefault("id", k)
                    recs.append(r)
            return recs, True
    raise DataError(f"unrecognised JSON container: {type(obj).__name__}")


def _detect_keys(recs: List, keyed: bool) -> Tuple[str, Optional[str], Optional[str]]:
    if not all(isinstance(r, dict) for r in recs):
        raise DataError("records are not JSON objects")
    keys = Counter(k for r in recs for k in r.keys())
    n = len(recs)
    idk = next((k for k in ID_KEYS if keys.get(k, 0) == n), None)
    tk = next((k for k in TEXT_KEYS if keys.get(k, 0) == n), None)
    if tk is None:                       # fall back: the single long-string field
        strf = [k for k in keys if keys[k] == n and all(isinstance(r[k], str) for r in recs)]
        strf = [k for k in strf if k not in ID_KEYS]
        if len(strf) == 1:
            tk = strf[0]
    lk = next((k for k in LABEL_KEYS if keys.get(k, 0) == n), None)
    if idk is None or tk is None:
        raise DataError(f"cannot find id/text keys among {dict(keys)}")
    return idk, tk, lk


def _to_label(v) -> List[int]:
    """Statute ids as a sorted unique list of ints (accepts int, numeric str, list, "1,7")."""
    if v is None:
        return []
    if isinstance(v, bool):
        raise DataError(f"label value {v!r} is a bool, not a statute id")
    if isinstance(v, (int, float)) and float(v) == int(v):
        return [int(v)]
    if isinstance(v, str):
        parts = [x for x in v.replace(";", ",").replace(" ", ",").split(",") if x.strip()]
        try:
            return sorted({int(float(x)) for x in parts})
        except ValueError:
            raise DataError(f"label value {v!r} is not a list of statute ids")
    if isinstance(v, (list, tuple)):
        out = set()
        for x in v:
            out.update(_to_label(x))
        return sorted(out)
    raise DataError(f"label value {v!r} is not a list of statute ids")


def load_split(path: str, split: str, log=print) -> Tuple[List[Dict], Dict]:
    """[{id, text, label|None}] + schema description. Raises DataError when ambiguous."""
    with open(path, encoding="utf-8") as f:
        obj = json.load(f)
    recs, keyed = _records(obj)
    if not recs:
        raise DataError(f"{split}: no records in {path}")
    idk, tk, lk = _detect_keys(recs, keyed)
    if split != "test" and lk is None:
        raise DataError(f"{split}: no label key among {sorted(recs[0].keys())}")
    out = []
    for r in recs:
        t = r[tk]
        if isinstance(t, list):                          # list of sentences/paragraphs
            t = "\n".join(str(x) for x in t)
        if not isinstance(t, str):
            raise DataError(f"{split}: text field {tk!r} is {type(t).__name__}")
        lab = _to_label(r[lk]) if (lk is not None and split != "test") else None
        out.append({"id": str(r[idk]), "text": t, "label": lab})
    schema = {"container": "dict keyed by id" if keyed else "list of objects",
              "id_key": idk, "text_key": tk, "label_key": lk, "n": len(out),
              "fields": sorted(recs[0].keys()), "md5": file_md5(path), "path": path}
    return out, schema


# =============================================================================================
# audit + leak guard
# =============================================================================================
def sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", "replace")).hexdigest()


def norm_hash(s: str) -> str:
    return sha1(re.sub(r"\s+", " ", s).strip().lower())


def _pct(xs: List[float], q: float) -> float:
    if not xs:
        return 0.0
    xs = sorted(xs)
    return float(xs[min(len(xs) - 1, int(q * (len(xs) - 1) + 0.5))])


def length_stats(vals: List[float]) -> Dict:
    if not vals:
        return {}
    return {"mean": sum(vals) / len(vals), "median": _pct(vals, 0.5), "p90": _pct(vals, 0.9),
            "p99": _pct(vals, 0.99), "max": max(vals), "min": min(vals)}


def leak_guard(data: Dict[str, List[Dict]], log=print) -> Dict:
    """Drop train/dev cases that also occur in test (by id or raw / normalised text hash)."""
    h = {s: [(d["id"], sha1(d["text"]), norm_hash(d["text"])) for d in data[s]] for s in data}
    sets = {s: {"id": {a for a, _, _ in v}, "sha1": {b for _, b, _ in v},
                "norm": {c for _, _, c in v}} for s, v in h.items()}
    rep = {}
    for a, b in (("train", "test"), ("dev", "test"), ("train", "dev")):
        rep[f"{a}_x_{b}"] = {k: len(sets[a][k] & sets[b][k]) for k in ("id", "sha1", "norm")}
    dropped = {}
    for s in ("train", "dev"):
        keep, drop = [], []
        for d, (i, x, y) in zip(data[s], h[s]):
            if i in sets["test"]["id"] or x in sets["test"]["sha1"] or y in sets["test"]["norm"]:
                drop.append(i)
            else:
                keep.append(d)
        dropped[s] = drop
        if drop:
            log(f"   !!! [leak-guard] {len(drop)} {s} case(s) also occur in test -> DROPPED from "
                f"{s} (ids {drop[:5]}...)")
        data[s] = keep
    rep["dropped_from_train"] = len(dropped["train"])
    rep["dropped_from_dev"] = len(dropped["dev"])
    rep["dropped_ids"] = {k: v[:200] for k, v in dropped.items()}
    n_train = len(data["train"]) + len(dropped["train"])
    rep["train_test_overlap_pct"] = 100.0 * len(dropped["train"]) / max(1, n_train)
    rep["status"] = "passed"
    return rep


def basic_audit(data: Dict[str, List[Dict]], schemas: Dict) -> Dict:
    a = {"schema": schemas, "splits": {}}
    for s, docs in data.items():
        ids = [d["id"] for d in docs]
        texts = [d["text"] for d in docs]
        e = {"n": len(docs), "empty_text": sum(1 for t in texts if not t.strip()),
             "duplicate_ids": len(ids) - len(set(ids)),
             "duplicate_texts": len(texts) - len({sha1(t) for t in texts}),
             "chars": length_stats([float(len(t)) for t in texts]),
             "words": length_stats([float(len(t.split())) for t in texts])}
        if s != "test":
            c = Counter(x for d in docs for x in (d["label"] or []))
            nl = [len(d["label"] or []) for d in docs]
            freq = sorted(c.values())
            e["n_labels"] = len(c)
            e["labels_per_doc"] = length_stats([float(x) for x in nl])
            e["docs_without_labels"] = sum(1 for x in nl if x == 0)
            e["label_freq_quantiles"] = {q: _pct([float(x) for x in freq], q)
                                         for q in (0.1, 0.25, 0.5, 0.75, 0.9)} if freq else {}
            e["pct_labels_lt5"] = 100.0 * sum(1 for x in freq if x < 5) / max(1, len(freq))
            e["top_labels"] = c.most_common(10)
        a["splits"][s] = e
    return a


def token_audit(data: Dict[str, List[Dict]], tok, sample: int = 2000, seed: int = 42,
                n_proc: int = 1) -> Dict:
    """BPE length statistics on a seeded sample (full tokenisation); chars-per-token ratio."""
    from cjpe_tok import count_tokens_many
    rng = random.Random(seed)
    out = {}
    pool = []
    for s, docs in data.items():
        idx = list(range(len(docs)))
        rng.shuffle(idx)
        pool += [(s, i) for i in idx[:max(1, sample // 3)]]
    counts = count_tokens_many([data[s][i]["text"] for s, i in pool], tok, n_proc)
    by = {}
    for (s, i), n in zip(pool, counts):
        by.setdefault(s, []).append((n, len(data[s][i]["text"]), data[s][i]["label"]))
    tot_c = sum(c for v in by.values() for _, c, _ in v)
    tot_t = sum(n for v in by.values() for n, _, _ in v)
    ratio = tot_c / max(1, tot_t)
    out["chars_per_token"] = ratio
    out["sample_size"] = len(pool)
    for s, v in by.items():
        toks = [float(n) for n, _, _ in v]
        e = {"tokens": length_stats(toks)}
        for thr in (512, 1024, 2048, 4096, 8192):
            e[f"pct_over_{thr}"] = 100.0 * sum(1 for t in toks if t > thr) / max(1, len(toks))
        if s != "test":
            buckets = {}
            for n, _, lab in v:
                b = bucket_of(n)
                buckets.setdefault(b, []).append(lab)
            e["labels_per_doc_by_length"] = {b: {"n": len(l), "mean_labels":
                                                 sum(len(x or []) for x in l) / len(l)}
                                         for b, l in sorted(buckets.items())}
        out[s] = e
    return out


BUCKETS = [(512, "<=512"), (1024, "<=1024"), (2048, "<=2048"), (4096, "<=4096"),
           (10 ** 12, ">4096")]


def bucket_of(n_tokens: float) -> str:
    for thr, name in BUCKETS:
        if n_tokens <= thr:
            return name
    return BUCKETS[-1][1]


def provenance(schemas: Dict, leak: Dict, n_train_used: int, source: str) -> Dict:
    return {
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
        "data_source": source,
        "md5": {s: schemas[s]["md5"] for s in schemas},
        "paths": {s: schemas[s]["path"] for s in schemas},
        "used_for": {"train.json": "training (the ONLY file whose labels are used for gradient "
                                   "updates)",
                     "dev.json": "model selection, early stopping, threshold tuning and local "
                                 "evaluation only (never trained on, no train+dev refit)",
                     "test.json": "inference only (text); no test-time training, no pseudo "
                                  "labels, nothing tuned on test predictions"},
        "train_cases_used": n_train_used,
        "train_cases_dropped_by_leak_guard": leak.get("dropped_from_train", 0),
        "dev_cases_dropped_by_leak_guard": leak.get("dropped_from_dev", 0),
        "leak_guard": leak.get("status", "unknown"),
    }


def print_audit(a: Dict, log=print) -> None:
    log("   [audit] split   n      empty dup_id dup_text  pos_rate  chars(median)")
    for s, e in a["splits"].items():
        pr = e.get("positive_rate")
        log(f"   [audit] {s:<6} {e['n']:>6} {e['empty_text']:>6} {e['duplicate_ids']:>6} "
            f"{e['duplicate_texts']:>8}  {'' if pr is None else f'{pr:.3f}':>8}  "
            f"{e['chars'].get('median', 0):,.0f}")
    t = a.get("tokens")
    if t:
        log(f"   [audit] chars/token {t['chars_per_token']:.2f} (sample of {t['sample_size']})")
        for s in ("train", "dev", "test"):
            if s in t:
                e = t[s]
                log(f"   [audit] {s} tokens: median {e['tokens'].get('median', 0):,.0f}, p90 "
                    f"{e['tokens'].get('p90', 0):,.0f}, max {e['tokens'].get('max', 0):,.0f}; "
                    f">512 {e['pct_over_512']:.0f}%  >1024 {e['pct_over_1024']:.0f}%  "
                    f">2048 {e['pct_over_2048']:.0f}%  >4096 {e['pct_over_4096']:.0f}%")
    lg = a.get("leak_guard")
    if lg:
        log(f"   [leak-guard] train x test {lg['train_x_test']}, dev x test {lg['dev_x_test']}, "
            f"train x dev {lg['train_x_dev']}; dropped train {lg['dropped_from_train']}, "
            f"dev {lg['dropped_from_dev']}")


def print_examples(data: Dict[str, List[Dict]], log=print) -> None:
    for s, docs in data.items():
        if docs:
            d = docs[0]
            log(f"   [example] {s}: id={d['id']!r} label={d['label']!r} text={d['text'][:300]!r}")


In [ ]:
%%writefile /kaggle/working/phase7/cjpe_tok.py
"""
cjpe_tok.py -- fast head/tail tokenisation, the token cache and the long-document
representations (section 5 of the brief).

Why slices: the Phase-1 BPE is pure Python. A judgment can be 50k+ tokens but every
representation we train on only needs the first and/or last few 511-token chunks, so we cut
character slices at *token boundaries* and tokenise only those.

Token-boundary rule (proved in the docstring of `cut_head` and tested in selftest): the
pre-tokeniser regex never lets a non-whitespace run continue into whitespace, so position p is
a pre-token boundary of the FULL text whenever text[p] is whitespace and text[p-1] is not. The
regex only looks forward, so tokenising text[:p] gives exactly the full-text tokens before p,
and tokenising text[p:] gives exactly the full-text tokens from p on.

Representations (`doc_repr`), each chunk = [EOS] + <= 511 body tokens (<= 512 total):
  hier     floor(K/2) first chunks + ceil(K/2) last chunks of the document, in order (K=1 ->
           the LAST chunk: the court's conclusion is at the end; Malik et al. 2021 (ILDC) found
           the last 512 tokens the most informative single window)
  first_k  first K chunks          last_k   last K chunks
  headtail ONE sequence of <= 1024 tokens: [EOS] first 383 [EOS] last 639
  sections first chunk + 2 chunks starting at the last "conclusion-like" heading in the last
           70% of the text (ORDER / CONCLUSION / HELD / RESULT / "for the foregoing reasons"
           ...), falling back to `hier` when there is none
  full     consecutive chunks from the start, up to 16 (8176 tokens; ablation only)
  spread   (Phase 5) K chunks evenly spaced over the cached head (up to 16 chunks) + the last
           chunk of the document, in order
  cite     (Phase 5) first chunk + 2 chunks starting at the densest window of statute citations
           ("section 302", "Article 14", "I.P.C.", "Act, 1956" ...) + last chunk; falls back to
           `hier` when the document has no citation
"""
import hashlib
import json
import os
import pickle
import re
import time
from multiprocessing import get_context
from typing import Dict, List, Optional

import numpy as np

from cjpe_common import file_md5, write_json

BODY = 511                     # body tokens per chunk; + EOS = 512
EOS_ID, PAD_ID = 0, 1
HT_HEAD, HT_TAIL = 383, 639    # headtail: 1 + 383 + 1 + 639 = 1024
CACHE_VERSION = 3
CITE_RE = re.compile(
    r"(?i)\b(?:section|sec\.|s\.|sections|article|art\.|rule|order)\s*\d+[a-z]?\b"
    r"|\b(?:i\.?\s?p\.?\s?c|cr\.?\s?p\.?\s?c|c\.?\s?p\.?\s?c|constitution)\b"
    r"|\bact,?\s*(?:18|19|20)\d\d\b")
SECTION_RE = re.compile(
    r"(?im)^\s*(?:ORDER|CONCLUSION|CONCLUSIONS|HELD|RESULT|DECISION|OPERATIVE PORTION)\b"
    r"|for the (?:foregoing|aforesaid|above|reasons) (?:reasons|stated|discussed)"
    r"|in the (?:result|circumstances)\b|we (?:accordingly )?(?:allow|dismiss|set aside)\b")

_TOK = None                    # per-process tokenizer (pool workers)


# =============================================================================================
# boundary-safe slicing
# =============================================================================================
def _ws(c: str) -> bool:
    return c.isspace()


def cut_head(text: str, j: int) -> int:
    """Smallest p >= j that is a safe boundary (text[p] whitespace, text[p-1] not); len(text)
    if none. text[:p] then tokenises exactly like the first tokens of the full text."""
    n = len(text)
    p = max(1, j)
    while p < n and not (_ws(text[p]) and not _ws(text[p - 1])):
        p += 1
    return min(p, n)


def cut_tail(text: str, i: int) -> int:
    """Largest p <= i that is a safe boundary; 0 if none (then the whole text is used)."""
    p = min(i, len(text) - 1)
    while p > 0 and not (_ws(text[p]) and not _ws(text[p - 1])):
        p -= 1
    return max(p, 0)


def _init_worker(bpe_dir: str) -> None:
    global _TOK
    from bpe_tokenizer import BPETokenizer
    _TOK = BPETokenizer.from_pretrained(bpe_dir)


def _encode(text: str) -> List[int]:
    return _TOK.encode(text)


def encode_doc(text: str, need_head: int, need_tail: int, want_sections: bool = False,
               need_sect: int = 2 * BODY, tok=None) -> Dict:
    """Head/tail token ids of one document; exact token count when the doc was fully encoded."""
    enc = (tok or _TOK).encode
    L = len(text)
    need = need_head + need_tail
    out = {"n_chars": L, "sect": None}
    full_thr = need * 9                              # short documents: encode everything
    if L <= full_thr:
        ids = enc(text)
        out.update(complete=True, head=np.asarray(ids, np.uint16), tail=None, n_tokens=len(ids))
    else:
        hc = cut_head(text, need_head * 6)
        while True:
            head = enc(text[:hc])
            if len(head) >= need_head or hc >= L:
                break
            hc = cut_head(text, hc * 2)
        tc = cut_tail(text, L - need_tail * 6)
        while True:
            tail = enc(text[tc:])
            if len(tail) >= need_tail or tc <= 0:
                break
            tc = cut_tail(text, L - (L - tc) * 2)
        if hc >= tc:                                 # slices meet: encode the whole document
            ids = enc(text)
            out.update(complete=True, head=np.asarray(ids, np.uint16), tail=None,
                       n_tokens=len(ids))
        else:
            seen_chars = hc + (L - tc)
            est = (len(head) + len(tail)) / max(1, seen_chars) * L
            out.update(complete=False, head=np.asarray(head[:need_head], np.uint16),
                       tail=np.asarray(tail[-need_tail:] if need_tail else [], np.uint16),
                       n_tokens=int(round(est)))
    if want_sections:
        pos = None
        for m in SECTION_RE.finditer(text):
            if m.start() >= 0.3 * L:
                pos = m.start()
        if pos is not None:
            s = cut_tail(text, pos) if pos > 0 else 0
            e = cut_head(text, s + need_sect * 6)
            ids = enc(text[s:e])
            out["sect"] = np.asarray(ids[:need_sect], np.uint16)
        # Phase 5: densest window of statute citations (window = ~2 chunks of characters)
        pos = [m.start() for m in CITE_RE.finditer(text)]
        out["cite"] = None
        if pos:
            win = need_sect * 5
            best, bi, j = 0, 0, 0
            for i in range(len(pos)):
                while pos[i] - pos[j] > win:
                    j += 1
                if i - j + 1 > best:
                    best, bi = i - j + 1, j
            s = cut_tail(text, max(0, pos[bi] - 200)) if pos[bi] > 200 else 0
            e = cut_head(text, s + need_sect * 6)
            ids = enc(text[s:e])
            out["cite"] = np.asarray(ids[:need_sect], np.uint16)
            out["n_cites"] = len(pos)
    return out


def _encode_doc_star(args):
    return encode_doc(*args)


def _count(text: str) -> int:
    return len(_TOK.encode(text))


def _pool(bpe_dir: str, n_proc: int):
    return get_context("fork").Pool(n_proc, initializer=_init_worker, initargs=(bpe_dir,))


def count_tokens_many(texts: List[str], tok, n_proc: int = 1) -> List[int]:
    if n_proc <= 1 or len(texts) < 50 or not getattr(tok, "_bpe_dir", None):
        return [len(tok.encode(t)) for t in texts]
    with _pool(tok._bpe_dir, n_proc) as p:
        return p.map(_count, texts, chunksize=8)


# =============================================================================================
# the cache
# =============================================================================================
def cache_key(file_md5s: Dict[str, str], tok_md5: str) -> str:
    """Base key: data files + tokenizer (+ CACHE_VERSION). The head/tail sizes and the sections
    flag are part of the FILE name, so a larger cache (e.g. from 00) serves smaller needs."""
    s = json.dumps({"v": CACHE_VERSION, "files": file_md5s, "tok": tok_md5}, sort_keys=True)
    return hashlib.md5(s.encode()).hexdigest()[:16]


def cache_name(key: str, need_head: int, need_tail: int, sections: bool) -> str:
    return f"cache_{key}_h{need_head}_t{need_tail}_s{int(bool(sections))}.pkl"


def build_cache(data: Dict[str, List[Dict]], bpe_dir: str, out_dir: str, key: str,
                need_head: int, need_tail: int, sections: bool, n_proc: int, log=print) -> str:
    """Tokenise every document once; writes <out_dir>/cache_<key>.pkl + cache_meta.json."""
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, cache_name(key, need_head, need_tail, sections))
    t0 = time.time()
    res = {}
    for split, docs in data.items():
        args = [(d["text"], need_head, need_tail, sections) for d in docs]
        if n_proc > 1 and len(args) >= 64:
            with _pool(bpe_dir, n_proc) as p:
                enc = p.map(_encode_doc_star, args, chunksize=16)
        else:
            _init_worker(bpe_dir)
            enc = [_encode_doc_star(a) for a in args]
        rows = []
        for d, e in zip(docs, enc):
            e["id"], e["label"] = d["id"], d["label"]
            rows.append(e)
        res[split] = rows
        log(f"   [cache] {split}: {len(rows)} docs tokenised ({time.time() - t0:.0f}s so far)")
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(res, f, protocol=4)
    os.replace(tmp, path)
    write_json(os.path.join(out_dir, "cache_meta.json"),
               {"key": key, "file": os.path.basename(path), "need_head": need_head,
                "need_tail": need_tail, "sections": sections, "seconds": time.time() - t0,
                "n": {s: len(v) for s, v in res.items()}})
    return path


def find_cache(key: str, need_head: int, need_tail: int, sections: bool,
               roots: List[str]) -> Optional[str]:
    """Any cache of the same data + tokenizer that holds at least the needed tokens."""
    pat = re.compile(rf"^cache_{re.escape(key)}_h(\d+)_t(\d+)_s([01])\.pkl$")
    best = None
    for r in roots:
        if not r or not os.path.isdir(r):
            continue
        for dp, _dn, fns in os.walk(r):
            for fn in fns:
                m = pat.match(fn)
                if not m:
                    continue
                h, t, sc = int(m.group(1)), int(m.group(2)), m.group(3) == "1"
                if h >= need_head and t >= need_tail and (sc or not sections):
                    size = h + t
                    if best is None or size < best[0]:
                        best = (size, os.path.join(dp, fn))
    return best[1] if best else None


def load_cache(path: str) -> Dict[str, List[Dict]]:
    with open(path, "rb") as f:
        return pickle.load(f)


# =============================================================================================
# representations
# =============================================================================================
def _with_eos(a: np.ndarray) -> np.ndarray:
    return np.concatenate([np.zeros(1, np.int64), a.astype(np.int64)])


def _first_chunks(ids: np.ndarray, n: int) -> List[np.ndarray]:
    return [ids[i * BODY:(i + 1) * BODY] for i in range(n) if len(ids[i * BODY:(i + 1) * BODY])]


def _last_chunks(ids: np.ndarray, n: int) -> List[np.ndarray]:
    out = []
    L = len(ids)
    for j in range(n, 0, -1):                  # in document order
        s, e = max(0, L - j * BODY), L - (j - 1) * BODY
        if e > s:
            out.append(ids[s:e])
    return out


def doc_chunks(e: Dict, repr_: str, K: int) -> List[np.ndarray]:
    """The list of model inputs (each starts with EOS, <= 512 tokens; headtail <= 1024)."""
    head = e["head"]
    tail = e["tail"] if e["tail"] is not None else head
    complete = bool(e["complete"])
    n_tok = len(head) if complete else None
    if repr_ == "headtail":
        if complete and n_tok <= 1023:
            return [_with_eos(head)]
        h = head[:HT_HEAD]
        t = tail[-HT_TAIL:]
        return [np.concatenate([_with_eos(h), _with_eos(t)])]
    if repr_ == "cite" and e.get("cite") is not None and len(e["cite"]):
        parts = _first_chunks(head, 1) + _first_chunks(e["cite"], 2) + _last_chunks(tail, 1)
        return [_with_eos(p) for p in parts if len(p)]
    if repr_ == "spread":
        n_all = -(-len(head) // BODY)
        if complete and n_all <= K:
            return [_with_eos(p) for p in _first_chunks(head, n_all)]
        avail = _first_chunks(head, min(n_all, 16))
        whole = complete and n_all <= 16             # avail already ends at the document end
        k = max(1, min(K if whole else K - 1, len(avail)))
        pick = sorted({int(round(i * (len(avail) - 1) / max(1, k - 1))) for i in range(k)})
        parts = [avail[i] for i in pick] + ([] if whole else _last_chunks(tail, 1))
        return [_with_eos(p) for p in parts if len(p)]
    if repr_ == "sections" and e.get("sect") is not None and len(e["sect"]):
        parts = _first_chunks(head, 1) + _first_chunks(e["sect"], 2)
        return [_with_eos(p) for p in parts]
    if repr_ in ("hier", "sections", "cite"):
        a, b = K // 2, K - K // 2
    elif repr_ == "first_k":
        a, b = K, 0
    elif repr_ == "last_k":
        a, b = 0, K
    elif repr_ == "full":
        return [_with_eos(p) for p in _first_chunks(head, K)]
    else:
        raise ValueError(f"unknown doc_repr {repr_!r}")
    if complete:
        ids = head
        n_all = -(-len(ids) // BODY)
        if n_all <= a + b:                       # short document: every chunk, in order
            return [_with_eos(p) for p in _first_chunks(ids, n_all)]
        parts = _first_chunks(ids, a) + _last_chunks(ids, b)
    else:
        parts = _first_chunks(head, a) + _last_chunks(tail, b)
    return [_with_eos(p) for p in parts if len(p)]


def needs_for(repr_: str, K: int):
    """(need_head, need_tail) tokens the cache must hold for a representation."""
    if repr_ == "full":
        return K * BODY, 0
    if repr_ == "headtail":
        return 2 * BODY, 2 * BODY
    if repr_ == "first_k":
        return K * BODY, BODY
    if repr_ == "last_k":
        return BODY, K * BODY
    a, b = K // 2, K - K // 2
    return max(a, 1) * BODY, max(b, 1) * BODY


In [ ]:
%%writefile /kaggle/working/phase7/cjpe_models.py
"""
cjpe_models.py -- the document classifier (section 6 of the brief).

    chunks of one document --Phase-1 Trunk (shared, reused submodules)--> hidden states
    -> masked mean over each chunk's tokens (fp32)            = chunk vectors c_1..c_n
    -> learned attention pooling  a_i = softmax_i(w . tanh(W c_i)) over the REAL chunks
    -> dropout 0.1 -> Linear(H, 2)                              = 2 logits (fp32)

All new parameters live in `model.classifier` so that ner_peft.apply_technique (which trains
every `classifier.*` parameter as "the head") works unchanged for every technique.
Padding chunks never reach the trunk: only real chunks are encoded, then scattered into a
(B, n_max, H) grid whose missing slots get exactly zero attention weight.
"""
import json
import os
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from ner_models import Trunk, count_params, load_phase1_lm, torch_load

PAD_ID = 1


class DocHead(nn.Module):
    def __init__(self, d: int, n_classes: int = 2, dropout: float = 0.1, pool_dim: int = 256):
        super().__init__()
        self.att_proj = nn.Linear(d, pool_dim)
        self.att_vec = nn.Linear(pool_dim, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(d, n_classes)
        for lin in (self.att_proj, self.att_vec, self.out):
            nn.init.normal_(lin.weight, std=0.02)
            if lin.bias is not None:
                nn.init.zeros_(lin.bias)

    def attention(self, c: torch.Tensor, cmask: torch.Tensor) -> torch.Tensor:
        """c (B, n, H) fp32, cmask (B, n) bool -> weights (B, n); masked slots exactly 0."""
        s = self.att_vec(torch.tanh(self.att_proj(c))).squeeze(-1)
        s = s.masked_fill(~cmask, float("-inf"))
        w = torch.softmax(s, dim=-1)
        return torch.where(cmask, w, torch.zeros_like(w))

    def forward(self, c, cmask):
        w = self.attention(c, cmask)
        doc = (w.unsqueeze(-1) * c).sum(1)
        return self.out(self.dropout(doc)), w


class DocClassifier(nn.Module):
    def __init__(self, trunk: Trunk, n_classes: int = 2, dropout: float = 0.1):
        super().__init__()
        self.trunk = trunk
        self.classifier = DocHead(trunk.hidden_size, n_classes, dropout)

    def chunk_vectors(self, input_ids, attention_mask):
        h = self.trunk(input_ids, attention_mask)                      # (C, T, H)
        m = attention_mask.unsqueeze(-1).float()
        return (h.float() * m).sum(1) / m.sum(1).clamp(min=1.0)        # (C, H) fp32

    def forward(self, input_ids, attention_mask, doc_index, slot, n_docs: int, n_slots: int):
        v = self.chunk_vectors(input_ids, attention_mask)
        H = v.shape[-1]
        grid = torch.zeros(n_docs, n_slots, H, device=v.device, dtype=v.dtype)
        cmask = torch.zeros(n_docs, n_slots, device=v.device, dtype=torch.bool)
        grid[doc_index, slot] = v
        cmask[doc_index, slot] = True
        # the head runs in fp32 even under autocast (section 6: logits and loss in fp32)
        with torch.autocast(device_type=v.device.type, enabled=False):
            logits, w = self.classifier(grid.float(), cmask)
        return logits.float(), w


def collate_docs(chunk_lists: List[List[np.ndarray]], device=None) -> Dict:
    """Flatten the chunks of several documents into one right-padded (C, T) batch."""
    flat, doc_index, slot = [], [], []
    for d, chunks in enumerate(chunk_lists):
        for s, c in enumerate(chunks):
            flat.append(c)
            doc_index.append(d)
            slot.append(s)
    T = max(len(c) for c in flat)
    assert T <= 1024, f"sequence of {T} tokens > 1024"
    ids = np.full((len(flat), T), PAD_ID, np.int64)
    mask = np.zeros((len(flat), T), np.int64)
    for i, c in enumerate(flat):
        ids[i, :len(c)] = c
        mask[i, :len(c)] = 1
    b = {"input_ids": torch.from_numpy(ids), "attention_mask": torch.from_numpy(mask),
         "doc_index": torch.tensor(doc_index, dtype=torch.long),
         "slot": torch.tensor(slot, dtype=torch.long),
         "n_docs": len(chunk_lists), "n_slots": max(len(c) for c in chunk_lists),
         "n_tokens": int(mask.sum())}
    if device is not None:
        for k in ("input_ids", "attention_mask", "doc_index", "slot"):
            b[k] = b[k].to(device, non_blocking=True)
    return b


def forward_batch(model: DocClassifier, b: Dict):
    return model(b["input_ids"], b["attention_mask"], b["doc_index"], b["slot"], b["n_docs"],
                 b["n_slots"])


def build_doc_model(phase1_dir: str, attn_mode: str = "bidir", layer_ids=None,
                    dropout: float = 0.1, n_labels: int = 2, bias_init=None):
    """Phase 5: Linear(H, n_labels) multi-label head; bias initialised to the label log-odds
    prior (rare labels start near 0 probability: stable BCE)."""
    lm, cfg = load_phase1_lm(phase1_dir)
    trunk = Trunk(lm, attn_mode, layer_ids)
    del lm
    model = DocClassifier(trunk, int(n_labels), dropout)
    if bias_init is not None:
        with torch.no_grad():
            model.classifier.out.bias.copy_(torch.as_tensor(np.asarray(bias_init, np.float32)))
    return model, cfg


def model_size(model: nn.Module) -> Dict[str, float]:
    tot = count_params(model)
    tr = count_params(model, trainable_only=True)
    return {"total_params": tot, "trainable_params": tr,
            "trainable_pct": 100.0 * tr / max(1, tot)}


def device_report(model: nn.Module) -> Dict[str, int]:
    out: Dict[str, int] = {}
    for p in list(model.parameters()) + list(model.buffers()):
        out[str(p.device)] = out.get(str(p.device), 0) + 1
    return out


def assert_one_device(model: nn.Module, device) -> None:
    """Lesson L1: after .to(device) EVERY parameter and buffer must be on that device."""
    want = torch.device(device)
    bad = [n for n, p in list(model.named_parameters()) + list(model.named_buffers())
           if p.device.type != want.type or (want.index is not None and p.device.index not in
                                             (None, want.index))]
    if bad:
        raise AssertionError(f"{len(bad)} tensors not on {want}: {bad[:5]}")


# ---------------------------------------------------------------------------------------------
# checkpoints of trained models
# ---------------------------------------------------------------------------------------------
STATE_BUFFER_TAGS = (".ada_mask", ".ada_ipt_", ".ada_unc_")


def trainable_state_dict(model: nn.Module, half: bool = False) -> Dict[str, torch.Tensor]:
    sd = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
    for n, b in model.named_buffers():
        if any(t in n for t in STATE_BUFFER_TAGS):
            sd[n] = b.detach().cpu()
    if half:
        sd = {n: (t.half() if t.is_floating_point() and ".ada_mask" not in n else t)
              for n, t in sd.items()}
    return sd


def load_trainable_state(model: nn.Module, sd: Dict[str, torch.Tensor]) -> None:
    params = dict(model.named_parameters())
    bufs = dict(model.named_buffers())
    unknown = [n for n in sd if n not in params and n not in bufs]
    if unknown:
        raise KeyError(f"checkpoint has unknown tensors: {unknown[:5]}")
    with torch.no_grad():
        for n, t in sd.items():
            dst = params.get(n, bufs.get(n))
            dst.copy_(t.to(dst.dtype))
    absent = {n for n, p in params.items() if p.requires_grad} - set(sd)
    if absent:
        raise KeyError(f"checkpoint lacks trainable parameters: {sorted(absent)[:5]}")


def rebuild_trained(run_config: Dict, best_pt: str, phase1_dir: Optional[str] = None,
                    device="cpu"):
    """Recreate a finished run's module structure and load its best.pt (teacher reuse, E8)."""
    from cjpe_peft import apply_technique
    p1 = phase1_dir or run_config["phase1_dir"]
    model, _ = build_doc_model(p1, run_config["attn_mode"], run_config.get("layer_ids"),
                               n_labels=len(run_config["label_ids"]))
    tech = "full_ft" if run_config["technique"] in ("kd", "student_hard") else \
        run_config["technique"]
    apply_technique(model, tech, run_config, log=lambda *a: None)
    load_trainable_state(model, torch_load(best_pt))
    return model.to(device).eval()


In [ ]:
%%writefile /kaggle/working/phase7/cjpe_peft.py
"""
cjpe_peft.py -- fine-tuning techniques for the document classifier (pure PyTorch, no `peft`).

  full_ft / kd / student_hard   every trunk parameter + head       (ner_peft, Phase 2)
  lorma      LoRMA+ y = (I + s*B*A) W0 x, rank 16, alpha 32       (ner_peft, Phase 2; compulsory)
             Bihany, Patel & Modi (2025), arXiv:2506.07621
  lora       LoRA r=16, alpha 32                                  (ner_peft; extra E7)
  adapters   Houlsby bottleneck 64 after attention-out and MLP-out + all norms (ner_peft)
  partial_ft top-6 blocks + final norm + head, layer-wise LR decay 0.85 (ner_peft)
  bitfit     bias terms only (M1/M4)                              (ner_peft; extra E7)
  adalora    dW = P diag(lambda) Q with an orthogonality regulariser, sensitivity importance
             (EMA + uncertainty) and a cubic budget schedule 16 -> 8 average rank.
             Zhang et al. (2023), ICLR, arXiv:2303.10512. Code copied from Phase 3 rr_peft.py
             (AdaLoRALinear + AdaLoRA), adapted only in where the head lives (`classifier.*`).

The head (`classifier.*`: attention pooling + output layer) is always trainable.

Lesson L1 (device-order bug of the Phase-3 pilot): the reused ner_peft modules create their
parameters on the default (CPU) device. `apply_technique` therefore moves the whole model to the
device of the frozen base (`trunk.embed.weight.device`) after inserting the modules, and then
asserts that every parameter and buffer sits on that one device, so the result is correct in
both orders ("apply then .to(device)" in production, ".to(device) then apply" in checks).
"""
import copy
import math
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from cjpe_common import DEFAULT_LR, HEAD_LR
from cjpe_models import assert_one_device, model_size
from ner_peft import (FUSED_QKV, TARGETS, LoRALinear, LoRMALinear, AdapterWrapped, _get,
                      _groups, _set)
from ner_peft import apply_technique as phase2_apply

PHASE2 = ("full_ft", "kd", "student_hard", "lorma", "lora", "adapters", "partial_ft", "bitfit")
ALL_TECHNIQUES = PHASE2 + ("adalora",)


# =============================================================================================
# AdaLoRA (copied from Phase 3 rr_peft.py)
# =============================================================================================
class AdaLoRALinear(nn.Module):
    """y = W0 x + b + (alpha/r) * P diag(lambda * mask) Q dropout(x); lambda = 0 at init."""

    def __init__(self, base: nn.Linear, r: int = 16, alpha: float = 32, dropout: float = 0.05):
        super().__init__()
        self.base = base
        dev = base.weight.device
        self.r, self.scale = r, alpha / r
        self.ada_P = nn.Parameter(torch.randn(base.out_features, r, device=dev) * 0.02)
        self.ada_E = nn.Parameter(torch.zeros(r, device=dev))
        self.ada_Q = nn.Parameter(torch.randn(r, base.in_features, device=dev) * 0.02)
        self.register_buffer("ada_mask", torch.ones(r, device=dev))
        for n in ("P", "E", "Q"):
            self.register_buffer(f"ada_ipt_{n}", torch.zeros_like(getattr(self, f"ada_{n}")))
            self.register_buffer(f"ada_unc_{n}", torch.zeros_like(getattr(self, f"ada_{n}")))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        y = self.base(x)
        lam = (self.ada_E * self.ada_mask).to(x.dtype)
        u = F.linear(self.drop(x), self.ada_Q.to(x.dtype)) * lam
        return y + self.scale * F.linear(u, self.ada_P.to(x.dtype))

    @torch.no_grad()
    def merged_weight(self):
        d = self.ada_P @ torch.diag(self.ada_E * self.ada_mask) @ self.ada_Q
        return self.base.weight + self.scale * d.to(self.base.weight.dtype)

    def orth_reg(self) -> torch.Tensor:
        P, Q = self.ada_P.float(), self.ada_Q.float()
        I = torch.eye(self.r, device=P.device)
        return ((P.t() @ P - I) ** 2).sum() + ((Q @ Q.t() - I) ** 2).sum()


class Technique:
    """No-op hooks (Phase-2 techniques)."""
    name = "base"

    def extra_loss(self, model) -> Optional[torch.Tensor]:
        return None

    def after_backward(self, model, step: int, total: int) -> None:
        return None

    def selectable(self, step: int, total: int) -> bool:
        return True

    def force_final(self, model) -> None:
        return None

    def final_report(self, model) -> Dict:
        return {}


class AdaLoRA(Technique):
    name = "adalora"

    def setup(self, model, cfg, log=print):
        trunk = model.trunk
        self.r = int(cfg.get("adalora_init_r", 16))
        self.target_r = int(cfg.get("adalora_target_r", 8))
        alpha = float(cfg.get("adalora_alpha", 32))
        self.tinit_frac = float(cfg.get("adalora_tinit_frac", 0.1))
        self.tfinal_frac = float(cfg.get("adalora_tfinal_frac", 0.2))
        self.delta_t = int(cfg.get("adalora_delta_t", 10))
        self.beta1 = float(cfg.get("adalora_beta1", 0.85))
        self.beta2 = float(cfg.get("adalora_beta2", 0.85))
        self.orth_w = float(cfg.get("adalora_orth", 0.1))
        for p in model.parameters():
            p.requires_grad = False
        for blk in trunk.blocks:
            for path in TARGETS[trunk.kind]:
                _set(blk, path, AdaLoRALinear(_get(blk, path), self.r, alpha,
                                              float(cfg.get("lora_dropout", 0.05))))
        self.modules_ = [(n, m) for n, m in model.named_modules() if isinstance(m, AdaLoRALinear)]
        self.b0 = self.r * len(self.modules_)
        self.bT = self.target_r * len(self.modules_)
        named = [(n, p) for n, p in model.named_parameters() if ".ada_" in n]
        for _, p in named:
            p.requires_grad = True
        log(f"   [adalora] {len(self.modules_)} matrices, rank {self.r} -> target "
            f"{self.target_r} (budget {self.b0} -> {self.bT} singular values)")
        return _groups(named, float(cfg["lr"]), float(cfg.get("weight_decay", 0.01)), "adalora")

    def extra_loss(self, model):
        if not self.modules_ or self.orth_w <= 0:
            return None
        return self.orth_w * sum(m.orth_reg() for _, m in self.modules_) / len(self.modules_)

    def phases(self, total: int):
        tf_start = max(0, min(total - int(self.tfinal_frac * total), total - 1))
        return min(int(self.tinit_frac * total), tf_start), tf_start

    def budget(self, step: int, total: int) -> int:
        ti, tf_start = self.phases(total)
        if step < ti:
            return self.b0
        if step >= tf_start:
            return self.bT
        mul = 1.0 - (step - ti) / max(1, tf_start - ti)
        return int(self.bT + (self.b0 - self.bT) * mul ** 3)

    def selectable(self, step, total):
        return step > self.phases(total)[1]

    @torch.no_grad()
    def _update_importance(self):
        for _, m in self.modules_:
            for n in ("P", "E", "Q"):
                p = getattr(m, f"ada_{n}")
                if p.grad is None:
                    continue
                ipt = (p * p.grad).abs()
                ema = getattr(m, f"ada_ipt_{n}")
                unc = getattr(m, f"ada_unc_{n}")
                ema.mul_(self.beta1).add_((1 - self.beta1) * ipt)
                unc.mul_(self.beta2).add_((1 - self.beta2) * (ipt - ema).abs())

    @torch.no_grad()
    def _scores(self) -> List[torch.Tensor]:
        out = []
        for _, m in self.modules_:
            s = lambda n: getattr(m, f"ada_ipt_{n}") * getattr(m, f"ada_unc_{n}")  # noqa: E731
            out.append(s("E") + s("P").mean(0) + s("Q").mean(1))
        return out

    @torch.no_grad()
    def _prune_to(self, b: int):
        scores = self._scores()
        flat = torch.cat(scores)
        if b >= flat.numel():
            return
        thr = torch.kthvalue(flat.float().cpu(), flat.numel() - b).values.item()
        for (_, m), s in zip(self.modules_, scores):
            keep = (s > thr).float()
            m.ada_mask.copy_(keep)
            m.ada_E.mul_(keep)

    @torch.no_grad()
    def after_backward(self, model, step, total):
        ok = all(p.grad is None or torch.isfinite(p.grad).all()
                 for _, m in self.modules_ for p in (m.ada_P, m.ada_E, m.ada_Q))
        if not ok:
            return
        self._update_importance()
        ti, tf_start = self.phases(total)
        if step < ti or step > tf_start:
            return
        if step != tf_start and (step - ti) % self.delta_t != 0:
            return
        self._prune_to(self.budget(step, total))

    def force_final(self, model):
        """A run stopped early by the time guard still ends at the target budget."""
        kept = sum(int(m.ada_mask.sum()) for _, m in self.modules_)
        if kept > self.bT:
            self._prune_to(self.bT)

    def final_report(self, model):
        ranks = {n.replace("trunk.blocks.", "L"): int(m.ada_mask.sum().item())
                 for n, m in model.named_modules() if isinstance(m, AdaLoRALinear)}
        per_layer: Dict[str, int] = {}
        for n, r in ranks.items():
            per_layer[n.split(".")[0]] = per_layer.get(n.split(".")[0], 0) + r
        return {"adalora_rank_per_layer": per_layer,
                "adalora_budget_final": int(sum(ranks.values())),
                "adalora_budget_initial": self.b0, "adalora_budget_target": self.bT}


# =============================================================================================
# entry point
# =============================================================================================
def default_hparams(technique: str) -> Dict:
    lr = DEFAULT_LR.get(technique, 1e-3)
    head = 10 * lr if technique in ("full_ft", "kd", "student_hard") else HEAD_LR
    return {"lr": lr, "head_lr": head}


def apply_technique(model: nn.Module, technique: str, cfg: Dict, log=print
                    ) -> Tuple[List[Dict], Technique]:
    """Insert the technique, set requires_grad, return (optimizer groups, hook object)."""
    if technique not in ALL_TECHNIQUES:
        raise ValueError(f"unknown technique {technique!r}")
    dev = model.trunk.embed.weight.device
    if technique == "adalora":
        tech = AdaLoRA()
        groups = tech.setup(model, cfg, log)
        head = [(n, p) for n, p in model.named_parameters() if n.startswith("classifier.")]
        for _, p in head:
            p.requires_grad = True
        groups += _groups(head, float(cfg.get("head_lr", HEAD_LR)),
                          float(cfg.get("weight_decay", 0.01)), "head")
        expected = {n for n, _ in head} | {n for n, p in model.named_parameters() if ".ada_" in n}
        actual = {n for n, p in model.named_parameters() if p.requires_grad}
        if actual != expected:
            raise AssertionError(f"[adalora] trainable mismatch {sorted(actual ^ expected)[:5]}")
    else:
        p2 = "full_ft" if technique in ("kd", "student_hard") else technique
        groups = phase2_apply(model, p2, cfg, log=lambda *a: None)
        tech = Technique()
        tech.name = technique
    model.to(dev)                                         # lesson L1
    assert_one_device(model, dev)
    s = model_size(model)
    log(f"   [{technique}] trainable {s['trainable_params']:,} / {s['total_params']:,} "
        f"({s['trainable_pct']:.3f}%)")
    return groups, tech


# =============================================================================================
# merge (used by the self-checks: merged == unmerged)
# =============================================================================================
def _clone_linear(base: nn.Linear) -> nn.Linear:
    lin = nn.Linear(base.in_features, base.out_features, bias=base.bias is not None,
                    device=base.weight.device, dtype=base.weight.dtype)
    with torch.no_grad():
        lin.weight.copy_(base.weight)
        if base.bias is not None:
            lin.bias.copy_(base.bias)
    return lin


def merge_model(model: nn.Module) -> int:
    """Fold LoRA / LoRMA / AdaLoRA updates into plain Linear layers. Returns the count."""
    hits = [(n, m) for n, m in model.named_modules()
            if isinstance(m, (LoRALinear, LoRMALinear, AdaLoRALinear))]
    for n, m in hits:
        lin = _clone_linear(m.base)
        with torch.no_grad():
            lin.weight.copy_(m.merged_weight())
        _set(model, n, lin)
    return len(hits)


In [ ]:
%%writefile /kaggle/working/phase7/cjpe_eval.py
"""
cjpe_eval.py -- binary Macro-F1 (implemented here, no sklearn), threshold tuning on dev, and the
per-run analysis tables (section 11.1).

Macro-F1 = mean of the F1 of class 0 and class 1 (both always counted: a class that is never
predicted gets F1 = 0, so predicting only the majority class is punished).
"""
from typing import Dict, List, Optional, Sequence

import numpy as np

from cjpe_data import BUCKETS, bucket_of


def confusion(gold: Sequence[int], pred: Sequence[int]) -> Dict[str, int]:
    g = np.asarray(gold, dtype=np.int64)
    p = np.asarray(pred, dtype=np.int64)
    return {"tp": int(((g == 1) & (p == 1)).sum()), "fp": int(((g == 0) & (p == 1)).sum()),
            "fn": int(((g == 1) & (p == 0)).sum()), "tn": int(((g == 0) & (p == 0)).sum())}


def _prf(tp: int, fp: int, fn: int):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def binary_scores(gold: Sequence[int], pred: Sequence[int]) -> Dict[str, float]:
    c = confusion(gold, pred)
    p1, r1, f1 = _prf(c["tp"], c["fp"], c["fn"])            # class 1 = positive
    p0, r0, f0 = _prf(c["tn"], c["fn"], c["fp"])            # class 0 viewed as positive
    n = max(1, sum(c.values()))
    return {"macro_f1": (f0 + f1) / 2, "macro_p": (p0 + p1) / 2, "macro_r": (r0 + r1) / 2,
            "accuracy": (c["tp"] + c["tn"]) / n, "f1_class0": f0, "f1_class1": f1,
            "p_class0": p0, "p_class1": p1, "r_class0": r0, "r_class1": r1,
            "precision": p1, "recall": r1, "f1_positive": f1, **c}


def macro_f1(gold, pred) -> float:
    return binary_scores(gold, pred)["macro_f1"]


def tune_tau(p1: Sequence[float], gold: Sequence[int], lo: float = 0.05, hi: float = 0.95,
             step: float = 0.01, min_frac: float = 0.05) -> Dict:
    """Best macro-F1 threshold on dev, keeping >= min_frac of predictions in each class.
    Falls back to 0.5 when no threshold beats it."""
    p = np.asarray(p1, dtype=np.float64)
    g = np.asarray(gold, dtype=np.int64)
    base = macro_f1(g, (p >= 0.5).astype(int))
    best_t, best_f = 0.5, base
    n = len(p)
    for t in np.arange(lo, hi + 1e-9, step):
        pr = (p >= t).astype(int)
        pos = pr.mean() if n else 0
        if pos < min_frac or pos > 1 - min_frac:
            continue
        f = macro_f1(g, pr)
        if f > best_f + 1e-12:
            best_t, best_f = float(round(t, 4)), f
    return {"tau": best_t, "f1_tuned": best_f, "f1_at_0p5": base,
            "used": "tuned" if best_t != 0.5 else "0.5"}


def length_table(gold, pred, n_tokens) -> List[Dict]:
    rows = []
    by: Dict[str, List[int]] = {}
    for i, n in enumerate(n_tokens):
        by.setdefault(bucket_of(n), []).append(i)
    for _, name in BUCKETS:
        idx = by.get(name, [])
        if not idx:
            continue
        s = binary_scores([gold[i] for i in idx], [pred[i] for i in idx])
        rows.append({"bucket": name, "n": len(idx), "macro_f1": s["macro_f1"],
                     "accuracy": s["accuracy"], "pos_rate_gold": float(np.mean([gold[i] for i in idx]))})
    return rows


def reliability(p1, gold, bins: int = 10) -> List[Dict]:
    p = np.asarray(p1, dtype=np.float64)
    g = np.asarray(gold, dtype=np.float64)
    rows = []
    for b in range(bins):
        lo, hi = b / bins, (b + 1) / bins
        m = (p >= lo) & ((p < hi) if b < bins - 1 else (p <= hi))
        if m.any():
            rows.append({"bin": f"{lo:.1f}-{hi:.1f}", "n": int(m.sum()),
                         "mean_p1": float(p[m].mean()), "observed_pos_rate": float(g[m].mean())})
    return rows


def ece(p1, gold, bins: int = 10) -> float:
    rel = reliability(p1, gold, bins)
    n = sum(r["n"] for r in rel)
    return float(sum(r["n"] * abs(r["mean_p1"] - r["observed_pos_rate"]) for r in rel) / max(1, n))


def full_report(ids, gold, p1, tau: float, n_tokens) -> Dict:
    pred = [int(x >= tau) for x in p1]
    s = binary_scores(gold, pred)
    s05 = binary_scores(gold, [int(x >= 0.5) for x in p1])
    return {"tau": tau, "scores": s, "scores_at_0p5": s05,
            "by_length": length_table(gold, pred, n_tokens),
            "reliability": reliability(p1, gold), "ece": ece(p1, gold),
            "n": len(ids), "pred_pos_rate": float(np.mean(pred)) if pred else 0.0}


In [ ]:
%%writefile /kaggle/working/phase7/cjpe_kd.py
"""
cjpe_kd.py -- knowledge distillation for E6 (section 7.2).

Teacher : full fine-tuning of TEACHER_MODEL (default M4) trained inside E6 (job
          `teacher_<model>`), or a finished E1 run of that model found in the attached inputs
          whose fingerprint (data, tokenizer, representation) matches.
          After training, the teacher's logits for every train / dev / test document are cached
          once in `teacher_logits.npz` (eval mode, no dropout, no_grad).
Students: the four Phase-1 models pruned to every second layer (Trunk(layer_ids=...)), full
          fine-tuning, same `hier` representation, so the logits align per DOCUMENT.
Loss    : L = a * BCE(student, y) + (1 - a) * T^2 * BCE(s/T, sigmoid(t/T))  (multi-label),
          a = 0.5, T = 2 (Hinton et al., 2015, "Distilling the Knowledge in a Neural Network").
"""
import glob
import os
from typing import Dict, Optional

import numpy as np
import torch
import torch.nn.functional as F

from cjpe_common import read_json


def kd_loss(s_logits: torch.Tensor, t_logits: torch.Tensor, labels: torch.Tensor,
            alpha: float = 0.5, T: float = 2.0, weight: Optional[torch.Tensor] = None
            ) -> torch.Tensor:
    """Phase 5 (multi-label): a * BCE(student, gold) + (1-a) * T^2 * soft-target BCE against
    sigmoid(teacher / T); `weight` = per-label pos_weight."""
    from lsi_ml import kd_loss_ml
    return kd_loss_ml(s_logits, t_logits, labels, alpha, T, weight)


def save_teacher_logits(path: str, logits: Dict[str, Dict[str, np.ndarray]]) -> None:
    """logits[split] = {"ids": array(str), "logits": (n, 2) float32}."""
    flat = {}
    for s, d in logits.items():
        flat[f"{s}_ids"] = np.asarray(d["ids"])
        flat[f"{s}_logits"] = np.asarray(d["logits"], np.float32)
    tmp = path + ".tmp.npz"
    np.savez_compressed(tmp, **flat)
    os.replace(tmp, path)


def load_teacher_logits(path: str) -> Dict[str, Dict[str, np.ndarray]]:
    z = np.load(path, allow_pickle=False)
    out = {}
    for k in z.files:
        if k.endswith("_ids"):
            s = k[:-4]
            out[s] = {"ids": z[k], "logits": z[f"{s}_logits"]}
    return out


def teacher_map(path: str, split: str = "train") -> Dict[str, np.ndarray]:
    d = load_teacher_logits(path)[split]
    return {str(i): l for i, l in zip(d["ids"], d["logits"])}


def find_reusable_teacher(input_roots, model_folder: str, fingerprint: Dict) -> Optional[str]:
    """A finished E1 run of the teacher model with an identical fingerprint, or None."""
    for r in input_roots:
        if not r or not os.path.isdir(r):
            continue
        pat = os.path.join(r, "**", "phase5_out", "E1_full_ft", model_folder, "DONE.json")
        for done in glob.glob(pat, recursive=True):
            run = os.path.dirname(done)
            rc = read_json(os.path.join(run, "logs", "run_config.json"), {}) or {}
            if not os.path.exists(os.path.join(run, "checkpoints", "best.pt")):
                continue
            if all(rc.get(k) == v for k, v in fingerprint.items()):
                return run
    return None


In [ ]:
%%writefile /kaggle/working/phase7/lsi_ml.py
"""
lsi_ml.py -- Phase 5 (Legal Statute Identification) multi-label core: targets, losses,
metrics, dev-only decision rules and the per-run analysis (brief sections 3.3, 3.5, 5).

Labels inside the pipeline are multi-hot uint8 rows over the TRAIN label map (`label_ids`,
sorted statute ids seen in train.json). Statute ids that occur only in dev are never predicted
(the evaluator only accepts valid ids) and are reported as `dev_only_labels`.
"""
from typing import Dict, List, Optional, Sequence

import numpy as np
import torch
import torch.nn.functional as F

GRID = np.round(np.arange(0.02, 0.901, 0.02), 4)
RARE, FREQ = 5, 20                       # train-support buckets: <5 rare, 5-20 medium, >20 freq


# =============================================================================================
# labels / targets
# =============================================================================================
def multihot(label_lists: Sequence, lab2i: Dict[int, int], L: int) -> np.ndarray:
    Y = np.zeros((len(label_lists), L), np.uint8)
    for r, labs in enumerate(label_lists):
        for x in labs or []:
            j = lab2i.get(int(x))
            if j is not None:
                Y[r, j] = 1
    return Y


def targets(labels, idx, device) -> torch.Tensor:
    return torch.from_numpy(np.stack([labels[i] for i in idx]).astype(np.float32)).to(device)


def pos_weight(Y: np.ndarray, mode: str = "sqrt_inv", cap: float = 30.0) -> Optional[np.ndarray]:
    if mode == "none":
        return None
    pos = Y.sum(0).astype(np.float64)
    neg = len(Y) - pos
    r = neg / np.maximum(pos, 1.0)
    w = np.sqrt(r) if mode == "sqrt_inv" else r
    w = np.clip(w, 1.0, cap)
    w[pos == 0] = 1.0
    return w.astype(np.float32)


def prior_logit(Y: np.ndarray) -> np.ndarray:
    p = (Y.sum(0).astype(np.float64) + 0.5) / (len(Y) + 1.0)
    p = np.clip(p, 1e-4, 0.5)
    return np.log(p / (1 - p)).astype(np.float32)


# =============================================================================================
# losses (per document = sum over labels; fp32)
# =============================================================================================
def ml_loss(logits: torch.Tensor, Y: torch.Tensor, pw: Optional[torch.Tensor] = None,
            kind: str = "bce") -> torch.Tensor:
    x = logits.float()
    if kind == "asl":                    # asymmetric loss (Ridnik et al. 2021): g-=4, g+=0, clip
        p = torch.sigmoid(x)
        pm = (p - 0.05).clamp(min=0)
        lp = Y * torch.log(p.clamp(min=1e-8))
        ln = (1 - Y) * torch.log((1 - pm).clamp(min=1e-8)) * pm.pow(4)
        return -(lp + ln).sum(1)
    if kind == "focal":                  # focal BCE, gamma 2, with pos_weight on positives
        b = F.binary_cross_entropy_with_logits(x, Y, reduction="none")
        p = torch.sigmoid(x)
        pt = Y * p + (1 - Y) * (1 - p)
        w = (1 - pt).pow(2)
        if pw is not None:
            w = w * (1 + (pw - 1) * Y)
        return (w * b).sum(1)
    return F.binary_cross_entropy_with_logits(x, Y, pos_weight=pw, reduction="none").sum(1)


def kd_loss_ml(s_logits, t_logits, Y, alpha=0.5, T=2.0, pw=None) -> torch.Tensor:
    """a * BCE(student, gold) + (1-a) * T^2 * BCE(student/T, sigmoid(teacher/T)); mean/doc."""
    s = s_logits.float()
    hard = ml_loss(s, Y, pw).mean()
    soft = F.binary_cross_entropy_with_logits(s / T, torch.sigmoid(t_logits.float() / T),
                                              reduction="none").sum(1).mean()
    return alpha * hard + (1 - alpha) * T * T * soft


def sigmoid_np(z: np.ndarray) -> np.ndarray:
    return (1.0 / (1.0 + np.exp(-np.clip(z, -40, 40)))).astype(np.float32)


# =============================================================================================
# metrics
# =============================================================================================
def counts(G: np.ndarray, P: np.ndarray):
    G = G.astype(bool)
    P = P.astype(bool)
    tp = (G & P).sum(0).astype(np.int64)
    fp = (~G & P).sum(0).astype(np.int64)
    fn = (G & ~P).sum(0).astype(np.int64)
    return tp, fp, fn


def _f1(tp, fp, fn):
    p = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0)
    r = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    f = np.where(p + r > 0, 2 * p * r / np.maximum(p + r, 1e-12), 0.0)
    return p, r, f


def ml_scores(G: np.ndarray, P: np.ndarray, train_support: Optional[np.ndarray] = None,
              n_dev_only_pos: int = 0) -> Dict:
    """macro_f1 = mean F1 over labels with dev support (what an evaluator built from the dev
    reference would see); macro_f1_all = over every train label (0 for unseen+unpredicted)."""
    tp, fp, fn = counts(G, P)
    p, r, f = _f1(tp, fp, fn)
    sup = G.sum(0) > 0
    out = {"macro_f1": float(f[sup].mean()) if sup.any() else 0.0,
           "macro_p": float(p[sup].mean()) if sup.any() else 0.0,
           "macro_r": float(r[sup].mean()) if sup.any() else 0.0,
           "macro_f1_all": float(f.mean()) if len(f) else 0.0}
    TP, FP, FN = int(tp.sum()), int(fp.sum()), int(fn.sum()) + int(n_dev_only_pos)
    mp = TP / max(1, TP + FP)
    mr = TP / max(1, TP + FN)
    out.update({"micro_f1": 2 * mp * mr / max(1e-12, mp + mr), "micro_p": mp, "micro_r": mr,
                "subset_acc": float((G.astype(bool) == P.astype(bool)).all(1).mean())
                if len(G) else 0.0,
                "mean_pred_labels": float(P.sum(1).mean()) if len(P) else 0.0,
                "mean_gold_labels": float(G.sum(1).mean()) if len(G) else 0.0,
                "n_labels_with_dev_support": int(sup.sum()), "tp": TP, "fp": FP, "fn": FN})
    if train_support is not None:
        for name, m in (("rare", train_support < RARE),
                        ("medium", (train_support >= RARE) & (train_support <= FREQ)),
                        ("frequent", train_support > FREQ)):
            mm = m & sup
            out[f"macro_f1_{name}"] = float(f[mm].mean()) if mm.any() else None
            out[f"n_{name}"] = int(mm.sum())
    return out


def macro(G, P) -> float:
    tp, fp, fn = counts(G, P)
    f = _f1(tp, fp, fn)[2]
    sup = G.sum(0) > 0
    return float(f[sup].mean()) if sup.any() else 0.0


# =============================================================================================
# decision rules (dev only)
# =============================================================================================
def apply_rule(Pr: np.ndarray, taus: np.ndarray, kmax: int) -> np.ndarray:
    """threshold per label, then >= 1 label per document (top-1) and <= kmax labels."""
    pred = Pr >= taus[None, :]
    if len(Pr) == 0:
        return pred
    top1 = Pr.argmax(1)
    pred[np.arange(len(Pr)), top1] = True
    over = np.where(pred.sum(1) > kmax)[0]
    for i in over:
        keep = np.argsort(-np.where(pred[i], Pr[i], -1.0))[:kmax]
        pred[i] = False
        pred[i, keep] = True
    return pred


def fit_global(Pr, G, kmax) -> (float, float):
    best_t, best_f = 0.5, -1.0
    L = Pr.shape[1]
    for t in GRID:
        f = macro(G, apply_rule(Pr, np.full(L, t, np.float32), kmax))
        if f > best_f + 1e-12:
            best_t, best_f = float(t), f
    return best_t, best_f


def fit_perlabel(Pr, G, tg: float, m: int = 5, lam: float = 0.5) -> np.ndarray:
    L = Pr.shape[1]
    taus = np.full(L, tg, np.float32)
    sup = G.sum(0)
    cand = np.where(sup >= m)[0]
    if not len(cand):
        return taus
    best_f = np.full(len(cand), -1.0)
    best_t = np.full(len(cand), tg)
    Gc = G[:, cand].astype(bool)
    for t in GRID:
        Pc = Pr[:, cand] >= t
        tp = (Gc & Pc).sum(0)
        fp = (~Gc & Pc).sum(0)
        fn = (Gc & ~Pc).sum(0)
        f = _f1(tp, fp, fn)[2]
        better = f > best_f + 1e-12
        best_f[better] = f[better]
        best_t[better] = t
    taus[cand] = lam * best_t + (1 - lam) * tg
    return taus


def choose_rule(Pr: np.ndarray, G: np.ndarray, kmax: int, seed: int = 0,
                allow_perlabel: bool = True) -> Dict:
    L = Pr.shape[1]
    tg, fg = fit_global(Pr, G, kmax)
    taus_pl = fit_perlabel(Pr, G, tg)
    f_pl_in = macro(G, apply_rule(Pr, taus_pl, kmax))
    # honest estimate: 2-fold cross-fitting on dev
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(Pr))
    halves = [perm[: len(perm) // 2], perm[len(perm) // 2:]]
    cv_g, cv_p = [], []
    for a, b in ((0, 1), (1, 0)):
        A, B = halves[a], halves[b]
        if len(A) < 4 or len(B) < 4:
            continue
        t_a, _ = fit_global(Pr[A], G[A], kmax)
        pl_a = fit_perlabel(Pr[A], G[A], t_a)
        cv_g.append(macro(G[B], apply_rule(Pr[B], np.full(L, t_a, np.float32), kmax)))
        cv_p.append(macro(G[B], apply_rule(Pr[B], pl_a, kmax)))
    cvg = float(np.mean(cv_g)) if cv_g else None
    cvp = float(np.mean(cv_p)) if cv_p else None
    use_pl = bool(allow_perlabel and cvg is not None and cvp is not None and cvp > cvg + 0.002)
    taus = taus_pl if use_pl else np.full(L, tg, np.float32)
    return {"name": "perlabel" if use_pl else "global", "tau_global": tg, "kmax": int(kmax),
            "taus": [round(float(x), 4) for x in taus],
            "dev_macro_global": fg, "dev_macro_perlabel_insample": f_pl_in,
            "cv_macro_global": cvg, "cv_macro_perlabel": cvp}


def rule_taus(rule: Dict, L: int) -> np.ndarray:
    t = np.asarray(rule.get("taus") or [rule.get("tau_global", 0.5)] * L, np.float32)
    return t if len(t) == L else np.full(L, float(rule.get("tau_global", 0.5)), np.float32)


# =============================================================================================
# per-run analysis
# =============================================================================================
def per_label_rows(G, P, taus, label_ids, train_support) -> List[Dict]:
    tp, fp, fn = counts(G, P)
    p, r, f = _f1(tp, fp, fn)
    return [{"label": int(label_ids[j]), "train_support": int(train_support[j]),
             "dev_support": int(G[:, j].sum()), "precision": float(p[j]), "recall": float(r[j]),
             "f1": float(f[j]), "predicted_count": int(P[:, j].sum()), "tau": float(taus[j])}
            for j in range(G.shape[1])]


def confused_pairs(G, P, label_ids, top: int = 30) -> List[Dict]:
    from collections import Counter
    c = Counter()
    Gb, Pb = G.astype(bool), P.astype(bool)
    for i in range(len(G)):
        miss = np.where(Gb[i] & ~Pb[i])[0]
        spur = np.where(Pb[i] & ~Gb[i])[0]
        for a in miss[:20]:
            for b in spur[:20]:
                c[(int(label_ids[a]), int(label_ids[b]))] += 1
    return [{"missed": a, "predicted_instead": b, "docs": n} for (a, b), n in c.most_common(top)]


def length_rows(G, P, n_tokens) -> List[Dict]:
    from cjpe_data import BUCKETS, bucket_of
    by: Dict[str, List[int]] = {}
    for i, n in enumerate(n_tokens):
        by.setdefault(bucket_of(n), []).append(i)
    rows = []
    for _, name in BUCKETS:
        idx = by.get(name, [])
        if idx:
            s = ml_scores(G[idx], P[idx])
            rows.append({"bucket": name, "n": len(idx), "macro_f1": s["macro_f1"],
                         "micro_f1": s["micro_f1"], "mean_gold_labels": s["mean_gold_labels"]})
    return rows


def ece_ml(Pr, G, bins: int = 10, max_n: int = 2_000_000) -> float:
    p = Pr.ravel()
    g = G.ravel().astype(np.float64)
    if len(p) > max_n:
        sel = np.random.default_rng(0).choice(len(p), max_n, replace=False)
        p, g = p[sel], g[sel]
    e, n = 0.0, len(p)
    for b in range(bins):
        lo, hi = b / bins, (b + 1) / bins
        m = (p >= lo) & ((p < hi) if b < bins - 1 else (p <= hi))
        if m.any():
            e += m.sum() * abs(float(p[m].mean()) - float(g[m].mean()))
    return float(e / max(1, n))


def full_report(G, Pr, rule, n_tokens, label_ids, train_support, n_dev_only_pos=0) -> Dict:
    L = Pr.shape[1]
    taus = rule_taus(rule, L)
    P = apply_rule(Pr, taus, rule["kmax"])
    P05 = apply_rule(Pr, np.full(L, 0.5, np.float32), rule["kmax"])
    Pg = apply_rule(Pr, np.full(L, rule["tau_global"], np.float32), rule["kmax"])
    return {"rule": {k: v for k, v in rule.items() if k != "taus"},
            "scores": ml_scores(G, P, train_support, n_dev_only_pos),
            "scores_at_0p5": ml_scores(G, P05, train_support, n_dev_only_pos),
            "scores_global": ml_scores(G, Pg, train_support, n_dev_only_pos),
            "by_length": length_rows(G, P, n_tokens),
            "confused_pairs": confused_pairs(G, P, label_ids),
            "ece": ece_ml(Pr, G), "n": int(len(G)),
            "_P": P, "_per_label": per_label_rows(G, P, taus, label_ids, train_support)}


In [ ]:
%%writefile /kaggle/working/phase7/cjpe_predict.py
"""
cjpe_predict.py -- batched inference, the dev/test output files, the submission zip and its
validation (Phase 5, multi-label).

Every finished model writes:
  dev/dev_probs.npz       ids, gold (uint8), probs (float16)      dev/dev_report.json
  dev/per_label_dev.csv   label, supports, P/R/F1, predicted count, tau
  dev/errors.csv          dev docs whose label set is wrong (missed / spurious ids, snippet)
  dev/dev_metrics.json    scores of the chosen rule (+ global, 0.5)
  submission/test_probs.npz, submission/prediction.json, submission/submission.zip,
  submission/validation_report.json
"""
import csv
import json
import os
import zipfile
from typing import Dict, List

import numpy as np
import torch

from cjpe_common import write_json
from lsi_ml import apply_rule, full_report, rule_taus
from cjpe_models import collate_docs, forward_batch
from validate_submission import validate


def batches_by_chunks(chunk_lists: List[List[np.ndarray]], max_chunks: int,
                      order=None) -> List[List[int]]:
    """Group document indices so that each group has <= max_chunks chunks (>= 1 doc)."""
    order = list(range(len(chunk_lists))) if order is None else list(order)
    out, cur, n = [], [], 0
    for i in order:
        c = len(chunk_lists[i])
        if cur and n + c > max_chunks:
            out.append(cur)
            cur, n = [], 0
        cur.append(i)
        n += c
    if cur:
        out.append(cur)
    return out


@torch.no_grad()
def predict_logits(model, chunk_lists, device, amp_ctx, max_chunks: int = 32) -> np.ndarray:
    """(n_docs, L) fp32 logits; documents sorted by length for less padding, then unsorted."""
    model.eval()
    n = len(chunk_lists)
    out = np.zeros((n, model.classifier.out.out_features), np.float32)
    order = sorted(range(n), key=lambda i: (len(chunk_lists[i]),
                                            max(len(c) for c in chunk_lists[i])))
    mc = max_chunks
    groups = batches_by_chunks(chunk_lists, mc, order)
    gi = 0
    while gi < len(groups):
        g = groups[gi]
        try:
            b = collate_docs([chunk_lists[i] for i in g], device)
            with amp_ctx():
                logits, _ = forward_batch(model, b)
            out[g] = logits.float().cpu().numpy()
            gi += 1
        except RuntimeError as e:                      # OOM: smaller groups, same order
            if "out of memory" not in str(e).lower() or mc <= 1:
                raise
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            mc = max(1, mc // 2)
            rest = [i for gg in groups[gi:] for i in gg]
            groups = groups[:gi] + batches_by_chunks(chunk_lists, mc, rest)
    return out


def softmax1(logits: np.ndarray) -> np.ndarray:
    z = logits - logits.max(1, keepdims=True)
    e = np.exp(z)
    return (e[:, 1] / e.sum(1)).astype(np.float64)


def _write_rows(path: str, header: List[str], rows: List[List]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
    os.replace(tmp, path)


def _strip(rep: Dict) -> Dict:
    return {k: v for k, v in rep.items() if not k.startswith("_")}


def write_dev_outputs(run_dir: str, ids, G, Pr, rule, n_tokens, texts_snip: Dict[str, str],
                      label_ids, train_support, n_dev_only_pos: int = 0) -> Dict:
    rep = full_report(G, Pr, rule, n_tokens, label_ids, train_support, n_dev_only_pos)
    P = rep["_P"]
    dd = os.path.join(run_dir, "dev")
    os.makedirs(dd, exist_ok=True)
    np.savez_compressed(os.path.join(dd, "dev_probs.npz"), ids=np.asarray(ids),
                        gold=G.astype(np.uint8), probs=Pr.astype(np.float16),
                        label_ids=np.asarray(label_ids))
    pl = rep["_per_label"]
    _write_rows(os.path.join(dd, "per_label_dev.csv"), list(pl[0].keys()),
                [list(r.values()) for r in pl])
    lab = np.asarray(label_ids)
    err = []
    for k, i in enumerate(ids):
        g, p = G[k].astype(bool), P[k]
        if (g != p).any():
            err.append([i, " ".join(map(str, lab[g & ~p])), " ".join(map(str, lab[p & ~g])),
                        int(g.sum()), int(p.sum()), int(n_tokens[k]), texts_snip.get(i, "")])
    if not err:
        err.append(["", "", "", "", "", "", "no dev document with a wrong label set"])
    _write_rows(os.path.join(dd, "errors.csv"),
                ["id", "missed", "spurious", "n_gold", "n_pred", "n_tokens", "snippet"], err)
    out = _strip(rep)
    write_json(os.path.join(dd, "dev_report.json"), out)
    write_json(os.path.join(dd, "dev_metrics.json"),
               {"rule": out["rule"], "scores": out["scores"], "scores_global": out["scores_global"],
                "scores_at_0p5": out["scores_at_0p5"]})
    return rep


def write_submission(run_dir: str, test_ids: List[str], Pr, rule: Dict, label_ids) -> Dict:
    L = Pr.shape[1]
    P = apply_rule(Pr, rule_taus(rule, L), int(rule["kmax"]))
    lab = np.asarray(label_ids, dtype=np.int64)
    sub = os.path.join(run_dir, "submission")
    os.makedirs(sub, exist_ok=True)
    np.savez_compressed(os.path.join(sub, "test_probs.npz"), ids=np.asarray(test_ids),
                        probs=Pr.astype(np.float16), label_ids=lab)
    items = [{"id": str(i), "pred": sorted(int(x) for x in set(lab[P[k]].tolist()))}
             for k, i in enumerate(test_ids)]
    pj = os.path.join(sub, "prediction.json")
    with open(pj + ".tmp", "w", encoding="utf-8") as f:
        json.dump(items, f)
    os.replace(pj + ".tmp", pj)
    zp = os.path.join(sub, "submission.zip")
    with zipfile.ZipFile(zp + ".tmp", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    os.replace(zp + ".tmp", zp)
    rep = validate(zp, test_ids, [int(x) for x in lab])
    rep["rule"] = rule.get("name")
    for f in ("dev_probs.npz", "per_label_dev.csv", "errors.csv"):
        p = os.path.join(run_dir, "dev", f)
        rep[f"{f}_bytes"] = os.path.getsize(p) if os.path.exists(p) else 0
    rep["test_probs.npz_bytes"] = os.path.getsize(os.path.join(sub, "test_probs.npz"))
    empty = [k for k in rep if k.endswith("_bytes") and not rep[k]]
    if empty:
        rep["ok"] = False
        rep["errors"].append(f"empty output files: {empty}")
    write_json(os.path.join(sub, "validation_report.json"), rep)
    return rep


In [ ]:
%%writefile /kaggle/working/phase7/cjpe_train.py
"""
cjpe_train.py -- ONE training job (one model x one technique), run as a subprocess on ONE GPU
(the orchestrator sets CUDA_VISIBLE_DEVICES). Reads RUN/logs/run_config.json.

    python cjpe_train.py --run-dir RUN [--mode train|smoke|overfit|teacher_reuse]

train  : self-checks on the real device -> throughput pre-flight -> time plan (section 8.4)
         -> training with dev evaluation (2x per epoch), threshold tuning, early stopping,
         30-minute resumable checkpoints, time guard -> best.pt -> final dev report, test
         prediction, validated submission.zip -> DONE.json (or status DEGENERATE)
         (+ teacher logits for train/dev/test when run_config["is_teacher"])
smoke  : self-checks + 20-step throughput only (00 notebook)
overfit: 32 training documents must reach loss < 0.1 within 100 steps (selftest)
teacher_reuse : load a finished E1 run, write its teacher logits + dev report

Exit codes: 0 ok, 1 error, 2 does not fit the time budget, 4 too many NaN steps,
5 early-fail watchdog, 6 critical self-check failed, 7 OOM at one document per micro-batch.
Every state change is written to RUN/status.json (also from `finally`), so a crash always
leaves a diagnosable file (rule R3).
"""
import argparse
import contextlib
import copy
import json
import math
import os
import random
import sys
import time
import traceback
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn.functional as F

from cjpe_common import (DEFAULT_EPOCHS, MAX_TRAINABLE_PCT, SPEC, disk_guard, DiskFull,
                         log_to, read_json, write_json, now_str)
from lsi_ml import (apply_rule, choose_rule, macro, ml_loss, ml_scores, multihot, pos_weight,
                    prior_logit, rule_taus, sigmoid_np, targets)
from cjpe_kd import kd_loss, save_teacher_logits, teacher_map
from cjpe_models import (assert_one_device, build_doc_model, collate_docs, forward_batch,
                         load_trainable_state, model_size, rebuild_trained, trainable_state_dict)
from cjpe_peft import apply_technique, merge_model
from cjpe_predict import (batches_by_chunks, predict_logits, softmax1, write_dev_outputs,
                          write_submission)
from cjpe_tok import doc_chunks, load_cache
from ner_models import torch_load


class JobExit(Exception):
    def __init__(self, code: int, reason: str):
        super().__init__(reason)
        self.code, self.reason = code, reason


# =============================================================================================
# small helpers
# =============================================================================================
class Ctx:
    def __init__(self, run_dir: str, cfg: Dict):
        self.run_dir = run_dir
        self.cfg = cfg
        self.log_path = os.path.join(run_dir, "logs", "console.log")
        self.t0 = time.time()
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    def log(self, msg: str) -> None:
        log_to(self.log_path, msg, prefix=f"[{self.cfg.get('job', '?')}] ")

    def status(self, state: str, reason: str = "", **extra) -> None:
        p = os.path.join(self.run_dir, "status.json")
        st = read_json(p, {}) or {}
        hist = st.get("history", [])
        hist.append({"state": state, "time": now_str(), "reason": reason[:500]})
        st.update({"state": state, "reason": reason[:2000], "time": now_str(),
                   "history": hist[-50:], **extra})
        write_json(p, st)

    def jlog(self, rec: Dict) -> None:
        with open(os.path.join(self.run_dir, "logs", "train_log.jsonl"), "a") as f:
            f.write(json.dumps(rec) + "\n")


def amp_setup(device):
    """The Phase-1 AMP pattern: fp16 autocast + GradScaler on T4 (no bf16), bf16 on Ampere+."""
    if device.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8:
        dtype = torch.bfloat16
    elif device.type == "cuda":
        dtype = torch.float16
    else:
        dtype = None

    def amp_ctx():
        if dtype is None:
            return contextlib.nullcontext()
        return torch.autocast(device_type="cuda", dtype=dtype)
    return dtype, amp_ctx


def make_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def is_oom(e: BaseException) -> bool:
    return "out of memory" in str(e).lower()


def peak_gb() -> float:
    return torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0


def set_seed(s: int) -> None:
    random.seed(s)
    np.random.seed(s % (2 ** 32))
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


# =============================================================================================
# data
# =============================================================================================
def load_split_docs(cfg: Dict, K: Optional[int] = None):
    """{split: {"ids", "labels", "chunks", "n_tokens"}} for the configured representation."""
    cache = load_cache(cfg["cache_path"])
    K = int(K or cfg["K"])
    lab = [int(x) for x in cfg["label_ids"]]
    lab2i = {x: i for i, x in enumerate(lab)}
    out = {}
    for split in ("train", "dev", "test"):
        rows = cache[split]
        labs = [list(r["label"] or []) if split != "test" else [] for r in rows]
        out[split] = {"ids": [r["id"] for r in rows],
                      "labels": multihot(labs, lab2i, len(lab)),
                      "dev_only_pos": sum(1 for l in labs for x in l if int(x) not in lab2i),
                      "chunks": [doc_chunks(r, cfg["doc_repr"], K) for r in rows],
                      "n_tokens": [int(r["n_tokens"]) for r in rows]}
    return out


def assert_data_integrity(d: Dict, cfg: Dict) -> Dict:
    """Section 4.3: training ids are train ids only, disjoint from dev and test."""
    tr, dv, te = set(d["train"]["ids"]), set(d["dev"]["ids"]), set(d["test"]["ids"])
    assert not (tr & te), f"{len(tr & te)} training ids are test ids"
    assert not (tr & dv), f"{len(tr & dv)} training ids are dev ids"
    L = len(cfg["label_ids"])
    for s_ in ("train", "dev", "test"):
        assert d[s_]["labels"].shape == (len(d[s_]["ids"]), L), f"{s_}: label matrix shape"
    assert int(d["train"]["labels"].sum()) > 0, "train has no positive labels"
    for s in ("train", "dev", "test"):
        for ch in d[s]["chunks"][:2000]:
            assert ch, f"{s}: a document has no chunk"
            assert all(len(c) <= 1024 for c in ch), f"{s}: sequence > 1024 tokens"
            assert all(int(c.max()) < 32000 for c in ch), f"{s}: token id >= 32000"
    return {"train_ids_subset_of_train_json": True, "train_x_dev": 0, "train_x_test": 0}


def stratified_subsample(labels, frac: float, seed: int) -> List[int]:
    """Seeded subsample of documents; every document that carries a rare label (train support
    < 5) is always kept so rare statutes are not lost when the time budget forces frac < 1."""
    n = len(labels)
    if frac >= 0.999:
        return list(range(n))
    rng = np.random.default_rng(seed)
    keep = set(rng.choice(n, max(1, int(round(frac * n))), replace=False).tolist())
    Y = np.asarray(labels)
    if Y.ndim == 2 and n:
        rare = Y.sum(0) < 5
        if rare.any():
            keep |= set(np.where(Y[:, rare].any(1))[0].tolist())
    return sorted(keep)


def class_weights(labels, mode: str, device) -> Optional[torch.Tensor]:
    """Per-label BCE pos_weight: clip(sqrt(neg/pos), 1, 30) (mode sqrt_inv), neg/pos (inv)."""
    w = pos_weight(np.asarray(labels), "none" if mode == "none" else
                   ("inv" if mode == "inv" else "sqrt_inv"))
    return None if w is None else torch.tensor(w, dtype=torch.float32, device=device)


# =============================================================================================
# self-checks on the REAL device (lesson L1 / L2)
# =============================================================================================
def job_checks(base: Optional[torch.nn.Module], model, tech, cfg: Dict, d: Dict, device,
               amp_ctx) -> Dict:
    """critical: device placement, output shape, finite loss and gradients, data sanity.
    advisory: zero-init equals the base, merged equals unmerged, trainable-% cap."""
    res = {"checks": [], "critical_failed": [], "advisory_failed": []}

    def add(name, kind, fn):
        rec = {"name": name, "kind": kind}
        try:
            val = fn()
            if isinstance(val, dict):
                rec.update(val)
            rec.setdefault("verdict", "pass")
        except Exception as e:
            rec["verdict"] = "fail"
            rec["error"] = f"{type(e).__name__}: {e}"[:1500]
            rec["trace"] = traceback.format_exc()[-1500:]
        if rec["verdict"] != "pass":
            (res["critical_failed"] if kind == "critical" else res["advisory_failed"]).append(name)
        res["checks"].append(rec)

    tr = d["train"]
    idx = [i for i in range(min(len(tr["ids"]), 50))][:2]
    b = collate_docs([tr["chunks"][i] for i in idx], device)
    y = targets(tr["labels"], idx, device)
    L = len(cfg["label_ids"])

    add("all_tensors_on_one_device", "critical", lambda: (assert_one_device(model, device),
                                                         {"device": str(device)})[1])

    def fwd():
        model.train()
        with amp_ctx():
            logits, w = forward_batch(model, b)
        assert logits.shape == (len(idx), L), f"logits {tuple(logits.shape)}"
        assert torch.isfinite(logits).all(), "non-finite logits"
        loss = ml_loss(logits.float(), y).mean()
        ex = tech.extra_loss(model)
        if ex is not None:
            loss = loss + ex
        assert torch.isfinite(loss), "non-finite loss"
        loss.backward()
        gr = [p.grad for p in model.parameters() if p.requires_grad]
        assert any(g is not None and float(g.abs().sum()) > 0 for g in gr), "no gradient"
        assert all(g is None or bool(torch.isfinite(g).all()) for g in gr), "non-finite grad"
        model.zero_grad(set_to_none=True)
        wz = w[~(torch.arange(w.shape[1], device=w.device)[None] <
                 torch.bincount(b["doc_index"], minlength=w.shape[0])[:, None])]
        assert float(wz.detach().abs().sum()) == 0.0, "masked chunk got attention weight"
        return {"loss": float(loss), "logits_shape": list(logits.shape)}
    add("forward_backward_finite", "critical", fwd)

    def tiny_batch():
        ch = [[c[:64] for c in tr["chunks"][i][:1]] for i in idx]
        return collate_docs(ch, device)

    if base is not None and cfg["technique"] in ("lorma", "lora", "adalora", "adapters"):
        def zero_init():
            tb = tiny_batch()
            model.eval()
            base.eval()
            with torch.no_grad():
                a = model.chunk_vectors(tb["input_ids"], tb["attention_mask"])
                c = base.chunk_vectors(tb["input_ids"], tb["attention_mask"])
            diff = float((a - c).abs().max())
            scale = max(1.0, float(c.abs().max()))
            thr = (5e-2 if cfg["technique"] == "adapters" else 1e-4) * scale
            return {"value": diff, "threshold": thr, "verdict": "pass" if diff <= thr else "warn"}
        add("zero_init_equals_base", "advisory", zero_init)
    if cfg["technique"] in ("lorma", "lora", "adalora"):
        def merge_eq():
            m = copy.deepcopy(model)
            g = torch.Generator().manual_seed(7)
            with torch.no_grad():
                for n, p in m.named_parameters():
                    if any(t in n for t in ("lora_B", "lorma_B", "ada_E")):
                        p.add_((torch.randn(p.shape, generator=g) * 0.02).to(p.device, p.dtype))
            tb = tiny_batch()
            m.eval()
            with torch.no_grad():
                u = m.chunk_vectors(tb["input_ids"], tb["attention_mask"])
                n_merged = merge_model(m)
                v = m.chunk_vectors(tb["input_ids"], tb["attention_mask"])
            diff = float((u - v).abs().max())
            thr = 1e-3 * max(1.0, float(u.abs().max()))
            del m
            return {"value": diff, "threshold": thr, "merged_layers": n_merged,
                    "verdict": "pass" if diff <= thr else "warn"}
        add("merged_equals_unmerged", "advisory", merge_eq)
    s = model_size(model)
    real = s["total_params"] >= SPEC["rules"]["foundation_params"][0] * 0.4
    if cfg["technique"] in MAX_TRAINABLE_PCT and real:
        cap = MAX_TRAINABLE_PCT[cfg["technique"]]
        add("trainable_pct_cap", "advisory",
            lambda: {"value": s["trainable_pct"], "threshold": cap,
                     "verdict": "pass" if s["trainable_pct"] <= cap else "warn"})
    model.zero_grad(set_to_none=True)
    res["ok"] = not res["critical_failed"]
    res["selfcheck"] = "fail" if res["critical_failed"] else (
        "warn" if res["advisory_failed"] else "pass")
    return res


# =============================================================================================
# one optimizer step (with OOM back-off)
# =============================================================================================
class Stepper:
    def __init__(self, ctx, model, opt, sched, scaler, tech, amp_ctx, cw, cfg, tmap=None):
        self.ctx, self.model, self.opt, self.sched, self.scaler = ctx, model, opt, sched, scaler
        self.tech, self.amp_ctx, self.cw, self.cfg, self.tmap = tech, amp_ctx, cw, cfg, tmap
        self.micro = int(cfg["micro_chunks"])
        self.skipped = 0
        self.overflow = 0
        self.params = [p for p in model.parameters() if p.requires_grad]

    def loss_of(self, logits, y, ids):
        if self.tmap is not None:
            t = torch.tensor(np.stack([self.tmap[i] for i in ids]), device=logits.device)
            return kd_loss(logits, t, y, float(self.cfg.get("kd_alpha", 0.5)),
                           float(self.cfg.get("kd_T", 2.0)), self.cw) * len(ids)
        per = ml_loss(logits, y, self.cw, self.cfg.get("loss", "bce"))
        return per.sum()

    def step(self, data, doc_idx: List[int], global_step: int, total_steps: int):
        """Forward/backward over `doc_idx` (one effective batch) + optimizer step.
        Returns (mean loss, tokens) or (None, tokens) for a skipped non-finite step."""
        while True:
            try:
                return self._step(data, doc_idx, global_step, total_steps)
            except RuntimeError as e:
                if not is_oom(e):
                    raise
                self.opt.zero_grad(set_to_none=True)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                one_doc = max(len(data["chunks"][i]) for i in doc_idx)
                if self.micro <= one_doc:
                    raise JobExit(7, "CUDA OOM with one document per micro-batch")
                self.micro = max(one_doc, self.micro // 2)
                self.ctx.log(f"   [oom] micro-batch -> {self.micro} chunks (accumulation grows)")

    def _step(self, data, doc_idx, global_step, total_steps):
        model = self.model
        model.train()
        n = len(doc_idx)
        tot, toks = 0.0, 0
        for g in batches_by_chunks(data["chunks"], self.micro, doc_idx):
            b = collate_docs([data["chunks"][i] for i in g], self.ctx.device)
            y = targets(data["labels"], g, self.ctx.device)
            with self.amp_ctx():
                logits, _ = forward_batch(model, b)
            loss = self.loss_of(logits, y, [data["ids"][i] for i in g]) / n
            ex = self.tech.extra_loss(model)
            if ex is not None:
                loss = loss + ex * (len(g) / n)
            self.scaler.scale(loss).backward()
            tot += float(loss.detach())
            toks += b["n_tokens"]
        self.scaler.unscale_(self.opt)
        gn = torch.nn.utils.clip_grad_norm_(self.params, float(self.cfg.get("grad_clip", 1.0)))
        if not math.isfinite(tot):
            # a non-finite LOSS is a real problem (counted; > 5% of steps aborts the job)
            self.skipped += 1
            self.opt.zero_grad(set_to_none=True)
            self.scaler.update()
            return None, toks
        if not torch.isfinite(gn):
            # finite loss but inf/nan gradients = normal fp16 loss-scale overflow: GradScaler
            # skips the optimizer step itself and lowers the scale (not counted as a failure)
            self.overflow += 1
        self.tech.after_backward(model, global_step, total_steps)
        self.scaler.step(self.opt)
        self.scaler.update()
        self.opt.zero_grad(set_to_none=True)
        if self.sched is not None:
            self.sched.step()
        return tot, toks


def make_optimizer(groups):
    for g in groups:
        g["initial_lr"] = g["lr"]
    return torch.optim.AdamW([{k: v for k, v in g.items() if k != "name"} for g in groups],
                             betas=(0.9, 0.98), eps=1e-8)


def make_scheduler(opt, total_steps: int, warmup_frac: float = 0.06):
    warm = max(1, int(round(warmup_frac * total_steps)))

    def f(step):
        if step < warm:
            return (step + 1) / warm
        return max(0.0, 1.0 - (step - warm) / max(1, total_steps - warm))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


# =============================================================================================
# throughput + time plan (section 8.4)
# =============================================================================================
def measure_throughput(ctx, stepper, d, n_steps: int = 6):
    """Training tokens/s with real batches (no optimizer change: grads are dropped) and
    inference tokens/s."""
    model = stepper.model
    tr = d["train"]
    eff = int(ctx.cfg["effective_batch"])
    rng = np.random.default_rng(1)
    order = rng.permutation(len(tr["ids"])).tolist()
    sel = order[:max(1, min(len(order), eff))]
    params = stepper.params

    def one():
        toks = 0
        for g in batches_by_chunks(tr["chunks"], stepper.micro, sel):
            b = collate_docs([tr["chunks"][i] for i in g], ctx.device)
            y = targets(tr["labels"], g, ctx.device)
            with stepper.amp_ctx():
                logits, _ = forward_batch(model, b)
            stepper.scaler.scale(ml_loss(logits.float(), y).mean()).backward()
            toks += b["n_tokens"]
        for p in params:
            p.grad = None
        return toks
    model.train()
    while True:
        try:
            one()                                  # warm-up (allocations, kernels)
            break
        except RuntimeError as e:
            if not is_oom(e):
                raise
            for p in params:
                p.grad = None
            torch.cuda.empty_cache()
            one_doc = max(len(tr["chunks"][i]) for i in sel)
            if stepper.micro <= one_doc:
                raise JobExit(7, "CUDA OOM with one document per micro-batch (pre-flight)")
            stepper.micro = max(one_doc, stepper.micro // 2)
            ctx.log(f"   [oom] pre-flight: micro-batch -> {stepper.micro} chunks")
    if ctx.device.type == "cuda":
        torch.cuda.synchronize()
    t = time.time()
    toks = 0
    for _ in range(n_steps):
        toks += one()
    if ctx.device.type == "cuda":
        torch.cuda.synchronize()
    r_train = toks / max(1e-6, time.time() - t)
    # inference speed
    dv = d["dev"]
    sub = list(range(min(len(dv["ids"]), 64)))
    predict_logits(model, [dv["chunks"][i] for i in sub[:8]], ctx.device, stepper.amp_ctx,
                   int(ctx.cfg["infer_chunks"]))                     # warm-up
    if ctx.device.type == "cuda":
        torch.cuda.synchronize()
    t = time.time()
    ntok = sum(sum(len(c) for c in dv["chunks"][i]) for i in sub)
    predict_logits(model, [dv["chunks"][i] for i in sub], ctx.device, stepper.amp_ctx,
                   int(ctx.cfg["infer_chunks"]))
    if ctx.device.type == "cuda":
        torch.cuda.synchronize()
    r_inf = ntok / max(1e-6, time.time() - t)
    return r_train, max(r_inf, r_train)


def tokens_of(chunks, idx=None) -> int:
    it = chunks if idx is None else (chunks[i] for i in idx)
    return sum(sum(len(c) for c in ch) for ch in it)


def make_plan(ctx, d_by_K: Dict[int, Dict], r_train: float, r_inf: float) -> Dict:
    cfg = ctx.cfg
    budget = float(cfg["budget_s"])
    used = time.time() - ctx.t0
    margin = max(150.0, 0.05 * budget)
    avail = budget - used - margin
    e_max = int(cfg.get("epochs_max") or DEFAULT_EPOCHS.get(cfg["technique"], 4))
    e_min = int(cfg.get("min_epochs", 2))
    evals = int(cfg.get("evals_per_epoch", 2))
    rows = []
    choice = None
    fixed = cfg.get("fixed_plan")
    for K in sorted(d_by_K, reverse=True):
        d = d_by_K[K]
        ep = tokens_of(d["train"]["chunks"]) / r_train
        dev_inf = tokens_of(d["dev"]["chunks"]) / r_inf
        once = (tokens_of(d["test"]["chunks"]) + tokens_of(d["dev"]["chunks"])) / r_inf
        if cfg.get("is_teacher"):
            once += tokens_of(d["train"]["chunks"]) / r_inf
        per_epoch = ep + evals * dev_inf
        e_fit = int(max(0.0, avail - once) // max(1e-6, per_epoch))
        rows.append({"K": K, "epoch_min": per_epoch / 60, "inference_min": once / 60,
                     "epochs_fit": e_fit})
        if fixed:
            if K == max(d_by_K):
                choice = {"K": K, "epochs": int(fixed["epochs"]),
                          "train_fraction": float(fixed.get("frac", 1.0)), "reason": "fixed plan"}
            continue
        if choice is None and e_fit >= e_min:
            choice = {"K": K, "epochs": min(e_max, e_fit), "train_fraction": 1.0,
                      "reason": "fits" if K == max(d_by_K) else "K reduced to fit"}
    if choice is None:
        K = min(d_by_K)
        r = rows[-1]
        per_epoch = r["epoch_min"] * 60
        frac = (avail - r["inference_min"] * 60) / max(1e-6, e_min * per_epoch)
        min_frac = float(cfg.get("min_train_fraction", 0.3))
        if frac >= min_frac:
            choice = {"K": K, "epochs": e_min, "train_fraction": round(min(1.0, frac), 3),
                      "reason": "stratified subsample to fit"}
        else:
            e = 1 if (avail - r["inference_min"] * 60) >= min_frac * per_epoch else 0
            if e == 0:
                raise JobExit(2, f"not even {min_frac:.0%} of one epoch fits: available "
                                 f"{avail / 60:.1f} min, one epoch {per_epoch / 60:.1f} min")
            choice = {"K": K, "epochs": 1, "train_fraction": min_frac,
                      "reason": "minimum subsample, 1 epoch (time guard will cut if needed)"}
    d = d_by_K[choice["K"]]
    n_tr = len(d["train"]["ids"])
    n_used = len(stratified_subsample(d["train"]["labels"], choice["train_fraction"],
                                      int(cfg["seed"])))
    proj = (choice["epochs"] * (tokens_of(d["train"]["chunks"]) * n_used / max(1, n_tr) / r_train
                                + evals * tokens_of(d["dev"]["chunks"]) / r_inf)
            + (tokens_of(d["test"]["chunks"]) + tokens_of(d["dev"]["chunks"])) / r_inf + used)
    plan = {**choice, "K_requested": int(cfg["K"]), "budget_min": budget / 60,
            "available_min": avail / 60, "tokens_per_s_train": r_train,
            "tokens_per_s_infer": r_inf, "train_docs": n_tr, "train_docs_used": n_used,
            "projected_minutes": proj / 60, "options": rows, "epochs_max": e_max,
            "epochs_min": e_min, "micro_chunks": None}
    return plan


def print_plan(ctx, plan: Dict) -> None:
    ctx.log(f"   [plan] budget {plan['budget_min']:.0f} min, available {plan['available_min']:.0f} "
            f"min, train {plan['tokens_per_s_train']:,.0f} tok/s, infer "
            f"{plan['tokens_per_s_infer']:,.0f} tok/s")
    ctx.log("   [plan]   K  epoch(min)  inference(min)  epochs that fit")
    for r in plan["options"]:
        ctx.log(f"   [plan]  {r['K']:>2}  {r['epoch_min']:>10.1f}  {r['inference_min']:>14.1f}  "
                f"{r['epochs_fit']:>15}")
    flag = "" if plan["reason"] == "fits" else "   <<< " + plan["reason"].upper()
    ctx.log(f"   [plan] CHOSEN: K={plan['K']} epochs={plan['epochs']} train_fraction="
            f"{plan['train_fraction']} ({plan['train_docs_used']}/{plan['train_docs']} docs), "
            f"projected {plan['projected_minutes']:.0f} min{flag}")


# =============================================================================================
# evaluation during training
# =============================================================================================
def dev_eval(ctx, model, d, amp_ctx) -> Dict:
    """sigmoid probabilities on dev -> dev-only decision rule (global tau / cross-fitted
    per-label taus, >= 1 and <= kmax labels per document) -> Macro-F1."""
    dv = d["dev"]
    logits = predict_logits(model, dv["chunks"], ctx.device, amp_ctx,
                            int(ctx.cfg["infer_chunks"]))
    Pr = sigmoid_np(logits)
    G = dv["labels"]
    kmax = int(ctx.cfg.get("kmax", 10))
    rule = choose_rule(Pr, G, kmax, int(ctx.cfg["seed"]),
                       allow_perlabel=bool(ctx.cfg.get("tau_scan", True)))
    P = apply_rule(Pr, rule_taus(rule, Pr.shape[1]), kmax)
    s = ml_scores(G, P)
    s["accuracy"] = s["subset_acc"]
    L = Pr.shape[1]
    loss = float(ml_loss(torch.from_numpy(logits), torch.from_numpy(G.astype(np.float32)))
                 .mean().item())
    return {"p1": Pr, "logits": logits, "tau": float(rule["tau_global"]), "rule": rule,
            "scores": s, "f1": s["macro_f1"],
            "f1_at_0p5": macro(G, apply_rule(Pr, np.full(L, 0.5, np.float32), kmax)),
            "dev_loss": loss, "p1_std": float(Pr.std())}


# =============================================================================================
# checkpoints
# =============================================================================================
def save_best(ctx, model) -> None:
    p = os.path.join(ctx.run_dir, "checkpoints", "best.pt")
    os.makedirs(os.path.dirname(p), exist_ok=True)
    full = ctx.cfg["technique"] in ("full_ft", "kd", "student_hard", "partial_ft")
    torch.save(trainable_state_dict(model, half=full), p + ".tmp")
    os.replace(p + ".tmp", p)


def save_last(ctx, model, opt, sched, scaler, st: Dict) -> None:
    p = os.path.join(ctx.run_dir, "checkpoints", "last.pt")
    os.makedirs(os.path.dirname(p), exist_ok=True)
    try:
        disk_guard(os.path.dirname(ctx.run_dir), 3.0, log=ctx.log,
                   keep=os.path.join(ctx.run_dir, "checkpoints"))
    except DiskFull as e:
        ctx.log(f"   [disk] {e}: skipping last.pt")
        return
    obj = {"model": trainable_state_dict(model), "opt": opt.state_dict(),
           "sched": sched.state_dict(), "scaler": scaler.state_dict(), "st": st,
           "rng": {"py": random.getstate(), "np": np.random.get_state(),
                   "torch": torch.get_rng_state()}}
    torch.save(obj, os.path.join(ctx.run_dir, "checkpoints", "last.tmp"))
    os.replace(os.path.join(ctx.run_dir, "checkpoints", "last.tmp"), p)


# =============================================================================================
# the job
# =============================================================================================
def run_train(ctx: Ctx, mode: str) -> int:
    cfg = ctx.cfg
    set_seed(int(cfg["seed"]))
    dev = ctx.device
    amp_dtype, amp_ctx = amp_setup(dev)
    ctx.status("preflight")
    ctx.log(f"start {cfg['experiment']}/{cfg['technique']}/{cfg['model']} on {dev}"
            f"{' (' + torch.cuda.get_device_name(0) + ')' if dev.type == 'cuda' else ''}, "
            f"budget {cfg['budget_s'] / 60:.0f} min, attempt {cfg.get('attempt', 1)}")
    if dev.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    inj = os.environ.get("PHASE5_INJECT_FAIL", "")          # tests only: "<job>:<exit code>"
    if inj and inj.split(":")[0] == cfg["job"] and int(cfg.get("attempt", 1)) == 1:
        raise JobExit(int(inj.split(":")[1]), "injected failure (test)")

    # ---- data (all candidate K values, for the planner) ---------------------------------
    K0 = int(cfg["K"])
    reducible = cfg["doc_repr"] in ("hier", "first_k", "last_k")
    Ks = [K0] + ([k for k in (2, 1) if k < K0] if reducible and mode == "train" else [])
    d_by_K = {K: load_split_docs(cfg, K) for K in Ks}
    d = d_by_K[K0]
    integ = assert_data_integrity(d, cfg)
    write_json(os.path.join(ctx.run_dir, "logs", "integrity.json"), integ)
    ctx.log(f"   data: train {len(d['train']['ids'])}, dev {len(d['dev']['ids'])}, test "
            f"{len(d['test']['ids'])} docs; repr {cfg['doc_repr']} K={K0}; integrity ok")

    # ---- teacher logits (KD students) ---------------------------------------------------
    tmap = None
    if cfg["technique"] == "kd":
        tl = cfg.get("teacher_logits")
        if not tl or not os.path.exists(tl):
            raise JobExit(1, f"teacher logits not found ({tl})")
        tmap = teacher_map(tl, "train")
        miss = [i for i in d["train"]["ids"] if i not in tmap]
        if miss:
            raise JobExit(1, f"{len(miss)} train docs have no teacher logits")

    # ---- model ---------------------------------------------------------------------------
    model, _ = build_doc_model(cfg["phase1_dir"], cfg["attn_mode"], cfg.get("layer_ids"),
                               float(cfg.get("dropout", 0.1)), len(cfg["label_ids"]),
                               prior_logit(d["train"]["labels"]))
    need_base = (not cfg.get("skip_job_checks")) and cfg["technique"] in (
        "lorma", "lora", "adalora", "adapters")
    base = copy.deepcopy(model) if need_base else None
    groups, tech = apply_technique(model, cfg["technique"], cfg, log=ctx.log)
    model.to(dev)
    assert_one_device(model, dev)
    size = model_size(model)
    write_json(os.path.join(ctx.run_dir, "logs", "model_size.json"), size)

    checks = {"skipped": True, "selfcheck": "skipped"}
    if not cfg.get("skip_job_checks"):
        if base is not None:
            base.to(dev)
        checks = job_checks(base, model, tech, cfg, d, dev, amp_ctx)
        del base
        if dev.type == "cuda":
            torch.cuda.empty_cache()
    write_json(os.path.join(ctx.run_dir, "logs", "job_checks.json"), checks)
    for c in checks.get("checks", []):
        v = c.get("value")
        ctx.log(f"   [check] {c['name']:<28} {c['kind']:<9} {c['verdict']}"
                + (f" value={v:.3g} thr={c.get('threshold'):.3g}" if isinstance(v, float) else "")
                + (f"  {c.get('error', '')[:200]}" if c["verdict"] == "fail" else ""))
    if checks.get("critical_failed"):
        raise JobExit(6, f"critical self-check failed: {checks['critical_failed']}")
    if checks.get("advisory_failed"):
        ctx.log(f"   WARNING advisory self-checks: {checks['advisory_failed']} (continuing)")

    tr_all = d["train"]
    cw = class_weights(tr_all["labels"], cfg.get("class_weight", "sqrt_inv"), dev)
    opt = make_optimizer(groups)
    scaler = make_scaler(amp_dtype == torch.float16)
    stepper = Stepper(ctx, model, opt, None, scaler, tech, amp_ctx, cw, cfg, tmap)

    if mode == "overfit":
        return run_overfit(ctx, stepper, d)

    # ---- pre-flight throughput + plan ----------------------------------------------------
    plan_path = os.path.join(ctx.run_dir, "logs", "plan.json")
    last_pt = os.path.join(ctx.run_dir, "checkpoints", "last.pt")
    resume = None
    if os.path.exists(last_pt) and os.path.exists(plan_path) and cfg.get("resume", True):
        try:
            resume = torch_load(last_pt)
            plan = read_json(plan_path)
            ctx.log(f"   [resume] from step {resume['st']['step']} (plan reused)")
        except Exception as e:
            ctx.log(f"   [resume] unreadable last.pt ({e}); starting fresh")
            resume = None
    if resume is None:
        r_train, r_inf = measure_throughput(ctx, stepper, d)
        ctx.log(f"   [preflight] train {r_train:,.0f} tok/s, inference {r_inf:,.0f} tok/s, "
                f"micro-batch {stepper.micro} chunks, peak {peak_gb():.2f} GB")
        if mode == "smoke":
            write_json(os.path.join(ctx.run_dir, "logs", "smoke.json"),
                       {"tokens_per_s_train": r_train, "tokens_per_s_infer": r_inf,
                        "micro_chunks": stepper.micro, "peak_mem_gb": peak_gb(),
                        "selfcheck": checks.get("selfcheck"), **size})
            ctx.status("smoke_ok", "smoke ok")
            return 0
        plan = make_plan(ctx, d_by_K, r_train, r_inf)
        plan["micro_chunks"] = stepper.micro
        plan["selfcheck"] = checks.get("selfcheck")
        write_json(plan_path, plan)
    print_plan(ctx, plan)
    stepper.micro = int(plan.get("micro_chunks") or stepper.micro)
    if plan["K"] != K0:
        d = d_by_K[plan["K"]]
    del d_by_K

    keep = stratified_subsample(d["train"]["labels"], float(plan["train_fraction"]),
                                int(cfg["seed"]))
    eff = int(cfg["effective_batch"])
    steps_per_epoch = max(1, math.ceil(len(keep) / eff))
    total_steps = steps_per_epoch * int(plan["epochs"])
    sched = make_scheduler(opt, total_steps, float(cfg.get("warmup_frac", 0.06)))
    stepper.sched = sched
    evals = int(cfg.get("evals_per_epoch", 2))
    eval_at = sorted({max(1, round(steps_per_epoch * (k + 1) / evals)) for k in range(evals)})
    patience = int(cfg.get("patience", 2)) * evals

    st = {"step": 0, "epoch": 0, "pos": 0, "best_f1": -1.0, "best_epoch": None, "bad": 0,
          "evals": [], "skipped": 0, "train_seconds": 0.0}
    if resume is not None:
        load_trainable_state(model, resume["model"])
        opt.load_state_dict(resume["opt"])
        sched.load_state_dict(resume["sched"])
        scaler.load_state_dict(resume["scaler"])
        st.update(resume["st"])
        try:
            random.setstate(resume["rng"]["py"])
            np.random.set_state(resume["rng"]["np"])
            torch.set_rng_state(resume["rng"]["torch"])
        except Exception:
            pass
        with open(os.path.join(ctx.run_dir, "logs", "resume.json"), "w") as f:
            json.dump({"resumed_at_step": st["step"]}, f)
        del resume

    # inference reserve after the training deadline: final dev + test (+ teacher train logits)
    r_inf = float(plan["tokens_per_s_infer"])
    reserve = (tokens_of(d["dev"]["chunks"]) + tokens_of(d["test"]["chunks"])) / r_inf
    if cfg.get("is_teacher"):
        reserve += tokens_of(d["train"]["chunks"]) / r_inf
    reserve = 1.25 * reserve + 90.0
    deadline = ctx.t0 + float(cfg["budget_s"]) - reserve
    ctx.status("training", plan=plan["reason"])
    ctx.log(f"   [train] {len(keep)} docs/epoch, {steps_per_epoch} steps/epoch x "
            f"{plan['epochs']} epochs = {total_steps} steps; eval at steps {eval_at} of each "
            f"epoch; training deadline in {(deadline - time.time()) / 60:.0f} min")

    last_ckpt = time.time()
    t_train = time.time()
    stop_reason = "completed"
    kill_at = int(cfg.get("kill_at_step", -1))
    ema = None
    while st["epoch"] < int(plan["epochs"]):
        rng = np.random.default_rng(int(cfg["seed"]) + 1000 * st["epoch"])
        order = [keep[i] for i in rng.permutation(len(keep))]
        while st["pos"] < steps_per_epoch:
            if time.time() > deadline:
                stop_reason = "time guard"
                break
            s_in = st["pos"]
            batch = order[s_in * eff:(s_in + 1) * eff]
            if not batch:
                st["pos"] = steps_per_epoch
                break
            loss, toks = stepper.step(d["train"], batch, st["step"], total_steps)
            st["step"] += 1
            st["pos"] += 1
            if loss is None:
                st["skipped"] += 1
                if st["step"] >= 20 and st["skipped"] > 0.05 * st["step"]:
                    raise JobExit(4, f"{st['skipped']} of {st['step']} steps non-finite")
            else:
                ema = loss if ema is None else 0.98 * ema + 0.02 * loss
            if st["step"] % 50 == 0:
                ctx.log(f"   step {st['step']}/{total_steps} ep {st['epoch'] + 1} loss "
                        f"{ema:.4f} lr {sched.get_last_lr()[0]:.2e} peak {peak_gb():.1f} GB "
                        f"skipped {st['skipped']} fp16-overflow {stepper.overflow}")
            if st["step"] == kill_at:
                save_last(ctx, model, opt, sched, scaler, st)
                os._exit(137)
            if st["pos"] in eval_at:
                ev = dev_eval(ctx, model, d, amp_ctx)
                frac_ep = st["epoch"] + st["pos"] / steps_per_epoch
                rec = {"type": "eval", "step": st["step"], "epoch": round(frac_ep, 3),
                       "train_loss": ema, "dev_loss": ev["dev_loss"], "dev_macro_f1": ev["f1"],
                       "dev_acc": ev["scores"]["accuracy"], "tau": ev["tau"],
                       "f1_at_0p5": ev["f1_at_0p5"], "lr": sched.get_last_lr()[0],
                       "minutes": (time.time() - ctx.t0) / 60, "skipped_steps": st["skipped"]}
                ctx.jlog(rec)
                st["evals"].append(rec)
                sel = tech.selectable(st["step"], total_steps)
                ctx.log(f"   [eval] epoch {frac_ep:.2f}: dev macro-F1 {ev['f1']:.4f} (tau "
                        f"{ev['tau']:.2f}; @0.5 {ev['f1_at_0p5']:.4f}) loss {ev['dev_loss']:.4f}"
                        f"{'' if sel else ' (not selectable yet)'}")
                if sel and ev["f1"] > st["best_f1"] + 1e-9:
                    st["best_f1"], st["best_epoch"], st["bad"] = ev["f1"], round(frac_ep, 3), 0
                    save_best(ctx, model)
                elif sel:
                    st["bad"] += 1
                write_json(os.path.join(ctx.run_dir, "logs", "progress.json"),
                           {"best_dev_macro_f1": st["best_f1"], "best_epoch": st["best_epoch"],
                            "step": st["step"], "total_steps": total_steps,
                            "epochs_run": round(frac_ep, 3)})
                if st["epoch"] == 0 and st["pos"] == eval_at[-1] and ev["f1"] < 0.40 \
                        and ev["p1_std"] < 1e-3:
                    raise JobExit(5, f"watchdog: dev macro-F1 {ev['f1']:.3f} with a constant "
                                     f"output after epoch 1")
                if sel and st["bad"] >= patience:
                    stop_reason = "early stopping"
                    break
            if time.time() - last_ckpt > 60 * float(cfg.get("ckpt_every_min", 30)):
                save_last(ctx, model, opt, sched, scaler, st)
                last_ckpt = time.time()
        if stop_reason != "completed":
            break
        st["epoch"] += 1
        st["pos"] = 0
    st["train_seconds"] = time.time() - t_train
    ctx.log(f"   [train] stopped: {stop_reason} at step {st['step']}/{total_steps}")

    # ---- final model ---------------------------------------------------------------------
    best_pt = os.path.join(ctx.run_dir, "checkpoints", "best.pt")
    tech.force_final(model)
    if not os.path.exists(best_pt) or (cfg["technique"] == "adalora" and st["best_f1"] < 0):
        ev = dev_eval(ctx, model, d, amp_ctx)
        st["best_f1"], st["best_epoch"] = ev["f1"], st["epoch"] + st["pos"] / steps_per_epoch
        save_best(ctx, model)
        ctx.log(f"   [final] no selectable checkpoint during training -> current weights "
                f"(dev macro-F1 {ev['f1']:.4f})")
    load_trainable_state(model, torch_load(best_pt))
    lp = os.path.join(ctx.run_dir, "checkpoints", "last.pt")
    if os.path.exists(lp):
        os.remove(lp)
    return finish(ctx, model, tech, d, amp_ctx, st, plan, size, checks, stop_reason)


def finish(ctx, model, tech, d, amp_ctx, st, plan, size, checks, stop_reason) -> int:
    cfg = ctx.cfg
    ctx.status("evaluating")
    t = time.time()
    ev = dev_eval(ctx, model, d, amp_ctx)
    snips = load_snippets(cfg)
    label_ids = [int(x) for x in cfg["label_ids"]]
    tsup = d["train"]["labels"].sum(0)
    rep = write_dev_outputs(ctx.run_dir, d["dev"]["ids"], d["dev"]["labels"], ev["p1"],
                            ev["rule"], d["dev"]["n_tokens"], snips, label_ids, tsup,
                            int(d["dev"].get("dev_only_pos", 0)))
    ctx.status("predicting")
    tl = predict_logits(model, d["test"]["chunks"], ctx.device, amp_ctx, int(cfg["infer_chunks"]))
    dt = time.time() - t
    p_test = sigmoid_np(tl)
    vrep = write_submission(ctx.run_dir, d["test"]["ids"], p_test, ev["rule"], label_ids)
    write_json(os.path.join(ctx.run_dir, "dev", "rule.json"), ev["rule"])
    if cfg.get("is_teacher"):
        trl = predict_logits(model, d["train"]["chunks"], ctx.device, amp_ctx,
                             int(cfg["infer_chunks"]))
        save_teacher_logits(os.path.join(ctx.run_dir, "teacher_logits.npz"),
                            {"train": {"ids": d["train"]["ids"], "logits": trl},
                             "dev": {"ids": d["dev"]["ids"], "logits": ev["logits"]},
                             "test": {"ids": d["test"]["ids"], "logits": tl}})
        ctx.log("   [teacher] logits cached for train / dev / test")
    n_docs = len(d["dev"]["ids"]) + len(d["test"]["ids"])
    s = rep["scores"]
    r = ev["rule"]
    metrics = {
        "dev": s, "dev_macro_f1": s["macro_f1"], "dev_macro_f1_at_0p5":
            rep["scores_at_0p5"]["macro_f1"], "dev_macro_f1_global":
            rep["scores_global"]["macro_f1"], "rule": r["name"], "tau": r["tau_global"],
        "kmax": r["kmax"], "cv_macro_global": r.get("cv_macro_global"),
        "cv_macro_perlabel": r.get("cv_macro_perlabel"), "ece": rep["ece"],
        "n_labels": len(label_ids), "dev_only_labels": int(d["dev"].get("dev_only_pos", 0)),
        "best_epoch": st["best_epoch"],
        "epochs_run": st["evals"][-1]["epoch"] if st["evals"] else 0,
        "steps": st["step"], "skipped_steps": st["skipped"], "stop_reason": stop_reason,
        "train_minutes": st["train_seconds"] / 60, "total_minutes": (time.time() - ctx.t0) / 60,
        "docs_per_s_infer": n_docs / max(1e-6, dt), "ms_per_doc_infer": 1000 * dt / max(1, n_docs),
        "peak_mem_gb": peak_gb(), "test_mean_labels": vrep.get("mean_labels"),
        "K_effective": plan["K"],
        "train_fraction": plan["train_fraction"], "epochs_planned": plan["epochs"],
        "train_docs_used": plan["train_docs_used"],
        "tokens_per_s": plan["tokens_per_s_train"], "projected_minutes":
            plan["projected_minutes"], "selfcheck": checks.get("selfcheck"),
        "validation_ok": vrep["ok"], "degenerate": vrep.get("degenerate"),
        "technique_report": tech.final_report(model), **size,
    }
    write_json(os.path.join(ctx.run_dir, "metrics.json"), metrics)
    if not vrep["ok"]:
        ctx.status("failed", f"submission validation failed: {vrep['errors']}")
        return 4
    if vrep.get("degenerate"):
        # multi-label: still a valid submission; keep it (ranked by dev macro-F1 like any other
        # run) so a weak model never leaves an experiment without an upload. Flagged in the log,
        # metrics.json ("degenerate": true) and the CSV selfcheck column.
        ctx.log("   !!! WARNING: every test document got the same label set (degenerate)")
    write_json(os.path.join(ctx.run_dir, "DONE.json"),
               {"finished": now_str(), "dev_macro_f1": s["macro_f1"], "rule": r["name"],
                "stop_reason": stop_reason, "submission": os.path.join(
                    ctx.run_dir, "submission", "submission.zip")})
    ctx.status("done", f"dev macro-F1 {s['macro_f1']:.4f}")
    ctx.log(f"   DONE dev macro-F1 {s['macro_f1']:.4f} (micro {s['micro_f1']:.4f}, subset acc "
            f"{s['subset_acc']:.4f}, rare {s.get('macro_f1_rare')}, rule {r['name']} "
            f"tau {r['tau_global']:.2f}, cv global {r.get('cv_macro_global')} vs per-label "
            f"{r.get('cv_macro_perlabel')}); test mean labels {vrep.get('mean_labels'):.2f}; "
            f"submission validated")
    return 0


def load_snippets(cfg: Dict) -> Dict[str, str]:
    p = cfg.get("snippets_path")
    if p and os.path.exists(p):
        return read_json(p, {}) or {}
    return {}


def run_overfit(ctx, stepper, d) -> int:
    """32 train docs, effective batch 8, up to 100 steps: loss must go below 0.1."""
    tr = d["train"]
    idx = list(range(min(32, len(tr["ids"]))))
    for g in stepper.opt.param_groups:
        g["lr"] = g["initial_lr"] = max(g["lr"], float(ctx.cfg.get("overfit_lr", 3e-3)))
    stepper.sched = None
    last = None
    n = 0
    for n in range(1, 101):
        rng = np.random.default_rng(n)
        order = [idx[i] for i in rng.permutation(len(idx))]
        losses = []
        for k in range(0, len(order), 8):
            l, _ = stepper.step(tr, order[k:k + 8], n, 100)
            if l is not None:
                losses.append(l)
        last = float(np.mean(losses)) if losses else float("nan")
        if last < 0.1:
            break
    sub = {k: (v[idx] if k == "labels" else [v[i] for i in idx]) for k, v in tr.items()
           if k in ("ids", "labels", "chunks")}
    _dt, amp_ctx = amp_setup(ctx.device)
    Pr = sigmoid_np(predict_logits(stepper.model, sub["chunks"], ctx.device, amp_ctx, 32))
    f_fit = macro(sub["labels"], Pr >= 0.5)
    res = {"passed": bool((last is not None and last < 0.1) or f_fit >= 0.9),
           "final_loss": last, "macro_f1_on_fit_docs": f_fit, "epochs": n, "n_docs": len(idx)}
    write_json(os.path.join(ctx.run_dir, "logs", "overfit.json"), res)
    ctx.status("done", f"overfit {res}")
    return 0 if res["passed"] else 1


def run_teacher_reuse(ctx: Ctx) -> int:
    cfg = ctx.cfg
    dev = ctx.device
    _dt, amp_ctx = amp_setup(dev)
    src = cfg["reuse_from"]
    rc = read_json(os.path.join(src, "logs", "run_config.json"))
    model = rebuild_trained(rc, os.path.join(src, "checkpoints", "best.pt"), cfg["phase1_dir"],
                            dev)
    d = load_split_docs(cfg)
    logits = {s: predict_logits(model, d[s]["chunks"], dev, amp_ctx, int(cfg["infer_chunks"]))
              for s in ("train", "dev", "test")}
    save_teacher_logits(os.path.join(ctx.run_dir, "teacher_logits.npz"),
                        {s: {"ids": d[s]["ids"], "logits": logits[s]} for s in logits})
    st = {"best_epoch": None, "epoch": 0, "pos": 0, "step": 0, "skipped": 0, "evals": [],
          "train_seconds": 0.0}
    rm = read_json(os.path.join(src, "metrics.json"), {}) or {}
    plan = {"K": cfg["K"], "train_fraction": rm.get("train_fraction", 1.0),
            "epochs": rm.get("epochs_planned"), "train_docs_used": rm.get("train_docs_used"),
            "tokens_per_s_train": rm.get("tokens_per_s"), "projected_minutes": 0.0}
    ctx.log(f"   [teacher] reused finished E1 run {src}")
    return finish(ctx, model, __import__("cjpe_peft").Technique(), d, amp_ctx, st, plan,
                  model_size(model), {"selfcheck": "reused"}, f"reused {src}")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--run-dir", required=True)
    ap.add_argument("--mode", default="train", choices=["train", "smoke", "overfit",
                                                        "teacher_reuse"])
    a = ap.parse_args()
    cfg = read_json(os.path.join(a.run_dir, "logs", "run_config.json"))
    ctx = Ctx(a.run_dir, cfg)
    code, reason = 1, "unknown"
    try:
        code = run_teacher_reuse(ctx) if a.mode == "teacher_reuse" else run_train(ctx, a.mode)
        reason = "ok" if code == 0 else f"exit {code}"
    except JobExit as e:
        code, reason = e.code, e.reason
        ctx.log(f"   JOB EXIT {code}: {reason}")
    except Exception as e:
        code = 7 if is_oom(e) else 1
        reason = f"{type(e).__name__}: {e}"
        ctx.log(f"   JOB ERROR: {reason}\n{traceback.format_exc()[-3000:]}")
    finally:
        st = read_json(os.path.join(a.run_dir, "status.json"), {}) or {}
        if code != 0 and st.get("state") not in ("failed",):
            ctx.status("failed", reason, exit_code=code)
        elif code == 0 and st.get("state") not in ("done", "degenerate", "smoke_ok"):
            ctx.status("done", reason)
    sys.stdout.flush()
    sys.exit(code)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase7/summ_model.py
"""
summ_model.py -- the Phase-1 causal LM as a summariser + the Phase-6 techniques + decoding.

SummLM keeps the Phase-1 LM intact (its own forward, KV cache, positions, ALiBi, sliding window)
and registers a `Trunk` VIEW of the same blocks so the tested Phase-2..5 PEFT code
(cjpe_peft.apply_technique: LoRMA+, LoRA, AdaLoRA, Houlsby adapters, full FT, ...) inserts its
modules into the very blocks the LM runs. `classifier` is an empty module (no new head: the
tied LM head generates the summary).

New techniques (Phase-6 list, PDF 12.2.1):
  pfeiffer  Pfeiffer adapter: ONE bottleneck (d/16) after the FFN output of every block + norms
  qlora     LoRA r=16 on top of NF4-quantised frozen linears (blockwise absmax, block 64, two
            4-bit codes per byte; Dettmers et al. 2023), dequantised on the fly
  prefix    prefix tuning (Li & Liang 2021): P learned key/value vectors per layer, used as the
            KV-cache "past"; initialised from the frozen model's own keys/values of real text
  prompt    prompt tuning (Lester et al. 2021): P learned input embeddings; their keys/values are
            computed through the frozen network on every forward (exactly "prepend P embeddings"
            for a causal decoder), initialised from real token embeddings
Training loss: token cross-entropy on the SUMMARY tokens only; logits are computed only at the
positions that predict a summary token (SelectHead) -> no (B, T, V) tensor.
"""
import math
import os
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from ner_models import Trunk, count_params, load_phase1_lm, torch_load
from ner_peft import ADAPTER_SITES, AdapterWrapped, LoRALinear, _get, _groups, _set
from phase1_models import _banned_ngram_tokens

EOS_ID, PAD_ID = 0, 1
CTX = 1024
BLOCK_ATTR = {"gpt2": "blocks", "llama": "layers", "bloom": "blocks"}
NEW_TECHNIQUES = ("pfeiffer", "qlora", "prefix", "prompt")
NF4 = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
       -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
       0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
       0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0]


class SelectHead(nn.Module):
    """lm_head that (when `sel` is set) only computes logits at the selected positions."""

    def __init__(self, head: nn.Linear):
        super().__init__()
        self.head = head
        self.sel = None

    def forward(self, x):
        if self.sel is not None:
            x = x[self.sel]
        return self.head(x)


# =============================================================================================
# virtual tokens (prefix / prompt tuning)
# =============================================================================================
def _attn_dims(trunk) -> Tuple[int, int]:
    a = trunk.blocks[0]
    if trunk.kind == "gpt2":
        return a.attn.n_head, a.attn.head_dim
    if trunk.kind == "llama":
        return a.attn.kv, a.attn.hd
    return a.self_attention.h, a.self_attention.hd


class Prefix(nn.Module):
    def __init__(self, past: List[Tuple[torch.Tensor, torch.Tensor]]):
        super().__init__()
        self.k = nn.ParameterList([nn.Parameter(k[0].detach().float().clone()) for k, _ in past])
        self.v = nn.ParameterList([nn.Parameter(v[0].detach().float().clone()) for _, v in past])
        self.length = int(past[0][0].shape[2])

    def past(self, model, B, dtype):
        return [(k.to(dtype)[None].expand(B, -1, -1, -1), v.to(dtype)[None].expand(B, -1, -1, -1))
                for k, v in zip(self.k, self.v)]


class Prompt(nn.Module):
    def __init__(self, emb: torch.Tensor):
        super().__init__()
        self.emb = nn.Parameter(emb.detach().float().clone())
        self.length = int(emb.shape[0])

    def past(self, model, B, dtype):
        e = self.emb
        h = model.trunk.embed.register_forward_hook(lambda m, i, o: e[None].to(o.dtype))
        try:
            ids = torch.zeros(1, self.length, dtype=torch.long, device=e.device)
            out = model.lm(ids, use_cache=True)
        finally:
            h.remove()
        return [(k.expand(B, -1, -1, -1), v.expand(B, -1, -1, -1))
                for k, v in out.past_key_values]


# =============================================================================================
# NF4 linear (QLoRA base)
# =============================================================================================
class NF4Linear(nn.Module):
    def __init__(self, base: nn.Linear, block: int = 64):
        super().__init__()
        W = base.weight.detach().float()
        self.in_features, self.out_features = base.in_features, base.out_features
        n = W.numel()
        pad = (-n) % block
        flat = torch.cat([W.reshape(-1), W.new_zeros(pad)]).view(-1, block)
        absmax = flat.abs().amax(1).clamp(min=1e-8)
        table = torch.tensor(NF4, device=W.device)
        codes = (flat / absmax[:, None]).unsqueeze(-1).sub(table).abs().argmin(-1).to(torch.uint8)
        codes = codes.view(-1)
        if codes.numel() % 2:
            codes = torch.cat([codes, codes.new_zeros(1)])
        self.register_buffer("q_packed", (codes[0::2] << 4) | codes[1::2])
        self.register_buffer("q_absmax", absmax.half())
        self.register_buffer("q_table", table)
        self.n, self.block = n, block
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(),
                                                                requires_grad=False)

    @property
    def weight(self):
        p = self.q_packed
        codes = torch.stack([p >> 4, p & 15], 1).view(-1).long()
        vals = self.q_table[codes][: self.q_absmax.numel() * self.block].view(-1, self.block)
        W = (vals * self.q_absmax.float()[:, None]).view(-1)[: self.n]
        return W.view(self.out_features, self.in_features)

    def forward(self, x):
        return F.linear(x, self.weight.to(x.dtype), None if self.bias is None else
                        self.bias.to(x.dtype))


# =============================================================================================
# the model
# =============================================================================================
class SummLM(nn.Module):
    def __init__(self, lm, layer_ids=None):
        super().__init__()
        kind = {"GPT2StyleLM": "gpt2", "LlamaStyleLM": "llama", "BloomStyleLM": "bloom"}[
            type(lm).__name__]
        if layer_ids is not None:
            blocks = getattr(lm, BLOCK_ATTR[kind])
            setattr(lm, BLOCK_ATTR[kind], nn.ModuleList([blocks[i] for i in layer_ids]))
        self.trunk = Trunk(lm, "causal")                 # PEFT view: the SAME block modules
        self.lm = lm
        lm.lm_head = SelectHead(lm.lm_head)
        self.classifier = nn.Module()                     # no new head
        self.virtual = None
        self.kind = kind

    @property
    def n_virtual(self) -> int:
        return 0 if self.virtual is None else self.virtual.length

    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False,
                last_only=False, sel=None):
        B = input_ids.shape[0]
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        if self.virtual is not None:
            if past_key_values is None:
                dt = self.trunk.embed.weight.dtype
                if input_ids.is_cuda and torch.is_autocast_enabled():
                    try:
                        dt = torch.get_autocast_dtype("cuda")
                    except AttributeError:                       # torch < 2.4
                        dt = torch.get_autocast_gpu_dtype()
                past_key_values = self.virtual.past(self, B, dt)
            attention_mask = torch.cat([attention_mask.new_ones(B, self.n_virtual),
                                        attention_mask], 1)
        self.lm.lm_head.sel = sel
        try:
            return self.lm(input_ids, attention_mask=attention_mask,
                           past_key_values=past_key_values, use_cache=use_cache,
                           last_only=last_only)
        finally:
            self.lm.lm_head.sel = None

    def lm_logits_at_targets(self, input_ids, attention_mask, labels):
        """(logits (N, V) fp32, targets (N,)) at the positions that predict a summary token."""
        tgt = labels[:, 1:]
        sel = torch.zeros_like(labels, dtype=torch.bool)
        sel[:, :-1] = tgt != -100
        out = self(input_ids, attention_mask, sel=sel)
        return out.logits.float(), tgt[tgt != -100]


def build_summ_model(phase1_dir: str, layer_ids=None):
    lm, cfg = load_phase1_lm(phase1_dir)
    return SummLM(lm, layer_ids), cfg


# =============================================================================================
# techniques
# =============================================================================================
def _sample_ids(sample_ids, n):
    ids = [int(x) for x in (sample_ids if sample_ids is not None else [])][:n]
    while len(ids) < n:
        ids.append(3 + len(ids) % 200)
    return ids


def apply_summ_technique(model: SummLM, technique: str, cfg: Dict, log=print, sample_ids=None):
    """Returns (optimizer groups, technique hooks). `sample_ids`: real tokens for prefix/prompt
    initialisation."""
    from cjpe_models import assert_one_device, model_size
    from cjpe_peft import Technique, apply_technique
    if technique not in NEW_TECHNIQUES:
        return apply_technique(model, technique, cfg, log=log)
    dev = model.trunk.embed.weight.device
    lr = float(cfg["lr"])
    wd = float(cfg.get("weight_decay", 0.01))
    if technique == "qlora":
        groups, tech = apply_technique(model, "lora", cfg, log=lambda *a: None)
        n = 0
        for name, m in list(model.named_modules()):
            if isinstance(m, LoRALinear) and isinstance(m.base, nn.Linear):
                m.base = NF4Linear(m.base, int(cfg.get("qlora_block", 64)))
                n += 1
        tech.name = "qlora"
        log(f"   [qlora] {n} frozen linears stored as NF4 (4-bit) + LoRA r={cfg.get('lora_r', 16)}")
    else:
        for p in model.parameters():
            p.requires_grad = False
        trunk = model.trunk
        if technique == "pfeiffer":
            bott = int(cfg.get("pfeiffer_bottleneck") or max(8, trunk.hidden_size // 16))
            for blk in trunk.blocks:
                path = ADAPTER_SITES[trunk.kind][1]                     # FFN output only
                _set(blk, path, AdapterWrapped(_get(blk, path), bott))
            named = [(n, p) for n, p in model.named_parameters() if ".adapter." in n]
            norm_ids = {id(p) for m in trunk.norm_modules() for p in m.parameters()}
            named += [(n, p) for n, p in model.named_parameters() if id(p) in norm_ids]
        else:
            P = int(cfg.get("n_virtual", 20))
            ids = torch.tensor([_sample_ids(sample_ids, P)], dtype=torch.long, device=dev)
            with torch.no_grad():
                if technique == "prefix":
                    model.virtual = Prefix(model.lm(ids, use_cache=True).past_key_values)
                else:
                    model.virtual = Prompt(trunk.embed(ids)[0])
            named = [(n, p) for n, p in model.named_parameters() if n.startswith("virtual.")]
        for _, p in named:
            p.requires_grad = True
        groups = _groups(named, lr, wd if technique == "pfeiffer" else 0.0, technique)
        tech = Technique()
        tech.name = technique
        actual = {n for n, p in model.named_parameters() if p.requires_grad}
        if actual != {n for n, _ in named}:
            raise AssertionError(f"[{technique}] trainable-parameter mismatch")
    model.to(dev)
    assert_one_device(model, dev)
    s = model_size(model)
    log(f"   [{technique}] trainable {s['trainable_params']:,} / {s['total_params']:,} "
        f"({s['trainable_pct']:.3f}%)")
    return groups, tech


def rebuild_summ(run_config: Dict, best_pt: str, phase1_dir: Optional[str] = None,
                 device="cpu") -> SummLM:
    """A finished run's model (KD teacher)."""
    from cjpe_models import load_trainable_state
    model, _ = build_summ_model(phase1_dir or run_config["phase1_dir"],
                                run_config.get("layer_ids"))
    tech = "full_ft" if run_config["technique"] in ("kd", "student_hard") else \
        run_config["technique"]
    apply_summ_technique(model, tech, run_config, log=lambda *a: None)
    load_trainable_state(model, torch_load(best_pt))
    return model.to(device).eval()


# =============================================================================================
# batching + decoding
# =============================================================================================
def collate_train(seqs: List[Tuple[np.ndarray, np.ndarray]], device=None) -> Dict:
    """seqs: (prompt incl. EOS-start and separator, target) -> right-padded batch with labels
    -100 everywhere except the target tokens."""
    T = max(len(p) + len(t) for p, t in seqs)
    assert T <= CTX, f"sequence of {T} tokens > {CTX}"
    ids = np.full((len(seqs), T), PAD_ID, np.int64)
    mask = np.zeros((len(seqs), T), np.int64)
    lab = np.full((len(seqs), T), -100, np.int64)
    for i, (p, t) in enumerate(seqs):
        n = len(p) + len(t)
        ids[i, :len(p)] = p
        ids[i, len(p):n] = t
        mask[i, :n] = 1
        lab[i, len(p):n] = t
    b = {"input_ids": torch.from_numpy(ids), "attention_mask": torch.from_numpy(mask),
         "labels": torch.from_numpy(lab), "n_tokens": int(mask.sum()),
         "n_targets": int((lab != -100).sum())}
    if device is not None:
        for k in ("input_ids", "attention_mask", "labels"):
            b[k] = b[k].to(device, non_blocking=True)
    return b


def collate_prompts(prompts: List[np.ndarray], device) -> Tuple[torch.Tensor, torch.Tensor]:
    T = max(len(p) for p in prompts)
    ids = np.full((len(prompts), T), PAD_ID, np.int64)
    mask = np.zeros((len(prompts), T), np.int64)
    for i, p in enumerate(prompts):                         # LEFT padding for generation
        ids[i, T - len(p):] = p
        mask[i, T - len(p):] = 1
    return torch.from_numpy(ids).to(device), torch.from_numpy(mask).to(device)


def _ban(logits, rows, finished, n_ngram):
    if n_ngram and n_ngram > 1:
        for b, row in enumerate(rows):
            if not finished[b] and len(row) >= n_ngram:
                bad = _banned_ngram_tokens(row, n_ngram)
                if bad:
                    logits[b, bad] = float("-inf")


@torch.inference_mode()
def decode_batch(model, ids, mask, max_new: int, min_new: int = 0, num_beams: int = 1,
                 no_repeat_ngram: int = 4, length_penalty: float = 1.0) -> List[List[int]]:
    """Greedy (num_beams=1) or beam search with n-gram blocking over the GENERATED tokens only
    (copying phrases from the document stays possible). Returns token lists without EOS."""
    B = ids.shape[0]
    K = max(1, int(num_beams))
    out = model(ids, attention_mask=mask, use_cache=True, last_only=True)
    past, logits = out.past_key_values, out.logits[:, -1].float()
    if K > 1:
        past = [(k.repeat_interleave(K, 0), v.repeat_interleave(K, 0)) for k, v in past]
        logits = logits.repeat_interleave(K, 0)
        mask = mask.repeat_interleave(K, 0)
    R = B * K
    rows = [[] for _ in range(R)]
    fin = [False] * R
    scores = torch.zeros(B, K, device=ids.device)
    if K > 1:
        scores[:, 1:] = float("-inf")
    for step in range(max_new):
        logits[:, PAD_ID] = float("-inf")
        if step < min_new:
            logits[:, EOS_ID] = float("-inf")
        _ban(logits, rows, fin, no_repeat_ngram)
        if K == 1:
            nxt = logits.argmax(-1)
            nl = nxt.tolist()
            for b in range(R):
                if not fin[b]:
                    if nl[b] == EOS_ID:
                        fin[b] = True
                    else:
                        rows[b].append(nl[b])
            nxt = torch.where(torch.tensor(fin, device=nxt.device), torch.full_like(nxt, EOS_ID),
                              nxt)
        else:
            logp = logits.log_softmax(-1)
            ft = torch.tensor(fin, device=logp.device)
            if ft.any():
                logp[ft] = float("-inf")
                logp[ft, EOS_ID] = 0.0
            V = logp.shape[-1]
            cand = (scores.view(-1, 1) + logp).view(B, K * V)
            top, idx = cand.topk(K, dim=1)
            src = (torch.arange(B, device=idx.device)[:, None] * K + idx // V).view(-1)
            tok = (idx % V).view(-1)
            scores = top
            sl, tl = src.tolist(), tok.tolist()
            rows = [list(rows[s]) for s in sl]
            fin = [fin[s] for s in sl]
            for r in range(R):
                if not fin[r]:
                    if tl[r] == EOS_ID:
                        fin[r] = True
                    else:
                        rows[r].append(tl[r])
            past = [(k[src], v[src]) for k, v in past]
            mask = mask[src]
            nxt = tok
        if all(fin) or step == max_new - 1:
            break
        mask = torch.cat([mask, mask.new_ones(R, 1)], 1)
        out = model(nxt[:, None], attention_mask=mask, past_key_values=past, use_cache=True)
        past, logits = out.past_key_values, out.logits[:, -1].float()
    if K == 1:
        return rows
    lens = torch.tensor([max(1, len(r)) for r in rows], device=scores.device,
                        dtype=torch.float32).view(B, K)
    best = (scores / lens ** length_penalty).argmax(1).tolist()
    return [rows[b * K + best[b]] for b in range(B)]


def generate_many(model, prompts: List[np.ndarray], device, amp_ctx, max_new: int,
                  min_new: int = 0, num_beams: int = 1, no_repeat_ngram: int = 4,
                  batch: int = 16, log=None) -> List[List[int]]:
    """Length-sorted batches; OOM -> batch halves. Output order == input order."""
    model.eval()
    order = sorted(range(len(prompts)), key=lambda i: -len(prompts[i]))
    res: List[Optional[List[int]]] = [None] * len(prompts)
    i = 0
    bs = max(1, int(batch))
    while i < len(order):
        g = order[i:i + bs]
        try:
            ids, mask = collate_prompts([prompts[j] for j in g], device)
            with amp_ctx():
                outs = decode_batch(model, ids, mask, max_new, min_new, num_beams,
                                    no_repeat_ngram)
        except RuntimeError as e:
            if "out of memory" not in str(e).lower() or bs == 1:
                raise
            torch.cuda.empty_cache()
            bs = max(1, bs // 2)
            if log:
                log(f"   [oom] generation batch -> {bs}")
            continue
        for j, o in zip(g, outs):
            res[j] = o
        i += len(g)
    return res


In [ ]:
%%writefile /kaggle/working/phase7/pcr_ml.py
"""
pcr_ml.py -- torch-free retrieval utilities for Phase 7 (Prior Case Retrieval).

  * word tokenizer + BM25 (Robertson/Sparck-Jones idf, k1/b saturation, log(1+qtf) query
    weights, optional bigrams) over ONE split's candidate pool, as a dense (n_query, n_pool)
    float32 score matrix (scipy.sparse product)
  * micro_prf_at_k: Micro-P/R/F1@k exactly as the PDF defines it (pooled over every query's
    top-k against the FULL gold set; gold sets are neither truncated nor padded)
  * per-query z-normalisation, score fusion (dense x BM25), top-k with self-exclusion,
    reciprocal-rank fusion (E8 ensemble), average precision (dev proxy)
"""
import re
from collections import Counter
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

K_EVAL = 7
_WORD = re.compile(r"[a-z0-9]+")
STOP = set("""a an the and or of to in on at by for with from as is are was were be been being
it its this that these those which who whom whose what when where why how not no nor but if then
than so such into upon under over said shall may can could would should will has have had do does
did also any all each other there their them they he she his her him we our you your i me my
herein thereof therein whereas thus hence very only same own between before after above below""".split())


def words(text: str, ngram: int = 1) -> List[str]:
    w = [t for t in _WORD.findall(text.lower()) if len(t) > 1 and t not in STOP]
    if ngram >= 2:
        w = w + [f"{a}_{b}" for a, b in zip(w, w[1:])]
    return w


class BM25:
    """Fit on one pool; score(query texts) -> (nq, n_pool) float32."""

    def __init__(self, pool_texts: Sequence[str], k1: float = 1.2, b: float = 0.75,
                 ngram: int = 1, pool_tokens: Optional[List[List[str]]] = None):
        from scipy import sparse
        self.k1, self.b, self.ngram = float(k1), float(b), int(ngram)
        toks = pool_tokens if pool_tokens is not None else [words(t, ngram) for t in pool_texts]
        self.vocab: Dict[str, int] = {}
        rows, cols, vals = [], [], []
        for i, ws in enumerate(toks):
            for w, c in Counter(ws).items():
                j = self.vocab.setdefault(w, len(self.vocab))
                rows.append(i)
                cols.append(j)
                vals.append(c)
        n, V = len(toks), max(1, len(self.vocab))
        tf = sparse.csr_matrix((np.asarray(vals, np.float32), (rows, cols)), shape=(n, V))
        dl = np.asarray([len(ws) for ws in toks], np.float32)
        avg = float(dl.mean()) if n else 1.0
        df = np.bincount(np.asarray(cols, np.int64), minlength=V).astype(np.float32)
        self.idf = np.log1p((n - df + 0.5) / (df + 0.5)).astype(np.float32)
        tf = tf.tocoo()
        norm = self.k1 * (1 - self.b + self.b * dl[tf.row] / max(avg, 1e-9))
        w = tf.data * (self.k1 + 1) / (tf.data + norm) * self.idf[tf.col]
        self.D = sparse.csr_matrix((w.astype(np.float32), (tf.row, tf.col)), shape=(n, V))

    def query_matrix(self, texts: Sequence[str], tokens: Optional[List[List[str]]] = None):
        from scipy import sparse
        rows, cols, vals = [], [], []
        toks = tokens if tokens is not None else [words(t, self.ngram) for t in texts]
        for i, ws in enumerate(toks):
            for w, c in Counter(ws).items():
                j = self.vocab.get(w)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
                    vals.append(np.log1p(c))
        return sparse.csr_matrix((np.asarray(vals, np.float32), (rows, cols)),
                                 shape=(len(toks), self.D.shape[1]))

    def score(self, texts: Sequence[str], tokens=None) -> np.ndarray:
        return np.asarray((self.query_matrix(texts, tokens) @ self.D.T).todense(), np.float32)


# =============================================================================================
# ranking + metrics
# =============================================================================================
def mask_self(S: np.ndarray, q_ids: Sequence[str], pool_ids: Sequence[str]) -> np.ndarray:
    """A query never retrieves itself (when it also sits in the pool)."""
    pos = {p: j for j, p in enumerate(pool_ids)}
    S = S.astype(np.float32, copy=True)
    for i, q in enumerate(q_ids):
        j = pos.get(q)
        if j is not None:
            S[i, j] = -np.inf
    return S


def topk(S: np.ndarray, k: int = K_EVAL) -> np.ndarray:
    """(nq, k) pool indices, best first (ties broken by index -> deterministic)."""
    k = min(k, S.shape[1])
    idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    part = np.take_along_axis(S, idx, 1)
    order = np.lexsort((idx, -part), axis=1)
    return np.take_along_axis(idx, order, 1)


def micro_prf_at_k(ranked: Sequence[Sequence], gold: Sequence[Iterable], k: int = K_EVAL
                   ) -> Dict[str, float]:
    tp = n_pred = n_gold = 0
    for r, g in zip(ranked, gold):
        g = set(g)
        r = list(r)[:k]
        tp += len(set(r) & g)
        n_pred += len(r)
        n_gold += len(g)
    p = tp / n_pred if n_pred else 0.0
    rc = tp / n_gold if n_gold else 0.0
    f = 2 * p * rc / (p + rc) if p + rc else 0.0
    return {"micro_p": p, "micro_r": rc, "micro_f1": f, "tp": tp, "n_pred": n_pred,
            "n_gold": n_gold}


def eval_scores(S: np.ndarray, gold: Sequence[Iterable], k: int = K_EVAL) -> Dict[str, float]:
    return micro_prf_at_k(topk(S, k).tolist(), gold, k)


def average_precision(scores: np.ndarray, gold: Iterable) -> float:
    g = set(gold)
    if not g:
        return 0.0
    order = np.argsort(-scores, kind="stable")
    hits, s = 0, 0.0
    for rnk, j in enumerate(order, 1):
        if int(j) in g:
            hits += 1
            s += hits / rnk
            if hits == len(g):
                break
    return s / len(g)


def zrow(S: np.ndarray) -> np.ndarray:
    """Per-query z-score over the finite entries (masked entries stay -inf)."""
    S = S.astype(np.float32, copy=True)
    fin = np.isfinite(S)
    X = np.where(fin, S, np.nan)
    mu = np.nanmean(X, 1, keepdims=True)
    sd = np.nanstd(X, 1, keepdims=True)
    Z = (X - mu) / np.maximum(sd, 1e-6)
    return np.where(fin, Z, -np.inf).astype(np.float32)


def fuse(dense: np.ndarray, bm25: Optional[np.ndarray], w: float) -> np.ndarray:
    """w * z(dense) + (1 - w) * z(bm25); w = 1 -> dense only."""
    if bm25 is None or w >= 1.0:
        return zrow(dense)
    if w <= 0.0:
        return zrow(bm25)
    zd, zb = zrow(dense), zrow(bm25)
    fin = np.isfinite(zd) & np.isfinite(zb)
    return np.where(fin, w * np.where(fin, zd, 0) + (1 - w) * np.where(fin, zb, 0),
                    -np.inf).astype(np.float32)


def tune_fusion(dense: np.ndarray, bm25: Optional[np.ndarray], gold, weights: Sequence[float],
                k: int = K_EVAL) -> Tuple[float, Dict, List[Dict]]:
    curve = []
    for w in weights:
        m = eval_scores(fuse(dense, bm25, w), gold, k)
        curve.append({"dense_weight": float(w), **m})
    best = max(curve, key=lambda r: (r["micro_f1"], r["dense_weight"]))
    return best["dense_weight"], best, curve


def rrf(rank_lists: List[List[List[int]]], n_pool: int, k_rrf: int = 60, depth: int = 100
        ) -> np.ndarray:
    """Reciprocal-rank fusion of several systems' rankings -> (nq, n_pool) scores."""
    nq = len(rank_lists[0])
    S = np.zeros((nq, n_pool), np.float32)
    for sysr in rank_lists:
        for i, r in enumerate(sysr):
            for rnk, j in enumerate(r[:depth], 1):
                S[i, j] += 1.0 / (k_rrf + rnk)
    return S


def full_rank(S: np.ndarray, depth: int = 100) -> List[List[int]]:
    return topk(S, min(depth, S.shape[1])).tolist()


In [ ]:
%%writefile /kaggle/working/phase7/pcr_data.py
"""
pcr_data.py -- Phase-7 data: discovery (attached dataset, else the HF download), schema-tolerant
loading of the six files, the chunk token cache and the BM25 lexical channel.

Phase_7_PCR/{Train,Dev,Test}/{split}.json             queries: id, text (sentences),
                                                      relevant_candidates (Train/Dev only)
Phase_7_PCR/{Train,Dev,Test}/{split}_candidates.json  that split's candidate pool
A split's queries are only ever matched against THAT split's pool.

Chunks: every document becomes <= max_chunks token chunks of <= chunk_tokens tokens (EOS last).
Short documents are tokenised whole and cut into consecutive chunks; long ones are cut into
max_chunks equal-character sentence groups and the start of each group is tokenised -> the
chunks cover the WHOLE case at bounded tokenisation cost.
BM25 (pcr_ml.BM25, word level, the split's own pool): k1 / b / bigrams tuned on Dev
Micro-F1@7; the dev + test score matrices and the train hard-negative lists are cached.
"""
import hashlib
import json
import os
import pickle
import re
import time
from multiprocessing import get_context
from typing import Dict, List, Optional, Tuple

import numpy as np

from cjpe_common import SPEC, file_md5, find_files
from cjpe_data import DataError, hf_download
from pcr_ml import BM25, eval_scores, mask_self, topk, words

CACHE_VERSION = 1
SPLITS = ("train", "dev", "test")
CH_PER_TOK = 4.2
ID_KEYS = ("id", "query_id", "qid", "case_id", "doc_id", "candidate_id", "cid", "name")
TEXT_KEYS = ("text", "sentences", "document", "content", "body", "case_text", "judgment")
REL_KEYS = ("relevant_candidates", "relevant", "relevant_ids", "positives", "labels",
            "gold", "citations")
EOS_ID = 0
_TOK = None


# =============================================================================================
# discovery + parsing
# =============================================================================================
def discover(input_roots: List[str], log=print) -> Dict:
    found = {}
    for key, rel in SPEC["files"].items():
        name = os.path.basename(rel)
        sub = rel.split("/")[0].lower()
        hits = [h for h in find_files(name, input_roots, max_hits=50) if os.path.isfile(h)]
        hits.sort(key=lambda h: (-(f"/{sub}/" in h.lower()) - ("phase_7" in h.lower()), h))
        if hits:
            found[key] = hits[0]
    if len(found) == len(SPEC["files"]):
        return {"source": "kaggle_dataset", "paths": found}
    log(f"   [data] attached dataset incomplete (found {sorted(found)}); trying the HF download")
    dl = hf_download(log=log)
    if dl and len(dl) == len(SPEC["files"]):
        return {"source": "huggingface", "paths": dl}
    return {"source": None, "paths": found}


def _items(obj) -> List[Dict]:
    if isinstance(obj, dict):
        lists = [v for v in obj.values() if isinstance(v, list) and v and isinstance(v[0], dict)]
        if len(obj) <= 3 and lists:
            return lists[0]
        out = []
        for k, v in obj.items():
            out.append({"id": k, **v} if isinstance(v, dict) else {"id": k, "text": v})
        return out
    if isinstance(obj, list):
        return [r for r in obj if isinstance(r, dict)]
    raise DataError(f"unsupported JSON container {type(obj).__name__}")


def _pick(r: Dict, keys) -> Optional[str]:
    return next((k for k in keys if k in r), None)


def _sentences(t) -> List[str]:
    if isinstance(t, list):
        out = []
        for x in t:
            out += _sentences(x) if isinstance(x, list) else [str(x)]
        return [s for s in out if s.strip()]
    if isinstance(t, str):
        s = [x for x in re.split(r"\n+|(?<=[.;:?!])\s+(?=[A-Z(\"'])", t) if x.strip()]
        return s or ([t] if t.strip() else [])
    return [] if t is None else [str(t)]


def load_file(path: str, kind: str, split: str) -> Tuple[List[Dict], Dict]:
    with open(path, encoding="utf-8") as f:
        recs = _items(json.load(f))
    if not recs:
        raise DataError(f"{split} {kind}: no records in {path}")
    s0 = recs[:200]
    idk = next((k for k in ID_KEYS if all(k in r for r in s0)), None)
    tk = next((k for k in TEXT_KEYS if all(k in r for r in s0)), None)
    if idk is None or tk is None:
        raise DataError(f"{split} {kind}: cannot find id/text keys among {sorted(recs[0])}")
    rk = next((k for k in REL_KEYS if all(k in r for r in s0)), None) if kind == "q" else None
    if kind == "q" and split != "test" and rk is None:
        raise DataError(f"{split}: no relevance key among {sorted(recs[0])}")
    out = []
    for r in recs:
        d = {"id": str(r[idk]), "sents": _sentences(r[tk])}
        if rk:
            v = r.get(rk) or []
            d["rel_ids"] = [str(x) for x in (v if isinstance(v, list) else [v])]
        out.append(d)
    schema = {"id_key": idk, "text_key": tk, "rel_key": rk, "n": len(out), "path": path,
              "fields": sorted(recs[0]), "md5": file_md5(path)}
    return out, schema


def _id_lookup(pool_ids: List[str]) -> Dict[str, int]:
    m = {p: j for j, p in enumerate(pool_ids)}
    for j, p in enumerate(pool_ids):                     # "0000021652" == "21652"
        m.setdefault(p.lstrip("0") or "0", j)
    return m


def load_all(input_roots, log=print) -> Tuple[Dict, Dict, Dict]:
    found = discover(input_roots, log=log)
    if found["source"] is None:
        raise FileNotFoundError(
            "Phase-7 data not found. Either attach a dataset holding Phase_7_PCR/{Train,Dev,Test}"
            "/{split}.json + {split}_candidates.json, or add the Kaggle secret `HF` and turn "
            f"Internet ON. Found: {found['paths']}")
    data, schemas, audit = {}, {}, {}
    for s in SPLITS:
        q, sq = load_file(found["paths"][s], "q", s)
        pool, sp = load_file(found["paths"][f"{s}_candidates"], "pool", s)
        seen, uniq = set(), []
        for d in pool:
            if d["id"] not in seen:
                seen.add(d["id"])
                uniq.append(d)
        pool = uniq
        look = _id_lookup([d["id"] for d in pool])
        missing = n_rel = self_in = 0
        for d in q:
            if "rel_ids" in d:
                idx = []
                for x in d["rel_ids"]:
                    j = look.get(x, look.get(x.lstrip("0") or "0"))
                    if j is None:
                        missing += 1
                    elif pool[j]["id"] != d["id"] and j not in idx:
                        idx.append(j)
                n_rel += len(d["rel_ids"])
                d["rel"] = idx
            self_in += d["id"] in look
        data[s] = {"q": q, "pool": pool}
        schemas[s], schemas[f"{s}_candidates"] = sq, sp
        rel = [len(d.get("rel", [])) for d in q]
        audit[s] = {"queries": len(q), "pool": len(pool), "relevant_total": n_rel,
                    "relevant_missing_from_pool": missing, "queries_in_own_pool": self_in,
                    "rel_per_query_mean": float(np.mean(rel)) if rel else 0.0,
                    "rel_per_query_max": int(max(rel)) if rel else 0,
                    "queries_without_relevant": int(sum(1 for x in rel if x == 0)),
                    "query_chars_median": float(np.median([sum(map(len, d["sents"])) for d in q])),
                    "pool_chars_median": float(np.median([sum(map(len, d["sents"])) for d in pool]))}
        log(f"   [data] {s}: {len(q)} queries ({sq['id_key']!r}/{sq['text_key']!r}/"
            f"{sq['rel_key']!r}), pool {len(pool)} candidates; relevant per query mean "
            f"{audit[s]['rel_per_query_mean']:.2f}, {missing} relevant ids not in the pool, "
            f"{self_in} queries inside their own pool")
    ids = {s: {d["id"] for d in data[s]["q"]} for s in SPLITS}
    audit["query_overlap"] = {"train_dev": len(ids["train"] & ids["dev"]),
                              "train_test": len(ids["train"] & ids["test"]),
                              "dev_test": len(ids["dev"] & ids["test"])}
    return data, schemas, {"audit": audit, "source": found["source"]}


# =============================================================================================
# chunk token cache
# =============================================================================================
def _init_worker(bpe_dir: str) -> None:
    global _TOK
    from bpe_tokenizer import BPETokenizer
    _TOK = BPETokenizer.from_pretrained(bpe_dir)


def doc_chunk_texts(sents: List[str], C: int, K: int) -> Tuple[List[str], bool]:
    full = " ".join(sents)
    if len(full) <= K * C * CH_PER_TOK:
        return [full], True
    lens = np.cumsum([len(s) + 1 for s in sents])
    tot = float(lens[-1])
    groups, start = [], 0
    for k in range(1, K + 1):
        end = int(np.searchsorted(lens, tot * k / K, side="left")) + 1
        end = min(max(end, start + 1), len(sents))
        if start < len(sents):
            groups.append(" ".join(sents[start:end]))
        start = end
    lim = int(C * CH_PER_TOK * 1.4)
    return [g[:lim] for g in groups if g.strip()], False


def _encode(args):
    sents, C, K = args
    texts, whole = doc_chunk_texts(sents, C, K)
    body = C - 1
    if whole:
        ids = _TOK.encode(texts[0]) if texts[0] else []
        pieces = [ids[i:i + body] for i in range(0, max(1, len(ids)), body)] or [[]]
        if len(pieces) > K:
            pick = np.linspace(0, len(pieces) - 1, K).round().astype(int)
            pieces = [pieces[i] for i in sorted(set(pick.tolist()))]
    else:
        pieces = [_TOK.encode(t)[:body] for t in texts]
    return [np.asarray(list(p) + [EOS_ID], np.int32) for p in pieces if len(p) or len(pieces) == 1]


def cache_key(md5s: Dict[str, str], tok_md5: str, C: int, K: int) -> str:
    s = json.dumps({"v": CACHE_VERSION, "files": md5s, "tok": tok_md5, "C": C, "K": K},
                   sort_keys=True)
    return hashlib.md5(s.encode()).hexdigest()[:16]


def find_cache(key: str, roots: List[str]) -> Optional[str]:
    name = f"pcr_cache_{key}.pkl"
    for r in roots:
        if r and os.path.isdir(r):
            for dp, _dn, fns in os.walk(r):
                if name in fns:
                    return os.path.join(dp, name)
    return None


def build_cache(data: Dict, bpe_dir: str, out_dir: str, key: str, C: int, K: int, n_proc: int,
                log=print, bm25_grid=None, hard_k: int = 60) -> str:
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"pcr_cache_{key}.pkl")
    t0 = time.time()
    res = {"C": C, "K": K, "splits": {}}
    pool_ctx = get_context("fork").Pool(n_proc, initializer=_init_worker, initargs=(bpe_dir,)) \
        if n_proc > 1 else None
    if pool_ctx is None:
        _init_worker(bpe_dir)
    try:
        for s in SPLITS:
            out = {}
            for part in ("q", "pool"):
                docs = data[s][part]
                args = [(d["sents"], C, K) for d in docs]
                enc = pool_ctx.map(_encode, args, chunksize=4) if pool_ctx else \
                    [_encode(a) for a in args]
                out[part] = [{"id": d["id"], "chunks": e, "rel": d.get("rel"),
                              "n_chars": sum(map(len, d["sents"]))} for d, e in zip(docs, enc)]
            res["splits"][s] = out
            log(f"   [cache] {s}: {len(out['q'])} queries + {len(out['pool'])} candidates "
                f"tokenised ({time.time() - t0:.0f}s so far)")
    finally:
        if pool_ctx:
            pool_ctx.close()
    res["bm25"] = bm25_channel(data, log, bm25_grid, hard_k)
    with open(path + ".tmp", "wb") as f:
        pickle.dump(res, f, protocol=4)
    os.replace(path + ".tmp", path)
    return path


def bm25_channel(data: Dict, log=print, grid=None, hard_k: int = 60) -> Dict:
    """Tune BM25 on dev Micro-F1@7; dev/test score matrices + train hard negatives."""
    t0 = time.time()
    grid = grid or [(k1, b) for k1 in (0.9, 1.5, 2.5) for b in (0.5, 0.75, 0.9)]
    txt = {s: {p: [" ".join(d["sents"]) for d in data[s][p]] for p in ("q", "pool")}
           for s in SPLITS}
    gold_dev = [d.get("rel", []) for d in data["dev"]["q"]]
    curve, best = [], None
    for ng in (1, 2):
        tk = {s: {p: [words(t, ng) for t in txt[s][p]] for p in ("q", "pool")}
              for s in ("dev",)}
        for k1, b in grid:
            bm = BM25(None, k1, b, ng, pool_tokens=tk["dev"]["pool"])
            S = mask_self(bm.score(None, tk["dev"]["q"]), [d["id"] for d in data["dev"]["q"]],
                          [d["id"] for d in data["dev"]["pool"]])
            m = eval_scores(S, gold_dev)
            curve.append({"k1": k1, "b": b, "ngram": ng, **m})
            if best is None or m["micro_f1"] > best["micro_f1"]:
                best = curve[-1]
    out = {"params": {k: best[k] for k in ("k1", "b", "ngram")}, "dev_curve": curve,
           "dev_micro_f1": best["micro_f1"], "dev_micro_p": best["micro_p"],
           "dev_micro_r": best["micro_r"]}
    p = out["params"]
    for s in SPLITS:
        bm = BM25(txt[s]["pool"], p["k1"], p["b"], p["ngram"])
        S = mask_self(bm.score(txt[s]["q"]), [d["id"] for d in data[s]["q"]],
                      [d["id"] for d in data[s]["pool"]])
        if s == "train":
            out["train_top"] = topk(S, min(hard_k, S.shape[1])).astype(np.int32)
        else:
            out[s] = S.astype(np.float32)
    log(f"   [bm25] tuned on dev: k1={p['k1']} b={p['b']} ngram={p['ngram']} -> dev Micro-F1@7 "
        f"{out['dev_micro_f1']:.4f} (P {out['dev_micro_p']:.4f} R {out['dev_micro_r']:.4f}); "
        f"{time.time() - t0:.0f}s")
    return out


def load_cache(path: str) -> Dict:
    with open(path, "rb") as f:
        return pickle.load(f)


def chunk_stats(cache: Dict) -> Dict:
    out = {}
    for s, d in cache["splits"].items():
        for p in ("q", "pool"):
            n = [len(x["chunks"]) for x in d[p]]
            out[f"{s}_{p}_chunks_mean"] = float(np.mean(n)) if n else 0.0
            out[f"{s}_{p}_chunks_full_frac"] = float(np.mean([k >= cache["K"] for k in n])) \
                if n else 0.0
    return out


In [ ]:
%%writefile /kaggle/working/phase7/pcr_model.py
"""
pcr_model.py -- the Phase-1 LM as a shared-weight bi-encoder for Prior Case Retrieval, the
Phase-7 techniques, and batched chunk encoding.

PcrEncoder = Trunk (the Phase-1 embed -> blocks -> final norm, reused unchanged from Phase 2;
causal attention by default, `bidir` optional) -> pooling over the real tokens of ONE chunk
(`mean` or `last` = the EOS appended to every chunk) -> `classifier` head: a d x d projection
initialised to the IDENTITY (so every technique starts from the raw Phase-1 representation)
+ a learned logit scale (temperature, init 1/0.05) -> L2-normalised chunk vector.
A document vector is the normalised mean of its chunk vectors (pcr_train). Queries and
candidates go through the SAME encoder.

Techniques (PDF 13.2.1 list):
  full_ft / partial_ft / lorma / lora   cjpe_peft.apply_technique (Phases 2-6, unchanged)
  qlora     LoRA r=16 on NF4 4-bit frozen linears (summ_model.NF4Linear, Phase 6)
  prefix    prefix tuning (Li & Liang 2021): the activations of P virtual positions at EVERY
            block input are free parameters (initialised from the frozen model's own
            activations of real text); text tokens attend to them in every layer
  prompt    prompt tuning (Lester et al. 2021): P learned input embeddings only
  ia3       IA^3 (Liu et al. 2022): learned vectors l_k, l_v rescale the attention keys and
            values, l_ff rescales the FFN inner activation (input of down_proj); init 1
  kd / student_hard   full FT of a layer-pruned student (KD loss in pcr_train)
The virtual positions are prepended as P dummy tokens; hooks overwrite their states, pooling
ignores them.
"""
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from ner_models import Trunk, load_phase1_lm, torch_load
from ner_peft import FUSED_QKV, LoRALinear, _get, _groups, _set

EOS_ID = 0
NEW_TECHNIQUES = ("qlora", "prefix", "prompt", "ia3")
IA3_KV = {"gpt2": ["attn.qkv"], "bloom": ["self_attention.query_key_value"],
          "llama": ["attn.k_proj", "attn.v_proj"]}


class Head(nn.Module):
    def __init__(self, d: int, scale: float = 20.0):
        super().__init__()
        self.proj = nn.Linear(d, d)
        with torch.no_grad():
            self.proj.weight.copy_(torch.eye(d))
            self.proj.bias.zero_()
        self.logit_scale = nn.Parameter(torch.tensor(float(np.log(scale))))

    def scale(self) -> torch.Tensor:
        return self.logit_scale.clamp(max=float(np.log(100.0))).exp()


class IA3Out(nn.Module):
    """y = base(x) * l on the output slice [lo, hi) (keys / values); l initialised to 1."""

    def __init__(self, base: nn.Linear, lo: int, hi: int):
        super().__init__()
        self.base, self.lo, self.hi = base, lo, hi
        self.ia3_l = nn.Parameter(torch.ones(hi - lo, device=base.weight.device))

    @property
    def weight(self):
        return self.base.weight

    def forward(self, x):
        y = self.base(x)
        s = torch.ones(y.shape[-1], device=y.device, dtype=y.dtype)
        s = torch.cat([s[:self.lo], self.ia3_l.to(y.dtype), s[self.hi:]])
        return y * s


class IA3In(nn.Module):
    """y = base(x * l_ff) -- rescales the FFN inner activation."""

    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.ia3_l = nn.Parameter(torch.ones(base.in_features, device=base.weight.device))

    @property
    def weight(self):
        return self.base.weight

    def forward(self, x):
        return self.base(x * self.ia3_l.to(x.dtype))


class Virtual(nn.Module):
    """P virtual positions: states[0] replaces the embedding output, states[l] (prefix only)
    the input of block l."""

    def __init__(self, states: List[torch.Tensor], mode: str):
        super().__init__()
        self.mode = mode
        self.states = nn.ParameterList([nn.Parameter(s.detach().float().clone()) for s in states])
        self.length = int(states[0].shape[0])


def _overwrite(x: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
    P = s.shape[0]
    return torch.cat([s.to(x.dtype)[None].expand(x.shape[0], -1, -1), x[:, P:]], 1)


class PcrEncoder(nn.Module):
    def __init__(self, lm, attn_mode: str = "causal", layer_ids=None, pooling: str = "mean"):
        super().__init__()
        self.trunk = Trunk(lm, attn_mode, layer_ids)
        self.classifier = Head(self.trunk.hidden_size)
        self.virtual: Optional[Virtual] = None
        self.pooling = pooling
        self._hooks = []

    @property
    def n_virtual(self) -> int:
        return 0 if self.virtual is None else self.virtual.length

    def install_virtual(self, v: Virtual) -> None:
        for h in self._hooks:
            h.remove()
        self.virtual = v
        st = v.states
        if v.mode == "prompt":
            self._hooks = [self.trunk.embed.register_forward_hook(
                lambda m, i, o: _overwrite(o, st[0]))]
            return
        self._hooks = []
        for li in range(len(st)):
            self._hooks.append(self.trunk.blocks[li].register_forward_pre_hook(
                lambda m, a, li=li: (_overwrite(a[0], st[li]),) + tuple(a[1:])))

    def forward(self, input_ids, attention_mask):
        P = self.n_virtual
        if P:
            B = input_ids.shape[0]
            input_ids = torch.cat([input_ids.new_full((B, P), EOS_ID), input_ids], 1)
            attention_mask = torch.cat([attention_mask.new_ones(B, P), attention_mask], 1)
        h = self.trunk(input_ids, attention_mask)[:, P:]
        m = attention_mask[:, P:].to(h.dtype)
        if self.pooling == "last":
            last = attention_mask[:, P:].long().sum(1).clamp(min=1) - 1
            pooled = h[torch.arange(h.shape[0], device=h.device), last]
        else:
            pooled = (h * m[..., None]).sum(1) / m.sum(1, keepdim=True).clamp(min=1)
        return F.normalize(self.classifier.proj(pooled).float(), dim=-1)


def build_pcr_model(phase1_dir: str, layer_ids=None, attn_mode="causal", pooling="mean"):
    lm, cfg = load_phase1_lm(phase1_dir)
    return PcrEncoder(lm, attn_mode, layer_ids, pooling), cfg


# =============================================================================================
# techniques
# =============================================================================================
def _sample_ids(sample_ids, n):
    ids = [int(x) for x in (sample_ids if sample_ids is not None else [])][:n]
    while len(ids) < n:
        ids.append(3 + len(ids) % 200)
    return ids


def apply_pcr_technique(model: PcrEncoder, technique: str, cfg: Dict, log=print,
                        sample_ids=None):
    """Returns (optimizer groups, technique hooks)."""
    from cjpe_models import assert_one_device, model_size
    from cjpe_peft import Technique, apply_technique
    if technique not in NEW_TECHNIQUES:
        return apply_technique(model, technique, cfg, log=log)
    dev = model.trunk.embed.weight.device
    lr, wd = float(cfg["lr"]), float(cfg.get("weight_decay", 0.01))
    if technique == "qlora":
        from summ_model import NF4Linear
        groups, tech = apply_technique(model, "lora", cfg, log=lambda *a: None)
        n = 0
        for _, m in list(model.named_modules()):
            if isinstance(m, LoRALinear) and isinstance(m.base, nn.Linear):
                m.base = NF4Linear(m.base, int(cfg.get("qlora_block", 64)))
                n += 1
        tech.name = "qlora"
        log(f"   [qlora] {n} frozen linears stored as NF4 (4-bit) + LoRA r={cfg.get('lora_r', 16)}")
    else:
        for p in model.parameters():
            p.requires_grad = False
        trunk = model.trunk
        if technique == "ia3":
            d = trunk.hidden_size
            for blk in trunk.blocks:
                for path in IA3_KV[trunk.kind]:
                    base = _get(blk, path)
                    if FUSED_QKV.get(trunk.kind) == path:
                        _set(blk, path, IA3Out(base, d, 3 * d))           # keys + values
                    else:
                        _set(blk, path, IA3Out(base, 0, base.out_features))
                _set(blk, "mlp.down_proj", IA3In(_get(blk, "mlp.down_proj")))
            named = [(n, p) for n, p in model.named_parameters() if n.endswith("ia3_l")]
        else:
            P = int(cfg.get("n_virtual", 20))
            ids = torch.tensor([_sample_ids(sample_ids, P)], dtype=torch.long, device=dev)
            with torch.no_grad():
                if technique == "prompt":
                    states = [trunk.embed(ids)[0]]
                else:
                    got = {}
                    hs = [b.register_forward_pre_hook(
                        lambda m, a, li=li: got.__setitem__(li, a[0][0].detach()))
                        for li, b in enumerate(trunk.blocks)]
                    try:
                        trunk(ids, torch.ones_like(ids))
                    finally:
                        for h in hs:
                            h.remove()
                    states = [got[li] for li in range(len(trunk.blocks))]
            model.install_virtual(Virtual(states, technique).to(dev))
            named = [(n, p) for n, p in model.named_parameters() if n.startswith("virtual.")]
        for _, p in named:
            p.requires_grad = True
        groups = _groups(named, lr, 0.0, technique)
        tech = Technique()
        tech.name = technique
    head = [(n, p) for n, p in model.named_parameters() if n.startswith("classifier.")]
    for _, p in head:
        p.requires_grad = True
    if technique != "qlora":
        groups += _groups(head, float(cfg.get("head_lr", lr)), wd, "head")
        actual = {n for n, p in model.named_parameters() if p.requires_grad}
        if actual != {n for n, _ in named} | {n for n, _ in head}:
            raise AssertionError(f"[{technique}] trainable-parameter mismatch")
    model.to(dev)
    assert_one_device(model, dev)
    s = model_size(model)
    log(f"   [{technique}] trainable {s['trainable_params']:,} / {s['total_params']:,} "
        f"({s['trainable_pct']:.3f}%)")
    return groups, tech


def rebuild_pcr(run_config: Dict, best_pt: str, device="cpu", sample_ids=None) -> PcrEncoder:
    """A finished run's encoder (KD teacher, E8)."""
    from cjpe_models import load_trainable_state
    model, _ = build_pcr_model(run_config["phase1_dir"], run_config.get("layer_ids"),
                               run_config.get("attn_mode", "causal"),
                               run_config.get("pooling", "mean"))
    tech = "full_ft" if run_config["technique"] in ("kd", "student_hard") else \
        run_config["technique"]
    apply_pcr_technique(model, tech, run_config, log=lambda *a: None, sample_ids=sample_ids)
    load_trainable_state(model, torch_load(best_pt))
    return model.to(device).eval()


# =============================================================================================
# batching + encoding
# =============================================================================================
def collate(chunks: List[np.ndarray], device):
    L = max(len(c) for c in chunks)
    ids = np.full((len(chunks), L), EOS_ID, np.int64)
    mask = np.zeros((len(chunks), L), np.int64)
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = c
        mask[i, :len(c)] = 1
    return torch.from_numpy(ids).to(device), torch.from_numpy(mask).to(device)


def micro_batches(chunks: List[np.ndarray], micro_tokens: int, n_virtual: int = 0):
    order = sorted(range(len(chunks)), key=lambda i: len(chunks[i]))
    out, cur, mx = [], [], 0
    for i in order:
        L = len(chunks[i]) + n_virtual
        if cur and max(mx, L) * (len(cur) + 1) > micro_tokens:
            out.append(cur)
            cur, mx = [], 0
        cur.append(i)
        mx = max(mx, L)
    if cur:
        out.append(cur)
    return out


def encode_chunks(model, chunks: List[np.ndarray], device, amp_ctx, micro_tokens: int,
                  grad: bool = False) -> torch.Tensor:
    """(N, d) float32 L2-normalised chunk vectors, in input order."""
    d = model.trunk.hidden_size
    out = torch.empty(len(chunks), d, device=device)
    ctx = torch.enable_grad() if grad else torch.no_grad()
    with ctx:
        for g in micro_batches(chunks, micro_tokens, model.n_virtual):
            ids, mask = collate([chunks[i] for i in g], device)
            with amp_ctx():
                e = model(ids, mask)
            out[torch.as_tensor(g, device=device)] = e.float()
    return out


def doc_vectors(chunk_vecs: torch.Tensor, owner: torch.Tensor, n_docs: int) -> torch.Tensor:
    """Normalised mean of each document's chunk vectors (differentiable)."""
    s = torch.zeros(n_docs, chunk_vecs.shape[1], device=chunk_vecs.device,
                    dtype=chunk_vecs.dtype).index_add(0, owner, chunk_vecs)
    return F.normalize(s, dim=-1)


In [ ]:
%%writefile /kaggle/working/phase7/pcr_train.py
"""
pcr_train.py -- ONE Phase-7 training job (one Phase-1 model x one technique), run as a
subprocess on ONE GPU by run_experiment.py. Reads RUN/logs/run_config.json.

    python pcr_train.py --run-dir RUN [--mode train|overfit]

train: chunk cache (+ BM25 channel) -> bi-encoder + technique -> critical self-checks on the
       real device (zero-init == Phase-1, GradCache gradient == exact gradient, gradients reach
       the trainable tensors, padding invariance) -> pre-flight (train s/step, encode tok/s) ->
       time plan (epochs / train fraction / inference chunks) -> contrastive training ->
       full dev + test encoding -> dev-tuned document aggregation (mean / max-chunk) and
       dense x BM25 fusion weight -> validated submission.zip (top-7 per test query) -> DONE.json

Loss (per step: B (query, positive) pairs, H BM25 hard negatives per query, every other
document of the batch as an in-batch negative; other relevant cases of a query are masked):
    L = 0.5 * [CE(q -> docs) + CE(positive -> queries)]     scores = s * cos, s learned
KD students (technique "kd"; teacher = the full-FT run of this notebook, online):
    L = alpha * L_student + (1 - alpha) * T^2 * KL(softmax(S_teacher / T) || softmax(S_student / T))
over exactly the same query x document score matrix.
GradCache (Gao et al. 2021): document vectors are computed without graph, the loss gradient
w.r.t. them is cached, then every micro-batch is re-encoded with graph and back-propagated
-> large contrastive batches at micro-batch memory (training runs without dropout so both
passes are identical; checked).
Exit codes as Phases 4-6: 0 ok, 1 error, 2 does not fit, 4 too many NaN steps, 5 watchdog,
6 critical self-check failed, 7 OOM at one chunk per micro-batch.
"""
import argparse
import copy
import json
import math
import os
import random
import sys
import time
import traceback
import zipfile
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn.functional as F

from cjpe_common import now_str, read_json, write_csv, write_json
from cjpe_models import assert_one_device, load_trainable_state, model_size
from cjpe_train import (Ctx, JobExit, amp_setup, is_oom, make_optimizer, make_scaler,
                        make_scheduler, peak_gb, save_best, save_last, set_seed)
from ner_models import torch_load
from pcr_data import load_cache
from pcr_ml import (K_EVAL, average_precision, eval_scores, fuse, mask_self, micro_prf_at_k,
                    topk, tune_fusion)
from pcr_model import (apply_pcr_technique, build_pcr_model, collate, doc_vectors,
                       encode_chunks, micro_batches, rebuild_pcr)

ZERO_INIT = ("lorma", "lora", "ia3", "full_ft", "partial_ft", "kd", "student_hard")
EPOCHS = {"full_ft": 4, "partial_ft": 4, "kd": 4, "student_hard": 4, "lorma": 5, "lora": 5,
          "qlora": 5, "ia3": 6, "prefix": 6, "prompt": 8}


# =============================================================================================
# data
# =============================================================================================
def pick_chunks(chunks: List[np.ndarray], k: int, rng=None) -> List[np.ndarray]:
    """k chunks: random subset (training) or evenly spaced (inference); order kept."""
    n = len(chunks)
    if n <= k:
        return list(chunks)
    if rng is not None:
        idx = sorted(rng.choice(n, k, replace=False).tolist())
    else:
        idx = sorted(set(np.linspace(0, n - 1, k).round().astype(int).tolist()))
    return [chunks[i] for i in idx]


def build_data(cfg: Dict, cache: Dict) -> Dict:
    tr = cache["splits"]["train"]
    top = cache["bm25"]["train_top"]
    pairs, hard = [], []
    for i, q in enumerate(tr["q"]):
        rel = set(q.get("rel") or [])
        hard.append([int(j) for j in top[i] if int(j) not in rel][:int(cfg.get("hard_pool", 30))])
        for j in sorted(rel):
            pairs.append((i, j))
    dv = cache["splits"]["dev"]
    rng = np.random.default_rng(int(cfg["seed"]))
    qs = [i for i, q in enumerate(dv["q"]) if q.get("rel")]
    qs = sorted(rng.permutation(qs)[:int(cfg.get("proxy_queries", 80))].tolist())
    R = int(cfg.get("proxy_depth", 25))
    bm = cache["bm25"]["dev"]
    cand = {i: sorted(set(topk(bm[i:i + 1], R)[0].tolist()) | set(dv["q"][i]["rel"]))
            for i in qs}
    docs = sorted(set(j for c in cand.values() for j in c))
    return {"pairs": pairs, "hard": hard, "proxy": {"queries": qs, "cand": cand, "docs": docs}}


def assert_integrity(cache: Dict, d: Dict, vocab: int, C: int) -> Dict:
    sp = cache["splits"]
    assert d["pairs"], "no (query, relevant candidate) training pairs"
    for s in ("train", "dev", "test"):
        assert sp[s]["pool"], f"{s}: empty candidate pool"
        for x in (sp[s]["q"][:300] + sp[s]["pool"][:300]):
            for c in x["chunks"]:
                assert 1 <= len(c) <= C and int(c.max()) < vocab, f"{s}: bad chunk"
        n = len(sp[s]["pool"])
        for q in sp[s]["q"]:
            assert all(0 <= j < n for j in (q.get("rel") or [])), f"{s}: relevant index"
    assert cache["bm25"]["dev"].shape == (len(sp["dev"]["q"]), len(sp["dev"]["pool"]))
    assert cache["bm25"]["test"].shape == (len(sp["test"]["q"]), len(sp["test"]["pool"]))
    return {"train_pairs": len(d["pairs"]), "pools_separate": True,
            "queries_without_hard_negatives": int(sum(1 for h in d["hard"] if not h))}


# =============================================================================================
# the step
# =============================================================================================
class Stepper:
    def __init__(self, ctx, model, opt, sched, scaler, tech, amp_ctx, cfg, cache, d,
                 teacher=None):
        self.ctx, self.model, self.opt, self.sched, self.scaler = ctx, model, opt, sched, scaler
        self.tech, self.amp_ctx, self.cfg, self.teacher = tech, amp_ctx, cfg, teacher
        self.tr = cache["splits"]["train"]
        self.d = d
        self.micro = int(cfg.get("micro_tokens", 8192))
        self.k_train = 1 if cfg.get("doc_repr") == "head" else int(cfg.get("k_train", 2))
        self.H = int(cfg.get("hard_negs", 3))
        self.skipped = self.overflow = 0
        self.params = [p for p in model.parameters() if p.requires_grad]

    # ---------------------------------------------------------------- batch construction
    def make_batch(self, pair_idx: List[int], rng) -> Dict:
        pairs = [self.d["pairs"][k] for k in pair_idx]
        qs = list(dict.fromkeys(q for q, _ in pairs))
        cand = list(dict.fromkeys(p for _, p in pairs))
        for q in qs:
            h = self.d["hard"][q]
            if h:
                for j in rng.choice(h, min(self.H, len(h)), replace=False).tolist():
                    if j not in cand:
                        cand.append(int(j))
        chunks, owner = [], []
        pick = (lambda x: x[:1]) if self.k_train == 1 else \
            (lambda x: pick_chunks(x, self.k_train, rng))
        docs = [self.tr["q"][q]["chunks"] for q in qs] + [self.tr["pool"][j]["chunks"] for j in cand]
        for n, ch in enumerate(docs):
            for c in pick(ch):
                chunks.append(c)
                owner.append(n)
        rel = [set(self.tr["q"][q].get("rel") or []) for q in qs]
        qpos = {q: i for i, q in enumerate(qs)}
        cpos = {c: j for j, c in enumerate(cand)}
        pos_q = [qpos[q] for q, _ in pairs]
        pos_c = [cpos[p] for _, p in pairs]
        relm = torch.zeros(len(qs), len(cand), dtype=torch.bool)
        for i in range(len(qs)):
            for j, c in enumerate(cand):
                relm[i, j] = c in rel[i]
        return {"chunks": chunks, "owner": owner, "nq": len(qs), "nc": len(cand),
                "pos_q": pos_q, "pos_c": pos_c, "relm": relm, "n_tokens": sum(map(len, chunks))}

    # ---------------------------------------------------------------- loss
    def loss_from_vectors(self, V: torch.Tensor, b: Dict, VT: Optional[torch.Tensor] = None):
        dev = V.device
        D = doc_vectors(V, torch.as_tensor(b["owner"], device=dev), b["nq"] + b["nc"])
        q, c = D[:b["nq"]], D[b["nq"]:]
        s = self.model.classifier.scale()
        S = s * q @ c.T                                                   # (nq, nc)
        relm = b["relm"].to(dev)
        pq = torch.as_tensor(b["pos_q"], device=dev)
        pc = torch.as_tensor(b["pos_c"], device=dev)
        n = len(b["pos_q"])
        keep = torch.ones(n, S.shape[1], dtype=torch.bool, device=dev)
        keep &= ~relm[pq]                                                 # other positives
        keep[torch.arange(n, device=dev), pc] = True
        L1 = S[pq].masked_fill(~keep, -1e4)
        loss = F.cross_entropy(L1, pc)
        S2 = (s * c[pc] @ q.T)                                            # (pairs, nq)
        keep2 = ~relm[:, pc].T                                            # queries citing it
        keep2[torch.arange(n, device=dev), pq] = True
        L2 = S2.masked_fill(~keep2, -1e4)
        loss = 0.5 * (loss + F.cross_entropy(L2, pq))
        if VT is not None:
            DT = doc_vectors(VT, torch.as_tensor(b["owner"], device=dev), b["nq"] + b["nc"])
            ST = self.teacher.classifier.scale().detach() * DT[:b["nq"]] @ DT[b["nq"]:].T
            T = float(self.cfg.get("kd_T", 2.0))
            a = float(self.cfg.get("kd_alpha", 0.5))
            lt = ST[pq].masked_fill(~keep, -1e4) / T
            kl = F.kl_div(F.log_softmax(L1 / T, -1), F.log_softmax(lt, -1), reduction="batchmean",
                          log_target=True)
            loss = a * loss + (1 - a) * T * T * kl
        return loss

    # ---------------------------------------------------------------- gradcache step
    def step(self, pair_idx, rng, gstep, total):
        while True:
            try:
                return self._step(pair_idx, rng, gstep, total)
            except RuntimeError as e:
                if not is_oom(e):
                    raise
                self.opt.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                one = int(self.cfg.get("chunk_tokens", 256)) + self.model.n_virtual
                if self.micro <= one:
                    raise JobExit(7, "CUDA OOM with one chunk per micro-batch")
                self.micro = max(one, self.micro // 2)
                self.ctx.log(f"   [oom] micro-batch -> {self.micro} tokens")

    def grads(self, b: Dict) -> float:
        """GradCache forward/backward of one batch; returns the loss."""
        model, dev = self.model, self.ctx.device
        model.eval()                                            # no dropout: passes identical
        V = encode_chunks(model, b["chunks"], dev, self.amp_ctx, self.micro).requires_grad_()
        VT = None
        if self.teacher is not None:
            VT = encode_chunks(self.teacher, b["chunks"], dev, self.amp_ctx, self.micro)
        loss = self.loss_from_vectors(V, b, VT)
        self.scaler.scale(loss).backward()
        G = V.grad
        for g in micro_batches(b["chunks"], self.micro, model.n_virtual):
            ids, mask = collate([b["chunks"][i] for i in g], dev)
            with self.amp_ctx():
                e = model(ids, mask)
            (e.float() * G[torch.as_tensor(g, device=dev)]).sum().backward()
        return float(loss.detach())

    def _step(self, pair_idx, rng, gstep, total):
        b = self.make_batch(pair_idx, rng)
        loss = self.grads(b)
        self.scaler.unscale_(self.opt)
        gn = torch.nn.utils.clip_grad_norm_(self.params, float(self.cfg.get("grad_clip", 1.0)))
        if not math.isfinite(loss):
            self.skipped += 1
            self.opt.zero_grad(set_to_none=True)
            self.scaler.update()
            return None, b["n_tokens"]
        if not torch.isfinite(gn):
            self.overflow += 1
        self.tech.after_backward(self.model, gstep, total)
        self.scaler.step(self.opt)
        self.scaler.update()
        self.opt.zero_grad(set_to_none=True)
        if self.sched is not None:
            self.sched.step()
        return loss, b["n_tokens"]


# =============================================================================================
# encoding whole splits + scores
# =============================================================================================
def encode_docs(model, docs: List[Dict], k: int, device, amp_ctx, micro: int):
    """-> (doc vectors (n, d), chunk vectors (N, d), owner (N,)) on CPU float32."""
    chunks, owner = [], []
    for n, x in enumerate(docs):
        for c in (x["chunks"][:1] if k == 1 else pick_chunks(x["chunks"], k)):
            chunks.append(c)
            owner.append(n)
    model.eval()
    V = encode_chunks(model, chunks, device, amp_ctx, micro)
    o = torch.as_tensor(owner, device=V.device)
    D = doc_vectors(V, o, len(docs))
    return D.cpu(), V.cpu(), o.cpu()


def split_scores(model, cache, split, k, device, amp_ctx, micro) -> Dict[str, np.ndarray]:
    sp = cache["splits"][split]
    Q, _, _ = encode_docs(model, sp["q"], k, device, amp_ctx, micro)
    D, V, o = encode_docs(model, sp["pool"], k, device, amp_ctx, micro)
    mean = (Q @ D.T).numpy()
    cs = Q @ V.T                                                          # (nq, N chunks)
    mx = torch.full((Q.shape[0], D.shape[0]), -2.0)
    mx = mx.scatter_reduce(1, o[None].expand(Q.shape[0], -1), cs, reduce="amax")
    qi, pi = [x["id"] for x in sp["q"]], [x["id"] for x in sp["pool"]]
    return {"mean": mask_self(mean, qi, pi), "max": mask_self(mx.numpy(), qi, pi)}


AGGS = {"mean": lambda s: s["mean"], "max": lambda s: s["max"],
        "mean+max": lambda s: 0.5 * (s["mean"] + s["max"])}


@torch.no_grad()
def proxy_eval(ctx, model, cache, d, amp_ctx, micro, k) -> Dict:
    """MAP / F1@7 of re-ranking each sampled dev query's BM25 top-25 + gold (cheap, smooth)."""
    sp = cache["splits"]["dev"]
    pr = d["proxy"]
    Q, _, _ = encode_docs(model, [sp["q"][i] for i in pr["queries"]], k, ctx.device, amp_ctx, micro)
    D, _, _ = encode_docs(model, [sp["pool"][j] for j in pr["docs"]], k, ctx.device, amp_ctx, micro)
    col = {j: n for n, j in enumerate(pr["docs"])}
    aps, ranked, gold = [], [], []
    for r, i in enumerate(pr["queries"]):
        c = pr["cand"][i]
        s = (Q[r:r + 1] @ D[[col[j] for j in c]].T)[0].numpy()
        g = {n for n, j in enumerate(c) if j in set(sp["q"][i]["rel"])}
        aps.append(average_precision(s, g))
        ranked.append(np.argsort(-s, kind="stable")[:K_EVAL].tolist())
        gold.append(g)
    m = micro_prf_at_k(ranked, gold)
    return {"proxy_map": float(np.mean(aps)) if aps else 0.0, "proxy_f1": m["micro_f1"]}


# =============================================================================================
# self-checks
# =============================================================================================
def job_checks(base, model, stepper, cfg, d, device, amp_ctx) -> Dict:
    res = {"checks": [], "critical_failed": [], "advisory_failed": []}
    tr = stepper.tr
    sample = [c for x in tr["q"][:3] + tr["pool"][:3] for c in x["chunks"][:2]]
    nullctx = lambda: torch.autocast("cpu", enabled=False)  # noqa: E731

    def add(name, kind, fn):
        rec = {"name": name, "kind": kind}
        try:
            v = fn()
            rec.update(v if isinstance(v, dict) else {"verdict": "pass" if v else "fail"})
        except Exception as e:
            rec.update({"verdict": "fail", "error": f"{type(e).__name__}: {e}"[:400]})
        res["checks"].append(rec)
        if rec["verdict"] == "fail":
            res["critical_failed" if kind == "critical" else "advisory_failed"].append(name)

    def zero_init():
        a = encode_chunks(base, sample, device, nullctx, 10 ** 6)
        b = encode_chunks(model, sample, device, nullctx, 10 ** 6)
        v = float((a - b).abs().max())
        return {"verdict": "pass" if v < 2e-3 else "fail", "value": v, "threshold": 2e-3}
    if base is not None:
        add("zero_init_equals_phase1", "critical", zero_init)

    def gradcache_exact():
        rng = np.random.default_rng(0)
        b = stepper.make_batch(list(range(min(4, len(d["pairs"])))), rng)
        b["chunks"] = [c[:48] for c in b["chunks"]]
        old_amp, old_scaler = stepper.amp_ctx, stepper.scaler
        stepper.amp_ctx, stepper.scaler = nullctx, make_scaler(False)
        try:
            stepper.opt.zero_grad(set_to_none=True)
            stepper.micro = max(64, 2 * max(len(c) for c in b["chunks"]) + model.n_virtual)
            stepper.grads(b)
            g1 = [None if p.grad is None else p.grad.detach().clone() for p in stepper.params]
            stepper.opt.zero_grad(set_to_none=True)
            V = encode_chunks(model, b["chunks"], device, nullctx, 10 ** 6, grad=True)
            stepper.loss_from_vectors(V, b).backward()
            g2 = [None if p.grad is None else p.grad.detach().clone() for p in stepper.params]
            stepper.opt.zero_grad(set_to_none=True)
        finally:
            stepper.amp_ctx, stepper.scaler = old_amp, old_scaler
            stepper.micro = int(cfg.get("micro_tokens", 8192))
        num = sum(float((a - b_).abs().max()) for a, b_ in zip(g1, g2) if a is not None
                  and b_ is not None)
        den = sum(float(b_.abs().max()) for b_ in g2 if b_ is not None) + 1e-12
        nz = sum(1 for a in g1 if a is not None and float(a.abs().max()) > 0)
        v = num / den
        ok = v < 1e-3 and nz >= 0.5 * len(stepper.params)
        return {"verdict": "pass" if ok else "fail", "value": v, "threshold": 1e-3,
                "nonzero_grad_tensors": nz, "trainable_tensors": len(stepper.params)}
    add("gradcache_equals_exact_gradient_and_grads_flow", "critical", gradcache_exact)

    def padding():
        c = sample[:3]
        one = torch.cat([encode_chunks(model, [x], device, nullctx, 10 ** 6) for x in c])
        many = encode_chunks(model, c, device, nullctx, 10 ** 6)
        v = float((one - many).abs().max())
        return {"verdict": "pass" if v < 2e-3 else "fail", "value": v, "threshold": 2e-3}
    add("padding_invariance", "advisory", padding)
    add("pools_never_crossed", "critical", lambda: d["proxy"]["docs"] and max(d["proxy"]["docs"])
        < len(stepper.ctx_cache["splits"]["dev"]["pool"]))
    res["selfcheck"] = "fail" if res["critical_failed"] else (
        "pass" if not res["advisory_failed"] else "pass (advisory: "
        + ",".join(res["advisory_failed"]) + ")")
    return res


# =============================================================================================
# pre-flight + plan
# =============================================================================================
def measure(ctx, stepper, cache, amp_ctx, k_inf: int, n_steps: int = 3):
    rng = np.random.default_rng(1)
    snap = {n: p.detach().clone() for n, p in stepper.model.named_parameters() if p.requires_grad}
    B = int(stepper.cfg.get("batch_pairs", 16))
    pairs = list(range(len(stepper.d["pairs"])))
    t, n = 0.0, 0
    for k in range(n_steps + 1):
        idx = rng.choice(pairs, min(B, len(pairs)), replace=False).tolist()
        sync()
        t0 = time.time()
        b = stepper.make_batch(idx, rng)
        stepper.grads(b)
        stepper.opt.zero_grad(set_to_none=True)
        sync()
        if k:
            t += time.time() - t0
            n += 1
    with torch.no_grad():
        for nm, p in stepper.model.named_parameters():
            if nm in snap:
                p.copy_(snap[nm])
    sps = t / max(1, n)
    sp = cache["splits"]["dev"]
    docs = (sp["pool"] * 3)[:96]
    chunks = [c for x in docs for c in pick_chunks(x["chunks"], k_inf)][:384]
    sync()
    t0 = time.time()
    encode_chunks(stepper.model, chunks, ctx.device, amp_ctx, stepper.micro)
    sync()
    r_inf = sum(map(len, chunks)) / max(1e-6, time.time() - t0)
    return sps, r_inf


def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def infer_tokens(cache, k: int) -> int:
    n = 0
    for s in ("dev", "test"):
        for part in ("q", "pool"):
            for x in cache["splits"][s][part]:
                n += sum(len(c) for c in pick_chunks(x["chunks"], k))
    return n


def make_plan(ctx, cache, d, sps, r_inf, proxy_s, k_inf) -> Dict:
    cfg = ctx.cfg
    B = int(cfg.get("batch_pairs", 16))
    spe = max(1, math.ceil(len(d["pairs"]) / B))
    fixed = cfg.get("fixed_plan") or {}
    want = int(fixed.get("epochs", cfg.get("epochs_max") or EPOCHS.get(cfg["technique"], 4)))
    budget = float(cfg["budget_s"]) - (time.time() - ctx.t0)
    k = k_inf
    while True:
        inf_s = 1.25 * infer_tokens(cache, k) / max(1.0, r_inf) + 120
        n_evals = int(cfg.get("evals_per_epoch", 2)) * max(1, want) + 1
        avail = budget - inf_s - n_evals * proxy_s - 60
        steps = int(avail / max(sps, 1e-6))
        frac = min(1.0, float(fixed.get("frac", 1.0)))
        epochs = want if want == 0 else max(0, min(want, steps // max(1, int(spe * frac))))
        reason = "full plan"
        if want and epochs == 0:
            frac = steps / spe
            epochs = 1
            reason = "less than one epoch fits"
        if frac >= float(cfg.get("min_train_fraction", 0.1)) or want == 0 or k <= 2:
            break
        k = max(2, k // 2)
    if want and frac < float(cfg.get("min_train_fraction", 0.1)):
        raise JobExit(2, f"only {max(0, steps)} steps fit ({sps:.1f}s/step, inference "
                         f"{inf_s / 60:.0f} min) -- below the minimum train fraction")
    used = max(1, int(round(frac * len(d["pairs"])))) if want else 0
    return {"epochs": int(epochs), "train_fraction": round(frac, 4), "train_pairs": len(d["pairs"]),
            "train_pairs_used": used, "steps_per_epoch": max(1, math.ceil(used / B)),
            "sec_per_step": sps, "infer_tokens_per_s": r_inf, "k_infer": k,
            "reserve_min": inf_s / 60, "proxy_s": proxy_s, "budget_min": budget / 60,
            "projected_minutes": (epochs * math.ceil(used / B) * sps + inf_s + n_evals * proxy_s)
            / 60, "reason": reason if k == k_inf else f"{reason}; inference chunks {k_inf}->{k}"}


# =============================================================================================
# outputs
# =============================================================================================
def finish(ctx, model, cache, d, amp_ctx, st, plan, size, checks, stop, tech, micro) -> int:
    cfg = ctx.cfg
    if ctx.device.type == "cuda":
        torch.cuda.empty_cache()
    ctx.status("evaluating")
    k = int(plan["k_infer"]) if cfg.get("doc_repr") != "head" else 1
    t = time.time()
    sd = split_scores(model, cache, "dev", k, ctx.device, amp_ctx, micro)
    ctx.status("predicting")
    stt = split_scores(model, cache, "test", k, ctx.device, amp_ctx, micro)
    dt = time.time() - t
    sp = cache["splits"]
    n_docs = sum(len(sp[s][p]) for s in ("dev", "test") for p in ("q", "pool"))
    dev_q = sp["dev"]["q"]
    gold = [q.get("rel") or [] for q in dev_q]
    bm_dev, bm_test = cache["bm25"]["dev"], cache["bm25"]["test"]
    grid = [round(x, 2) for x in np.arange(0.0, 1.0001, 0.1)]
    wmin = float(cfg.get("min_dense_weight", 0.2))
    rows, best = [], None
    for agg, f in AGGS.items():
        _w, _b, curve = tune_fusion(f(sd), bm_dev, gold, grid)
        for r in curve:
            r["doc_agg"] = agg
            rows.append(r)
            if r["dense_weight"] >= wmin - 1e-9 and (best is None or r["micro_f1"] >=
                                                     best["micro_f1"] - 1e-12):   # ties: more dense
                best = r
    dense = {a: next(r for r in rows if r["doc_agg"] == a and r["dense_weight"] == 1.0)
             for a in AGGS}
    bm_only = next(r for r in rows if r["dense_weight"] == 0.0)
    agg, w = best["doc_agg"], best["dense_weight"]
    S_dev = fuse(AGGS[agg](sd), bm_dev, w)
    S_test = fuse(AGGS[agg](stt), bm_test, w)
    out_dev = os.path.join(ctx.run_dir, "dev")
    os.makedirs(out_dev, exist_ok=True)
    write_csv(os.path.join(out_dev, "fusion_curve.csv"), rows,
              ["doc_agg", "dense_weight", "micro_f1", "micro_p", "micro_r", "tp", "n_pred",
               "n_gold"])
    rk = topk(S_dev, K_EVAL)
    rk_dense = topk(zrow_safe(AGGS[agg](sd)), 100)
    pool_ids = [x["id"] for x in sp["dev"]["pool"]]
    with open(os.path.join(out_dev, "dev_predictions.jsonl"), "w") as f:
        for i, q in enumerate(dev_q):
            g = set(gold[i])
            hit = [int(j) in g for j in rk[i]]
            rpos = [n + 1 for n, j in enumerate(rk_dense[i]) if int(j) in g]
            f.write(json.dumps({"id": q["id"], "predicted": [pool_ids[j] for j in rk[i]],
                                "gold": [pool_ids[j] for j in sorted(g)], "hits": int(sum(hit)),
                                "n_gold": len(g), "n_chunks": len(q["chunks"]),
                                "n_chars": q.get("n_chars"),
                                "dense_gold_ranks": rpos[:20],
                                "dense_recall_at_100": len(rpos) / max(1, len(g))}) + "\n")
    np.savez_compressed(os.path.join(out_dev, "scores.npz"),
                        dev=AGGS[agg](sd).astype(np.float16), test=AGGS[agg](stt).astype(np.float16))
    # ---- submission --------------------------------------------------------------------
    sub = os.path.join(ctx.run_dir, "submission")
    os.makedirs(sub, exist_ok=True)
    tq, tp = sp["test"]["q"], [x["id"] for x in sp["test"]["pool"]]
    rt = topk(S_test, K_EVAL)
    pred = {q["id"]: [tp[j] for j in rt[i]] for i, q in enumerate(tq)}
    with open(os.path.join(sub, "prediction.json"), "w", encoding="utf-8") as f:
        json.dump(pred, f)
    zp = os.path.join(sub, "submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(sub, "prediction.json"), "prediction.json")
    from validate_submission import validate
    v = validate(zp, [q["id"] for q in tq], tp)
    rec100 = float(np.mean([json.loads(l)["dense_recall_at_100"] for l in
                            open(os.path.join(out_dev, "dev_predictions.jsonl"))] or [0]))
    rep = {"chosen": {"doc_agg": agg, "dense_weight": w, **best}, "dense_only": dense,
           "bm25_only": bm_only, "bm25_params": cache["bm25"]["params"], "k_infer": k}
    write_json(os.path.join(out_dev, "dev_report.json"), rep)
    model.cpu()
    metrics = {
        "dev_score": best["micro_f1"], "score_metric": "micro_f1@7 (dense x BM25, dev-tuned)",
        "dev_micro_f1": best["micro_f1"], "dev_micro_p": best["micro_p"],
        "dev_micro_r": best["micro_r"], "dense_weight": w, "doc_agg": agg,
        "dev_dense_f1": dense[agg]["micro_f1"], "dev_dense_mean_f1": dense["mean"]["micro_f1"],
        "dev_dense_max_f1": dense["max"]["micro_f1"], "dev_bm25_f1": bm_only["micro_f1"],
        "dev_dense_recall_at_100": rec100, "dev_proxy_map": st.get("best"),
        "dev_proxy_map_init": st.get("init_proxy"), "k_infer": k,
        "k_train": 1 if cfg.get("doc_repr") == "head" else int(cfg.get("k_train", 2)),
        "doc_repr": cfg.get("doc_repr"), "pooling": cfg.get("pooling"),
        "attn_mode": cfg.get("attn_mode"), "chunk_tokens": cache["C"],
        "best_epoch": st["best_epoch"],
        "epochs_run": st["evals"][-1]["epoch"] if st["evals"] else 0, "steps": st["step"],
        "skipped_steps": st["skipped"], "stop_reason": stop,
        "train_minutes": st["train_seconds"] / 60, "total_minutes": (time.time() - ctx.t0) / 60,
        "ms_per_doc_infer": 1000 * dt / max(1, n_docs), "peak_mem_gb": peak_gb(),
        "train_fraction": plan["train_fraction"], "epochs_planned": plan["epochs"],
        "train_pairs": plan["train_pairs"], "train_queries": len(sp["train"]["q"]),
        "tokens_per_s": plan.get("train_tokens_per_s"),
        "infer_tokens_per_s": plan["infer_tokens_per_s"],
        "projected_minutes": plan["projected_minutes"], "selfcheck": checks.get("selfcheck"),
        "validation_ok": v["ok"], "degenerate": v.get("degenerate"),
        "technique_report": tech.final_report(model), **size,
    }
    write_json(os.path.join(ctx.run_dir, "metrics.json"), metrics)
    if not v["ok"]:
        ctx.status("failed", f"submission validation failed: {v['errors']}")
        return 4
    write_json(os.path.join(ctx.run_dir, "DONE.json"),
               {"finished": now_str(), "dev_score": best["micro_f1"], "stop_reason": stop,
                "submission": zp})
    ctx.status("degenerate" if v.get("degenerate") else "done",
               f"dev micro-F1@7 {best['micro_f1']:.4f}")
    ctx.log(f"   DONE dev Micro-F1@7 {best['micro_f1']:.4f} (P {best['micro_p']:.4f} R "
            f"{best['micro_r']:.4f}) with {agg} aggregation, dense weight {w}; dense only "
            f"{dense[agg]['micro_f1']:.4f}, BM25 only {bm_only['micro_f1']:.4f}; dense recall@100 "
            f"{rec100:.3f}; submission validated")
    return 0


def zrow_safe(S):
    from pcr_ml import zrow
    return zrow(S)


# =============================================================================================
# the job
# =============================================================================================
def run_train(ctx: Ctx, mode: str) -> int:
    cfg = ctx.cfg
    set_seed(int(cfg["seed"]))
    dev = ctx.device
    amp_dtype, amp_ctx = amp_setup(dev)
    ctx.status("preflight")
    ctx.log(f"start {cfg['experiment']}/{cfg['technique']}/{cfg['model']} on {dev}"
            f"{' (' + torch.cuda.get_device_name(0) + ')' if dev.type == 'cuda' else ''}, "
            f"budget {cfg['budget_s'] / 60:.0f} min, attempt {cfg.get('attempt', 1)}")
    if dev.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    inj = os.environ.get("PHASE7_INJECT_FAIL", "")
    if inj and inj.split(":")[0] == cfg["job"] and int(cfg.get("attempt", 1)) == 1:
        raise JobExit(int(inj.split(":")[1]), "injected failure (test)")
    cache = load_cache(cfg["cache_path"])
    d = build_data(cfg, cache)

    model, p1cfg = build_pcr_model(cfg["phase1_dir"], cfg.get("layer_ids"),
                                   cfg.get("attn_mode", "causal"), cfg.get("pooling", "mean"))
    vocab = int(p1cfg["config"]["vocab_size"])
    base = copy.deepcopy(model) if (cfg["technique"] in ZERO_INIT and
                                    not cfg.get("skip_job_checks")) else None
    sample = cache["splits"]["train"]["q"][0]["chunks"][0][:64].tolist()
    groups, tech = apply_pcr_technique(model, cfg["technique"], cfg, ctx.log, sample)
    model.to(dev)
    assert_one_device(model, dev)
    size = model_size(model)
    write_json(os.path.join(ctx.run_dir, "logs", "model_size.json"), size)
    integ = assert_integrity(cache, d, vocab, int(cache["C"]))
    write_json(os.path.join(ctx.run_dir, "logs", "integrity.json"), integ)
    sp = cache["splits"]
    ctx.log(f"   data: train {len(sp['train']['q'])} queries / {len(d['pairs'])} pairs / pool "
            f"{len(sp['train']['pool'])}; dev {len(sp['dev']['q'])} / {len(sp['dev']['pool'])}; "
            f"test {len(sp['test']['q'])} / {len(sp['test']['pool'])}; chunks of {cache['C']} "
            f"tokens (<= {cache['K']} per case); proxy {len(d['proxy']['queries'])} dev queries")

    teacher = None
    if cfg["technique"] == "kd":
        tdir = cfg["teacher_dir"]
        trc = read_json(os.path.join(tdir, "logs", "run_config.json"))
        teacher = rebuild_pcr(trc, os.path.join(tdir, "checkpoints", "best.pt"), dev, sample)
        for p in teacher.parameters():
            p.requires_grad = False
        ctx.log(f"   [kd] teacher {trc['model']} ({trc['technique']}) loaded from {tdir}; "
                f"alpha {cfg.get('kd_alpha')} T {cfg.get('kd_T')}")

    opt = make_optimizer(groups)
    scaler = make_scaler(amp_dtype == torch.float16)
    stepper = Stepper(ctx, model, opt, None, scaler, tech, amp_ctx, cfg, cache, d, teacher)
    stepper.ctx_cache = cache

    checks = {"selfcheck": "skipped"}
    if not cfg.get("skip_job_checks"):
        if base is not None:
            base.to(dev)
        tch, stepper.teacher = stepper.teacher, None
        checks = job_checks(base, model, stepper, cfg, d, dev, amp_ctx)
        stepper.teacher = tch
    del base
    write_json(os.path.join(ctx.run_dir, "logs", "job_checks.json"), checks)
    for c in checks.get("checks", []):
        v = c.get("value")
        ctx.log(f"   [check] {c['name']:<46} {c['kind']:<9} {c['verdict']}"
                + (f" value={v:.3g}" if isinstance(v, float) else "")
                + (f"  {c.get('error', '')[:200]}" if c["verdict"] == "fail" else ""))
    if checks.get("critical_failed"):
        raise JobExit(6, f"critical self-check failed: {checks['critical_failed']}")
    if mode == "overfit":
        return run_overfit(ctx, stepper)

    # ---- pre-flight + plan ------------------------------------------------------------------
    plan_path = os.path.join(ctx.run_dir, "logs", "plan.json")
    last_pt = os.path.join(ctx.run_dir, "checkpoints", "last.pt")
    k_inf = 1 if cfg.get("doc_repr") == "head" else int(cfg.get("k_infer", 8))
    k_proxy = 1 if cfg.get("doc_repr") == "head" else min(k_inf, 4)
    resume = None
    if os.path.exists(last_pt) and os.path.exists(plan_path):
        try:
            resume = torch_load(last_pt)
            plan = read_json(plan_path)
            ctx.log(f"   [resume] from step {resume['st']['step']} (plan reused)")
        except Exception as e:
            ctx.log(f"   [resume] unreadable last.pt ({e}); starting fresh")
            resume = None
    init = None
    if resume is None:
        sps, r_inf = measure(ctx, stepper, cache, amp_ctx, k_inf)
        t0 = time.time()
        init = proxy_eval(ctx, model, cache, d, amp_ctx, stepper.micro, k_proxy)
        proxy_s = time.time() - t0
        ctx.log(f"   [preflight] train {sps:.2f} s/step ({cfg.get('batch_pairs')} pairs + "
                f"{cfg.get('hard_negs')} hard negatives each), encode {r_inf:,.0f} tok/s, proxy "
                f"eval {proxy_s:.0f}s, micro-batch {stepper.micro} tokens, peak {peak_gb():.2f} GB")
        ctx.log(f"   [zero-shot] Phase-1 representation (before training): dev proxy MAP "
                f"{init['proxy_map']:.4f}, F1@7 {init['proxy_f1']:.4f}")
        plan = make_plan(ctx, cache, d, sps, r_inf, proxy_s, k_inf)
        plan["micro_tokens"] = stepper.micro
        plan["selfcheck"] = checks.get("selfcheck")
        plan["init_proxy"] = init["proxy_map"]
        write_json(plan_path, plan)
    stepper.micro = int(plan.get("micro_tokens") or stepper.micro)
    ctx.log(f"   [plan] CHOSEN: epochs={plan['epochs']} train_fraction={plan['train_fraction']} "
            f"({plan['train_pairs_used']}/{plan['train_pairs']} pairs), inference chunks "
            f"{plan['k_infer']}, inference reserve {plan['reserve_min']:.0f} min, projected "
            f"{plan['projected_minutes']:.0f} of {plan['budget_min']:.0f} min ({plan['reason']})")

    B = int(cfg.get("batch_pairs", 16))
    used = int(plan["train_pairs_used"])
    rng0 = np.random.default_rng(int(cfg["seed"]))
    keep = sorted(rng0.permutation(len(d["pairs"]))[:max(1, used)].tolist())
    spe = max(1, math.ceil(len(keep) / B))
    total = spe * int(plan["epochs"])
    sched = make_scheduler(opt, max(1, total), float(cfg.get("warmup_frac", 0.06)))
    stepper.sched = sched
    evals = int(cfg.get("evals_per_epoch", 2))
    eval_at = sorted({max(1, round(spe * (k + 1) / evals)) for k in range(evals)})
    patience = int(cfg.get("patience", 2)) * evals
    st = {"step": 0, "epoch": 0, "pos": 0, "best": -1.0, "best_epoch": None, "bad": 0,
          "evals": [], "skipped": 0, "train_seconds": 0.0, "init_proxy": plan.get("init_proxy")}
    if resume is not None:
        load_trainable_state(model, resume["model"])
        opt.load_state_dict(resume["opt"])
        sched.load_state_dict(resume["sched"])
        scaler.load_state_dict(resume["scaler"])
        st.update(resume["st"])
        try:
            random.setstate(resume["rng"]["py"])
            np.random.set_state(resume["rng"]["np"])
            torch.set_rng_state(resume["rng"]["torch"])
        except Exception:
            pass
        del resume
    elif init is not None:
        st["best"], st["best_epoch"] = init["proxy_map"], 0.0
        save_best(ctx, model)
    deadline = ctx.t0 + float(cfg["budget_s"]) - 60 * float(plan["reserve_min"])
    ctx.status("training", plan=plan["reason"])
    ctx.log(f"   [train] {len(keep)} pairs/epoch, {spe} steps/epoch x {plan['epochs']} = {total} "
            f"steps; training deadline in {(deadline - time.time()) / 60:.0f} min")
    last_ck, t_train, stop = time.time(), time.time(), "completed"
    kill_at = int(cfg.get("kill_at_step", -1))
    ema, tok_n, tok_t = None, 0, 0.0
    while st["epoch"] < int(plan["epochs"]):
        order = [keep[i] for i in np.random.default_rng(int(cfg["seed"]) + 1000 * st["epoch"])
                 .permutation(len(keep))]
        rng = np.random.default_rng(int(cfg["seed"]) + 7 + 1000 * st["epoch"] + st["pos"])
        while st["pos"] < spe:
            if time.time() > deadline:
                stop = "time guard"
                break
            batch = order[st["pos"] * B:(st["pos"] + 1) * B]
            if not batch:
                st["pos"] = spe
                break
            t1 = time.time()
            loss, ntok = stepper.step(batch, rng, st["step"], total)
            tok_n, tok_t = tok_n + ntok, tok_t + time.time() - t1
            st["step"] += 1
            st["pos"] += 1
            if loss is None:
                st["skipped"] += 1
                if st["step"] >= 20 and st["skipped"] > 0.05 * st["step"]:
                    raise JobExit(4, f"{st['skipped']} of {st['step']} steps non-finite")
            else:
                ema = loss if ema is None else 0.95 * ema + 0.05 * loss
            if st["step"] % 25 == 0:
                ctx.log(f"   step {st['step']}/{total} ep {st['epoch'] + 1} loss {ema:.4f} lr "
                        f"{sched.get_last_lr()[0]:.2e} scale {float(model.classifier.scale().detach()):.1f}"
                        f" peak {peak_gb():.1f} GB skipped {st['skipped']}")
            if st["step"] == kill_at:
                save_last(ctx, model, opt, sched, scaler, st)
                os._exit(137)
            if st["pos"] in eval_at:
                pe = proxy_eval(ctx, model, cache, d, amp_ctx, stepper.micro, k_proxy)
                fe = st["epoch"] + st["pos"] / spe
                rec = {"type": "eval", "step": st["step"], "epoch": round(fe, 3),
                       "train_loss": ema, "dev_proxy_map": pe["proxy_map"],
                       "dev_proxy_f1": pe["proxy_f1"], "lr": sched.get_last_lr()[0],
                       "minutes": (time.time() - ctx.t0) / 60, "skipped_steps": st["skipped"]}
                ctx.jlog(rec)
                st["evals"].append(rec)
                ctx.log(f"   [eval] epoch {fe:.2f}: dev proxy MAP {pe['proxy_map']:.4f} F1@7 "
                        f"{pe['proxy_f1']:.4f} (zero-shot {st['init_proxy'] or 0:.4f})")
                if ema is not None and not math.isfinite(ema):
                    raise JobExit(5, "watchdog: non-finite training loss")
                if pe["proxy_map"] > st["best"] + 1e-6:
                    st["best"], st["best_epoch"], st["bad"] = pe["proxy_map"], round(fe, 3), 0
                    save_best(ctx, model)
                else:
                    st["bad"] += 1
                write_json(os.path.join(ctx.run_dir, "logs", "progress.json"),
                           {"best_proxy_map": st["best"], "best_epoch": st["best_epoch"],
                            "step": st["step"], "total_steps": total, "epochs_run": round(fe, 3)})
                if st["bad"] >= patience:
                    stop = "early stopping"
                    break
            if time.time() - last_ck > 60 * float(cfg.get("ckpt_every_min", 30)):
                save_last(ctx, model, opt, sched, scaler, st)
                last_ck = time.time()
        if stop != "completed":
            break
        st["epoch"] += 1
        st["pos"] = 0
    st["train_seconds"] = time.time() - t_train
    plan["train_tokens_per_s"] = tok_n / max(1e-6, tok_t)
    ctx.log(f"   [train] stopped: {stop} at step {st['step']}/{total}")
    stepper.teacher = teacher = None
    best_pt = os.path.join(ctx.run_dir, "checkpoints", "best.pt")
    tech.force_final(model)
    if not os.path.exists(best_pt):
        save_best(ctx, model)
    load_trainable_state(model, torch_load(best_pt))
    if os.path.exists(last_pt):
        os.remove(last_pt)
    return finish(ctx, model, cache, d, amp_ctx, st, plan, size, checks, stop, tech, stepper.micro)


def run_overfit(ctx, stepper) -> int:
    """4 fixed pairs, up to 60 steps: the contrastive loss must fall below 30% of its start."""
    idx = list(range(min(4, len(stepper.d["pairs"]))))
    for g in stepper.opt.param_groups:
        g["lr"] = g["initial_lr"] = max(g["lr"], float(ctx.cfg.get("overfit_lr", 3e-3)))
    stepper.sched = None
    first = last = None
    n = 0
    for n in range(1, 61):
        l, _ = stepper.step(idx, np.random.default_rng(0), n, 60)
        if l is not None:
            first = l if first is None else first
            last = l
        if last is not None and last < 0.3 * first:
            break
    ok = last is not None and first is not None and last < 0.3 * first
    res = {"passed": bool(ok), "first_loss": first, "final_loss": last, "steps": n}
    write_json(os.path.join(ctx.run_dir, "logs", "overfit.json"), res)
    ctx.status("done", f"overfit {res}")
    return 0 if ok else 1


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--run-dir", required=True)
    ap.add_argument("--mode", default="train", choices=["train", "overfit"])
    a = ap.parse_args()
    cfg = read_json(os.path.join(a.run_dir, "logs", "run_config.json"))
    ctx = Ctx(a.run_dir, cfg)
    code, reason = 1, "unknown"
    try:
        code = run_train(ctx, a.mode)
        reason = "ok" if code == 0 else f"exit {code}"
    except JobExit as e:
        code, reason = e.code, e.reason
        ctx.log(f"   JOB EXIT {code}: {reason}")
    except Exception as e:
        code = 7 if is_oom(e) else 1
        reason = f"{type(e).__name__}: {e}"
        ctx.log(f"   JOB ERROR: {reason}\n{traceback.format_exc()[-3000:]}")
    finally:
        st = read_json(os.path.join(a.run_dir, "status.json"), {}) or {}
        if code != 0 and st.get("state") not in ("failed",):
            ctx.status("failed", reason, exit_code=code)
        elif code == 0 and st.get("state") not in ("done", "degenerate"):
            ctx.status("done", reason)
    sys.stdout.flush()
    sys.exit(code)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase7/validate_submission.py
"""
validate_submission.py -- Phase-7 submission check (PDF 13.3.5).

submission.zip must hold exactly `prediction.json` at its root: ONE JSON object
{"<test query id>": ["<cand id>", ... 7 ids]} -- exactly 7 unique ids per query, every id from
the test candidate pool, a query never lists itself, one entry for every test query.

    python validate_submission.py submission.zip [--test-ids test_ids.json --pool pool_ids.json]
"""
import argparse
import json
import zipfile
from typing import Dict, List, Optional

K = 7


def validate(zip_path: str, test_ids: Optional[List[str]] = None,
             pool_ids: Optional[List[str]] = None) -> Dict:
    errors: List[str] = []
    try:
        with zipfile.ZipFile(zip_path) as z:
            names = [n for n in z.namelist() if not n.endswith("/")]
            if names != ["prediction.json"]:
                errors.append(f"zip must contain only prediction.json at its root, has {names}")
                if "prediction.json" not in names:
                    return {"ok": False, "errors": errors}
            obj = json.loads(z.read("prediction.json").decode("utf-8"))
    except Exception as e:
        return {"ok": False, "errors": [f"unreadable: {type(e).__name__}: {e}"]}
    if not isinstance(obj, dict):
        return {"ok": False, "errors": [f"prediction.json must be an object, got {type(obj).__name__}"]}
    pool = set(pool_ids) if pool_ids is not None else None
    for q, r in obj.items():
        if not isinstance(r, list) or len(r) != K:
            errors.append(f"{q}: needs a list of exactly {K} ids")
            continue
        if not all(isinstance(x, str) and x for x in r):
            errors.append(f"{q}: ids must be non-empty strings")
        if len(set(r)) != K:
            errors.append(f"{q}: duplicate ids")
        if q in r:
            errors.append(f"{q}: lists itself")
        if pool is not None and not set(r) <= pool:
            errors.append(f"{q}: {len(set(r) - pool)} ids not in the test candidate pool")
        if len(errors) > 20:
            break
    if test_ids is not None:
        miss = set(test_ids) - set(obj)
        extra = set(obj) - set(test_ids)
        if miss:
            errors.append(f"{len(miss)} test queries missing (e.g. {sorted(miss)[:3]})")
        if extra:
            errors.append(f"{len(extra)} unknown query ids (e.g. {sorted(extra)[:3]})")
    lists = {tuple(v) for v in obj.values() if isinstance(v, list)}
    return {"ok": not errors, "errors": errors[:20], "n": len(obj),
            "degenerate": len(obj) > 3 and len(lists) == 1}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("zip")
    ap.add_argument("--test-ids")
    ap.add_argument("--pool")
    a = ap.parse_args()
    t = json.load(open(a.test_ids)) if a.test_ids else None
    p = json.load(open(a.pool)) if a.pool else None
    print(json.dumps(validate(a.zip, t, p), indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase7/select_best.py
"""
select_best.py -- per-experiment BEST selection and the six-upload plan (sections 11.2, 11.3).

    python select_best.py --exp-dir /kaggle/working/phase7_out/E2_lorma      # BEST/ for one exp
    python select_best.py --plan [--out-root ...]                             # UPLOAD_PLAN.csv

BEST: highest dev score among DONE jobs (not DEGENERATE, not the E6 teacher); ties -> fewer
trainable parameters.

Plan: exactly one row per counted experiment E1..E6, every Phase-1 model at least once (relaxed
only when impossible, and then reported), FAILED / DEGENERATE / missing-zip rows excluded,
maximise the mean dev score, and the six prediction.json files must be pairwise different
(identical submissions do not count on Codabench). Exhaustive search (<= 4^6 combinations).
"""
import argparse
import hashlib
import itertools
import json
import os
import shutil
import zipfile
from typing import Dict, List, Optional

from cjpe_common import (COUNTED, EXPERIMENTS, MODELS, OUT_ROOT, default_input_roots,
                         read_csv, read_json, write_csv, write_json)


def _f(x, default=-1.0) -> float:
    try:
        return float(x)
    except (TypeError, ValueError):
        return default


def _unify(rows: List[Dict]) -> List[Dict]:
    """dev_score := Micro-F1@7 of the submitted (dev-tuned) ranking for every row."""
    from aggregate import common_metric
    m = common_metric(rows)
    for r in rows:
        r["dev_score"] = r.get(m)
    return rows


def prediction_hash(zip_path: str) -> Optional[str]:
    try:
        with zipfile.ZipFile(zip_path) as z:
            items = json.loads(z.read("prediction.json"))
        m = sorted((str(k), list(map(str, v))) for k, v in items.items())
        return hashlib.md5(json.dumps(m).encode()).hexdigest()
    except Exception:
        return None


def is_candidate(r: Dict, require_technique: Optional[str] = None) -> bool:
    if r.get("status") != "DONE":
        return False
    if r.get("technique", "").endswith("(teacher)"):
        return False
    if require_technique and r.get("technique") != require_technique:
        return False
    z = r.get("submission_zip")
    return bool(z) and os.path.exists(z)


def select_exp(exp_dir: str, rows: List[Dict], log=print) -> Optional[Dict]:
    exp = os.path.basename(exp_dir.rstrip("/"))
    rows = _unify(rows)
    cands = [r for r in rows if is_candidate(r)]
    best_dir = os.path.join(exp_dir, "BEST")
    if not cands:
        log(f"   [select] {exp}: no finished, non-degenerate model -> no BEST/")
        degen = [r for r in rows if r.get("status") == "DEGENERATE" and r.get("submission_zip")
                 and os.path.exists(r["submission_zip"])]
        if not degen:
            return None
        log(f"   [select] {exp}: only DEGENERATE models exist; keeping the best of them in BEST/ "
            f"(flagged)")
        cands = degen
    cands.sort(key=lambda r: (-_f(r.get("dev_score")), _f(r.get("trainable_params"), 1e18)))
    w = cands[0]
    os.makedirs(best_dir, exist_ok=True)
    shutil.copy2(w["submission_zip"], os.path.join(best_dir, "submission.zip"))
    run_dir = os.path.dirname(os.path.dirname(w["submission_zip"]))
    rc = os.path.join(run_dir, "logs", "run_config.json")
    if os.path.exists(rc):
        shutil.copy2(rc, os.path.join(best_dir, "run_config.json"))
    sel = {"experiment": w.get("experiment"), "job": w.get("job"), "model_id": w.get("model_id"),
           "technique": w.get("technique"), "dev_score": _f(w.get("dev_score")),
           "status": w.get("status"), "run_dir": run_dir,
           "ranking": [{"job": r.get("job"), "model_id": r.get("model_id"),
                        "dev_score": _f(r.get("dev_score")),
                        "trainable_params": r.get("trainable_params")} for r in cands]}
    write_json(os.path.join(best_dir, "SELECTION.json"), sel)
    log(f"   [select] {exp}: BEST = {w.get('job')} (dev score {_f(w.get('dev_score')):.4f})")
    return sel


def plan_uploads(rows: List[Dict], out_path: str, log=print) -> Dict:
    by_exp: Dict[str, List[Dict]] = {}
    rows = _unify(rows)
    for e in COUNTED:
        tech = EXPERIMENTS[e]["technique"]
        c = [r for r in rows if r.get("experiment") == e and is_candidate(r, tech)]
        for r in c:
            r["_hash"] = prediction_hash(r["submission_zip"])
        by_exp[e] = sorted([r for r in c if r["_hash"]],
                           key=lambda r: -_f(r.get("dev_score")))
    missing = [e for e in COUNTED if not by_exp[e]]
    exps = [e for e in COUNTED if by_exp[e]]
    best, best_key, relaxed = None, None, False
    # last resort (Phase 5): if no combination has pairwise different predictions (e.g. weak
    # models all predicting the most frequent statute), still plan one upload per experiment
    for require_all, distinct in ((True, True), (False, True), (True, False), (False, False)):
        for combo in itertools.product(*[by_exp[e] for e in exps]):
            hashes = [r["_hash"] for r in combo]
            if distinct and len(set(hashes)) != len(hashes):
                continue
            models = {r.get("model_id") for r in combo}
            if require_all and len(models) < min(4, len(exps)):
                continue
            key = (len(models), sum(_f(r.get("dev_score")) for r in combo))
            if best is None or key > best_key:
                best, best_key = combo, key
        if best is not None:
            relaxed = not (require_all and distinct)
            break
    plan_rows = []
    if best:
        first = [r for r in best if r.get("experiment") in ("E1", "E2")]
        ordered = first[:1] + [r for r in best if r not in first[:1]]
        for k, r in enumerate(ordered, 1):
            plan_rows.append({"order": k, "experiment": r.get("experiment"),
                              "technique": r.get("technique"), "model_id": r.get("model_id"),
                              "job": r.get("job"), "dev_score": r.get("dev_score"),
                              "path": r.get("submission_zip"), "role": "required"})
        used = {r.get("submission_zip") for r in best}
        used_h = {r["_hash"] for r in best}
        extra = 0
        for m in MODELS:
            pool = [r for e in exps for r in by_exp[e] if r.get("model_id") == m
                    and r.get("submission_zip") not in used and r["_hash"] not in used_h]
            if pool and extra < 4:
                r = max(pool, key=lambda r: _f(r.get("dev_score")))
                extra += 1
                used_h.add(r["_hash"])
                plan_rows.append({"order": len(plan_rows) + 1, "experiment": r.get("experiment"),
                                  "technique": r.get("technique"), "model_id": m,
                                  "job": r.get("job"), "dev_score": r.get("dev_score"),
                                  "path": r.get("submission_zip"), "role": "optional extra"})
    write_csv(out_path, plan_rows, ["order", "experiment", "technique", "model_id", "job",
                                    "dev_score", "role", "path"])
    models = sorted({r["model_id"] for r in plan_rows if r["role"] == "required"})
    rep = {"n_required": sum(1 for r in plan_rows if r["role"] == "required"),
           "missing_experiments": missing, "models_covered": models,
           "all_models_covered": len(models) == 4, "relaxed_model_constraint": relaxed}
    log("   [plan] upload order (Codabench, <= 10/day):")
    for r in plan_rows:
        log(f"   [plan] {r['order']:>2}. {r['experiment']} {r['technique']:<11} {r['model_id']} "
            f"dev score {_f(r['dev_score']):.4f} [{r['role']}]  {r['path']}")
    if missing:
        log(f"   [plan] MISSING experiments (no valid submission yet): {missing}")
    if plan_rows and not rep["all_models_covered"]:
        log(f"   [plan] WARNING: only models {models} appear among the six uploads")
    return rep


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--exp-dir")
    ap.add_argument("--plan", action="store_true")
    ap.add_argument("--out-root", default=OUT_ROOT)
    a = ap.parse_args()
    if a.exp_dir:
        f = [p for p in os.listdir(a.exp_dir) if p.startswith("results_") and p.endswith(".csv")]
        rows = read_csv(os.path.join(a.exp_dir, f[0])) if f else []
        select_exp(a.exp_dir, rows)
    if a.plan:
        from aggregate import aggregate
        rows = aggregate(a.out_root, default_input_roots())
        plan_uploads(rows, os.path.join(a.out_root, "UPLOAD_PLAN.csv"))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase7/aggregate.py
"""
aggregate.py -- one CSV row per (experiment, job) from the files a job leaves behind, the
per-experiment CSVs, ALL_RESULTS.csv + pivots, and the E8 report (ensemble, error analysis).

    python aggregate.py [--out-root /kaggle/working/phase7_out] [--report]

`build_row(run_dir)` reads run_config.json, status.json, plan.json, progress.json, metrics.json,
model_size.json and job_checks.json; it never fails (missing files -> empty cells), so the CSV
can be rewritten at any moment (rule R2: a row exists for every planned job from minute one).
"""
import argparse
import glob
import json
import os
import shutil
from typing import Dict, List, Optional

import numpy as np

from cjpe_common import (COUNTED, CSV_COLUMNS, EPOCH_COLUMNS, EXPERIMENTS, MODELS, OUT_ROOT,
                         default_input_roots, iter_out_dirs, read_csv, read_json, write_csv,
                         write_json, log_to)

STATE_TO_STATUS = {"done": "DONE", "degenerate": "DEGENERATE", "failed": "FAILED",
                   "skipped_time": "SKIPPED_TIME", "preflight": "RUNNING",
                   "training": "RUNNING", "evaluating": "RUNNING", "predicting": "RUNNING",
                   "pending": "PENDING", "queued": "PENDING", "smoke_ok": "SMOKE_OK"}


def build_row(run_dir: str, teacher_total: Optional[float] = None) -> Dict:
    rc = read_json(os.path.join(run_dir, "logs", "run_config.json"), {}) or {}
    st = read_json(os.path.join(run_dir, "status.json"), {}) or {}
    plan = read_json(os.path.join(run_dir, "logs", "plan.json"), {}) or {}
    prog = read_json(os.path.join(run_dir, "logs", "progress.json"), {}) or {}
    met = read_json(os.path.join(run_dir, "metrics.json"), {}) or {}
    size = read_json(os.path.join(run_dir, "logs", "model_size.json"), {}) or {}
    chk = read_json(os.path.join(run_dir, "logs", "job_checks.json"), {}) or {}
    status = STATE_TO_STATUS.get(st.get("state", "pending"), "PENDING")
    if status == "FAILED" and st.get("exit_code") == 2:
        status = "SKIPPED_TIME"
    if status == "DONE" and not os.path.exists(os.path.join(run_dir, "DONE.json")):
        status = "FAILED" if met else status
    zp = os.path.join(run_dir, "submission", "submission.zip")
    tech = rc.get("technique", "")
    if rc.get("is_teacher"):
        tech = f"{tech} (teacher)"
    tot = size.get("total_params") or met.get("total_params")
    row = {
        "experiment": rc.get("experiment"), "technique": tech, "model_id": rc.get("model"),
        "arch": rc.get("arch"), "attn_mode": rc.get("attn_mode"), "pooling": rc.get("pooling"),
        "doc_repr": rc.get("doc_repr"), "chunk_tokens": rc.get("chunk_tokens"),
        "k_train": met.get("k_train", rc.get("k_train")),
        "k_infer": met.get("k_infer", plan.get("k_infer")), "hard_negs": rc.get("hard_negs"),
        "status": status,
        "fail_reason": "" if status in ("DONE", "PENDING", "RUNNING") else st.get("reason", ""),
        "selfcheck": (met.get("selfcheck", chk.get("selfcheck", "")) or "") + (
            " DEGENERATE" if met.get("degenerate") else ""),
        "train_queries": met.get("train_queries", rc.get("n_train")),
        "train_pairs": plan.get("train_pairs"),
        "train_fraction": met.get("train_fraction", plan.get("train_fraction")),
        "dev_queries": rc.get("n_dev"), "test_queries": rc.get("n_test"),
        "epochs_planned": met.get("epochs_planned", plan.get("epochs")),
        "epochs_run": met.get("epochs_run", prog.get("epochs_run")),
        "best_epoch": met.get("best_epoch", prog.get("best_epoch")), "lr": rc.get("lr"),
        "batch_pairs": rc.get("batch_pairs"),
        "dev_proxy_map": met.get("dev_proxy_map", prog.get("best_proxy_map")),
        "dev_proxy_map_init": met.get("dev_proxy_map_init", plan.get("init_proxy")),
        "trainable_params": size.get("trainable_params"), "total_params": tot,
        "trainable_pct": size.get("trainable_pct"),
        "params_ratio_vs_teacher": (tot / teacher_total) if (tot and teacher_total and rc.get(
            "technique") in ("kd", "student_hard")) else None,
        "projected_minutes": plan.get("projected_minutes"), "gpu_slot": rc.get("gpu_slot"),
        "attempts": rc.get("attempt"), "leak_guard": rc.get("leak_guard", "passed"),
        "seed": rc.get("seed"), "job": rc.get("job", os.path.basename(run_dir)),
        "submission_zip": zp if os.path.exists(zp) else "",
        "started_at": rc.get("started_at"), "finished_at": st.get("time") if status in (
            "DONE", "FAILED", "DEGENERATE", "SKIPPED_TIME") else "",
    }
    for k in ("score_metric", "dev_score", "dev_micro_f1", "dev_micro_p", "dev_micro_r",
              "dev_dense_f1", "dev_dense_mean_f1", "dev_dense_max_f1", "dev_bm25_f1",
              "dense_weight", "doc_agg", "dev_dense_recall_at_100", "train_minutes",
              "tokens_per_s", "infer_tokens_per_s", "ms_per_doc_infer", "peak_mem_gb"):
        row[k] = met.get(k)
    return row


def epoch_rows(run_dir: str) -> List[Dict]:
    rc = read_json(os.path.join(run_dir, "logs", "run_config.json"), {}) or {}
    p = os.path.join(run_dir, "logs", "train_log.jsonl")
    out = []
    if os.path.exists(p):
        with open(p) as f:
            for line in f:
                try:
                    r = json.loads(line)
                except Exception:
                    continue
                if r.get("type") == "eval":
                    out.append({"experiment": rc.get("experiment"), "model_id": rc.get("model"),
                                "job": rc.get("job"), **r})
    return out


def exp_rows(exp_dir: str, planned: Optional[List[Dict]] = None) -> List[Dict]:
    """Rows for every job folder of an experiment (+ PENDING rows for planned jobs)."""
    jobs = sorted(d for d in os.listdir(exp_dir) if os.path.isdir(os.path.join(exp_dir, d))
                  and d != "BEST") if os.path.isdir(exp_dir) else []
    teacher_total = None
    for j in jobs:
        rc = read_json(os.path.join(exp_dir, j, "logs", "run_config.json"), {}) or {}
        if rc.get("is_teacher"):
            teacher_total = (read_json(os.path.join(exp_dir, j, "logs", "model_size.json"), {})
                             or {}).get("total_params")
    rows = {}
    for p in planned or []:
        rows[p["job"]] = dict(p)
    for j in jobs:
        if not os.path.exists(os.path.join(exp_dir, j, "logs", "run_config.json")):
            st = read_json(os.path.join(exp_dir, j, "status.json"), {}) or {}
            if j in rows and st.get("state") in STATE_TO_STATUS:
                rows[j]["status"] = STATE_TO_STATUS[st["state"]]
                if rows[j]["status"] not in ("PENDING", "RUNNING"):
                    rows[j]["fail_reason"] = st.get("reason", "")
            continue
        r = build_row(os.path.join(exp_dir, j), teacher_total)
        if j in rows:
            r = {**rows[j], **{k: v for k, v in r.items() if v not in (None, "")}}
            if r.get("status") == "PENDING" and rows[j].get("status") not in (None, "PENDING"):
                r["status"] = rows[j]["status"]
                r["fail_reason"] = rows[j].get("fail_reason", "")
        rows[j] = r
    return list(rows.values())


def write_exp_csvs(exp_dir: str, planned: Optional[List[Dict]] = None) -> List[Dict]:
    folder = os.path.basename(exp_dir.rstrip("/"))
    rows = exp_rows(exp_dir, planned)
    write_csv(os.path.join(exp_dir, f"results_{folder}.csv"), rows, CSV_COLUMNS)
    ep = []
    for j in sorted(os.listdir(exp_dir)) if os.path.isdir(exp_dir) else []:
        if os.path.isdir(os.path.join(exp_dir, j)) and j != "BEST":
            ep += epoch_rows(os.path.join(exp_dir, j))
    write_csv(os.path.join(exp_dir, f"epoch_log_{folder}.csv"), ep, EPOCH_COLUMNS)
    return rows


def _status_rank(s: str) -> int:
    return {"DONE": 5, "DEGENERATE": 4, "RUNNING": 3, "FAILED": 2, "SKIPPED_TIME": 2,
            "PENDING": 1}.get(s, 0)


def aggregate(out_root: str = OUT_ROOT, input_roots=None, log=print) -> List[Dict]:
    """Merge every results_*.csv (working dir first, then attached inputs) -> ALL_RESULTS.csv.
    submission_zip paths are re-resolved where the CSV was found."""
    input_roots = default_input_roots() if input_roots is None else input_roots
    best: Dict = {}
    for od in iter_out_dirs(out_root, input_roots):
        for csvp in sorted(glob.glob(os.path.join(od, "*", "results_*.csv"))):
            exp_dir = os.path.dirname(csvp)
            for r in read_csv(csvp):
                job = r.get("job") or ""
                z = os.path.join(exp_dir, job, "submission", "submission.zip")
                r["submission_zip"] = z if (job and os.path.exists(z)) else ""
                r["_run_dir"] = os.path.join(exp_dir, job)
                key = (r.get("experiment"), job)
                old = best.get(key)
                if old is None or _status_rank(r.get("status", "")) > _status_rank(
                        old.get("status", "")):
                    best[key] = r
    rows = sorted(best.values(), key=lambda r: (str(r.get("experiment")), str(r.get("job"))))
    if out_root:
        os.makedirs(out_root, exist_ok=True)
        write_csv(os.path.join(out_root, "ALL_RESULTS.csv"), rows, CSV_COLUMNS)
        write_pivots(out_root, rows)
    return rows


def write_pivots(out_root: str, rows: List[Dict]) -> None:
    for metric, name in (("dev_micro_f1", "ALL_RESULTS_pivot.csv"),
                         ("dev_dense_f1", "ALL_RESULTS_pivot_dense_only.csv"),
                         ("dev_micro_r", "ALL_RESULTS_pivot_recall.csv"),
                         ("trainable_pct", "ALL_RESULTS_pivot_trainable_pct.csv"),
                         ("train_minutes", "ALL_RESULTS_pivot_train_minutes.csv")):
        table: Dict[str, Dict] = {}
        for r in rows:
            if r.get("status") not in ("DONE", "DEGENERATE"):
                continue
            k = f"{r.get('experiment')}:{r.get('technique')}"
            if r.get("experiment") == "E7":
                k = f"E7:{r.get('job', '').rsplit('_M', 1)[0]}"
            table.setdefault(k, {"experiment_technique": k})
            table[k][r.get("model_id")] = r.get(metric)
        write_csv(os.path.join(out_root, name), list(table.values()),
                  ["experiment_technique", "M1", "M2", "M3", "M4"])


# =============================================================================================
# E8 report: score-fusion ensemble of the BEST runs + retrieval error analysis + figures
# =============================================================================================
def _score(r: Dict, metric: str) -> float:
    try:
        return float(r.get(metric))
    except (TypeError, ValueError):
        return -1.0


def common_metric(rows: List[Dict]) -> str:
    """Phase 7 has one dev metric for every run (no network-dependent scorer)."""
    return "dev_micro_f1"


def _best_per_exp(rows: List[Dict]) -> List[Dict]:
    out = []
    for e in COUNTED:
        c = [r for r in rows if r.get("experiment") == e and r.get("status") == "DONE"
             and not r.get("technique", "").endswith("(teacher)")]
        if c:
            out.append(max(c, key=lambda r: _score(r, "dev_micro_f1")))
    return out


def _jsonl(path: str) -> List[Dict]:
    out = []
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    out.append(json.loads(line))
                except Exception:
                    pass
    return out


def _eval_data(out_root: str):
    for od in iter_out_dirs(out_root, default_input_roots()):
        p = os.path.join(od, "eval_data.npz")
        if os.path.exists(p):
            return np.load(p, allow_pickle=False)
    return None


def ensemble(rows: List[Dict], out_root: str, log=print) -> Optional[Dict]:
    """Mean of the members' per-query z-scored dense scores, fused with BM25 (weight tuned on
    dev, as every job does); also reciprocal-rank fusion. Written as a ready submission."""
    import zipfile
    from pcr_ml import K_EVAL, eval_scores, full_rank, fuse, rrf, topk, tune_fusion, zrow
    from validate_submission import validate
    ed = _eval_data(out_root)
    mem = []
    for m in _best_per_exp(rows):
        p = os.path.join(m["_run_dir"], "dev", "scores.npz")
        if os.path.exists(p):
            z = np.load(p)
            mem.append((m, z["dev"].astype(np.float32), z["test"].astype(np.float32)))
    if ed is None or len(mem) < 2:
        log("   [ensemble] eval_data.npz or fewer than 2 members with saved scores -> skipped")
        return None
    gold = [json.loads(g) for g in ed["dev_gold"]]
    bd, bt = ed["bm25_dev"].astype(np.float32), ed["bm25_test"].astype(np.float32)
    shape_ok = [x for x in mem if x[1].shape == bd.shape and x[2].shape == bt.shape]
    if len(shape_ok) < 2:
        log("   [ensemble] members were scored on different data -> skipped")
        return None
    dev_mean = np.mean([zrow(x[1]) for x in shape_ok], 0)
    test_mean = np.mean([zrow(x[2]) for x in shape_ok], 0)
    grid = [round(x, 2) for x in np.arange(0.0, 1.0001, 0.1)]
    w, best, curve = tune_fusion(dev_mean, bd, gold, [g for g in grid if g >= 0.2])
    sys_dev = [full_rank(fuse(x[1], bd, float(x[0].get("dense_weight") or 1))) for x in shape_ok]
    rrf_f1 = eval_scores(rrf(sys_dev, bd.shape[1]), gold)["micro_f1"]
    singles = {f"{m.get('experiment')}:{m.get('model_id')}": _score(m, "dev_micro_f1")
               for m, _, _ in shape_ok}
    best_single = max(singles.values())
    S = fuse(test_mean, bt, w)
    tq, tp = [str(x) for x in ed["test_q"]], [str(x) for x in ed["test_pool"]]
    rt = topk(S, K_EVAL)
    pred = {q: [tp[j] for j in rt[i]] for i, q in enumerate(tq)}
    d = os.path.join(out_root, "E8_report", "ensemble")
    os.makedirs(d, exist_ok=True)
    with open(os.path.join(d, "prediction.json"), "w", encoding="utf-8") as f:
        json.dump(pred, f)
    with zipfile.ZipFile(os.path.join(d, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(d, "prediction.json"), "prediction.json")
    v = validate(os.path.join(d, "submission.zip"), tq, tp)
    rep = {"members": list(singles), "dense_weight": w, "ensemble_dev_micro_f1": best["micro_f1"],
           "ensemble_dev_micro_p": best["micro_p"], "ensemble_dev_micro_r": best["micro_r"],
           "rrf_dev_micro_f1": rrf_f1, "best_single_dev_micro_f1": best_single,
           "validation_ok": v["ok"], "use_it": bool(v["ok"] and best["micro_f1"] >
                                                     best_single + 0.002),
           "note": "dev chose the members and the fusion weight -> optimistic"}
    write_json(os.path.join(d, "ensemble_report.json"), rep)
    log(f"   [ensemble] {len(shape_ok)} members: dev Micro-F1@7 {best['micro_f1']:.4f} (RRF "
        f"{rrf_f1:.4f}) vs best single {best_single:.4f} -> use_it={rep['use_it']}")
    return rep


def error_analysis(rows: List[Dict], out_root: str, dev_texts=None, log=print) -> None:
    """Retrieval tables: per-run dense / BM25 / fused scores, hits by query-length bucket and
    by gold-set size, the queries every run misses, and plots."""
    rep_dir = os.path.join(out_root, "E8_report")
    os.makedirs(rep_dir, exist_ok=True)
    done = [r for r in rows if r.get("status") in ("DONE", "DEGENERATE") and r.get("_run_dir")]
    buckets, per_q = [], {}
    for r in done:
        base = {"experiment": r.get("experiment"), "technique": r.get("technique"),
                "model_id": r.get("model_id"), "job": r.get("job")}
        preds = _jsonl(os.path.join(r["_run_dir"], "dev", "dev_predictions.jsonl"))
        if not preds:
            continue
        chars = np.array([p.get("n_chars") or 0 for p in preds], float)
        qs = np.quantile(chars, [0.33, 0.66]) if len(chars) > 3 else [0, 0]
        for name, sel in (("short", chars <= qs[0]), ("medium", (chars > qs[0]) & (chars <= qs[1])),
                          ("long", chars > qs[1])):
            P = [p for p, s_ in zip(preds, sel) if s_]
            tp = sum(p["hits"] for p in P)
            ng = sum(p["n_gold"] for p in P)
            prec, rec = tp / max(1, 7 * len(P)), tp / max(1, ng)
            buckets.append({**base, "query_length": name, "queries": len(P),
                            "micro_p": prec, "micro_r": rec,
                            "micro_f1": 2 * prec * rec / max(1e-12, prec + rec)})
        for p in preds:
            per_q.setdefault(p["id"], []).append(p["hits"])
    if buckets:
        write_csv(os.path.join(rep_dir, "query_length_buckets.csv"), buckets, list(buckets[0]))
    miss = [{"id": q, "runs": len(h), "runs_with_zero_hits": sum(1 for x in h if x == 0),
             "mean_hits": float(np.mean(h))} for q, h in per_q.items()]
    miss.sort(key=lambda x: (-x["runs_with_zero_hits"], x["mean_hits"]))
    if miss:
        write_csv(os.path.join(rep_dir, "hardest_dev_queries.csv"), miss[:50], list(miss[0]))
    cols = ["experiment", "technique", "model_id", "attn_mode", "pooling", "doc_repr",
            "hard_negs", "dev_micro_f1", "dev_micro_p", "dev_micro_r", "dev_dense_f1",
            "dev_dense_mean_f1", "dev_dense_max_f1", "dev_bm25_f1", "dense_weight", "doc_agg",
            "dev_dense_recall_at_100", "dev_proxy_map_init", "dev_proxy_map", "trainable_pct",
            "total_params", "params_ratio_vs_teacher", "train_minutes", "ms_per_doc_infer",
            "peak_mem_gb"]
    write_csv(os.path.join(rep_dir, "retrieval_table.csv"), done, cols)
    log(f"   [report] {len(done)} finished runs: retrieval_table, query_length_buckets "
        f"({len(buckets)}), hardest_dev_queries ({len(miss)})")
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        ok = [r for r in done if _score(r, "dev_micro_f1") >= 0]
        if ok:
            x = np.arange(len(ok))
            fig, ax = plt.subplots(figsize=(max(6, 0.5 * len(ok)), 4))
            ax.bar(x - 0.2, [_score(r, "dev_micro_f1") for r in ok], 0.4, label="fused (submitted)")
            ax.bar(x + 0.2, [_score(r, "dev_dense_f1") for r in ok], 0.4, label="dense only")
            bm = _score(ok[0], "dev_bm25_f1")
            if bm >= 0:
                ax.axhline(bm, color="k", ls="--", lw=1, label="BM25 only")
            ax.set_xticks(x)
            ax.set_xticklabels([f"{r['experiment']}\n{r['model_id']}" for r in ok], fontsize=7)
            ax.set_ylabel("dev Micro-F1@7")
            ax.legend(fontsize=7)
            ax.set_title("Phase 7 prior case retrieval: dev Micro-F1@7")
            fig.tight_layout()
            fig.savefig(os.path.join(rep_dir, "dev_score.png"), dpi=130)
            plt.close(fig)
            fig, ax = plt.subplots(figsize=(6, 4))
            for r in ok:
                try:
                    ax.scatter(float(r["trainable_pct"]), _score(r, "dev_dense_f1"), s=25)
                    ax.annotate(f"{r['experiment']}/{r['model_id']}",
                                (float(r["trainable_pct"]), _score(r, "dev_dense_f1")), fontsize=6)
                except (TypeError, ValueError, KeyError):
                    pass
            ax.set_xscale("log")
            ax.set_xlabel("trainable parameters (%)")
            ax.set_ylabel("dev Micro-F1@7 (dense only)")
            fig.tight_layout()
            fig.savefig(os.path.join(rep_dir, "quality_vs_trainable.png"), dpi=130)
            plt.close(fig)
    except Exception as e:
        log(f"   [report] plots skipped ({type(e).__name__}: {e})")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out-root", default=OUT_ROOT)
    ap.add_argument("--report", action="store_true")
    a = ap.parse_args()
    rows = aggregate(a.out_root)
    print(f"ALL_RESULTS.csv: {len(rows)} rows")
    if a.report:
        ensemble(rows, a.out_root)
        error_analysis(rows, a.out_root)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase7/run_experiment.py
"""
run_experiment.py -- the Phase-7 orchestrator (one notebook = one call).

    python run_experiment.py --exp E2 --models M4,M2,M3,M1 --gpu-mode jobs --resume auto \
                             --time-budget-h 2.0

Order of work (sections 3, 4, 8, 9 of the brief):
  1. Phase-1 discovery (by the `arch` field), the job list, results_<exp>.csv with one PENDING
     row per job + INDEX.md                                   <- outputs exist from minute one
  2. data discovery (attached dataset, else the HF download with the Kaggle secret `HF`),
     schema-tolerant loading of the 3 query files + 3 candidate pools, data audit, PROVENANCE
  3. chunk token cache + BM25 channel tuned on Dev (reused when an attached cache has the same
     key); eval_data.npz (dev gold, BM25 dev/test scores, ids) for E8
  4. job queue: one slot per visible GPU (no DDP); each job is a `pcr_train.py` subprocess
     with CUDA_VISIBLE_DEVICES=<slot>; per-job time budget from the notebook budget;
     retry ladder on failure; the CSV and INDEX.md are refreshed every minute
  5. BEST/ per experiment, ALL_RESULTS.csv (+ pivots), UPLOAD_PLAN.csv, final INDEX.md
Exit code: 0 if at least one job finished (DONE), else 1 (2 = data/weights missing).
The notebook prints the code and never raises (rule R1).
"""
import argparse
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from typing import Dict, List, Optional

from cjpe_common import (CACHE_DIR, CODE_DIR, COUNTED, CSV_COLUMNS, DEFAULT_LR,
                         DEFAULT_MODEL_ORDER, EXPERIMENTS, JOB_EXIT, MODELS, OUT_ROOT, SPEC,
                         default_input_roots, dir_size_gb, find_phase1_models, free_gb, log_to,
                         module_md5s, now_str, read_json, student_layers, write_csv, write_json)

LOG = None


def log(msg: str) -> None:
    log_to(LOG, msg)


def build_parser():
    ap = argparse.ArgumentParser()
    ap.add_argument("--exp", required=True, choices=sorted(EXPERIMENTS))
    ap.add_argument("--models", default=",".join(DEFAULT_MODEL_ORDER))
    ap.add_argument("--gpu-mode", default="jobs", choices=["jobs"])
    ap.add_argument("--resume", default="auto", choices=["auto", "off"])
    ap.add_argument("--time-budget-h", type=float, default=2.0)
    ap.add_argument("--out-root", default=OUT_ROOT)
    ap.add_argument("--cache-dir", default=CACHE_DIR)
    ap.add_argument("--doc-repr", default="hier", choices=["hier", "head"],
                    help="hier: mean of up to --k-infer chunks spread over the case; head: chunk 1")
    ap.add_argument("--chunk-tokens", type=int, default=256)
    ap.add_argument("--max-chunks", type=int, default=16, help="chunks cached per case")
    ap.add_argument("--k-train", type=int, default=2, help="chunks per case per training step")
    ap.add_argument("--k-infer", type=int, default=8, help="chunks per case at inference")
    ap.add_argument("--pooling", default="mean", choices=["mean", "last"])
    ap.add_argument("--attn-mode", default="causal", choices=["causal", "bidir"])
    ap.add_argument("--batch-pairs", type=int, default=16)
    ap.add_argument("--hard-negs", type=int, default=3)
    ap.add_argument("--micro-tokens", type=int, default=8192)
    ap.add_argument("--min-dense-weight", type=float, default=0.2,
                    help="fusion weight of the trained encoder is tuned on dev in [this, 1]")
    ap.add_argument("--n-virtual", type=int, default=20, help="prefix / prompt length")
    ap.add_argument("--epochs-max", type=int, default=None)
    ap.add_argument("--min-epochs", type=int, default=1)
    ap.add_argument("--evals-per-epoch", type=int, default=2)
    ap.add_argument("--patience", type=int, default=2)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--teacher-model", default="M4")
    ap.add_argument("--teacher-frac", type=float, default=0.4)
    ap.add_argument("--kd-alpha", type=float, default=0.5)
    ap.add_argument("--kd-T", type=float, default=2.0)
    ap.add_argument("--ablation-model", default="M4")
    ap.add_argument("--ablation-technique", default="lorma")
    ap.add_argument("--e7-jobs", default="ia3,prompt,zeroshot,pool_last,attn_bidir,repr_head,"
                                         "hn_0")
    ap.add_argument("--e7-fixed-frac", type=float, default=0.5)
    ap.add_argument("--e7-fixed-epochs", type=int, default=1)
    ap.add_argument("--skip-job-checks", action="store_true")
    ap.add_argument("--max-retries", type=int, default=2)
    ap.add_argument("--n-proc", type=int, default=max(1, min(4, os.cpu_count() or 1)))
    ap.add_argument("--reserve-min", type=float, default=6.0)
    ap.add_argument("--lr-mult", type=float, default=1.0)
    return ap


# =============================================================================================
# jobs
# =============================================================================================
def make_jobs(a, p1: Dict[str, str]) -> List[Dict]:
    models = [m.strip() for m in a.models.split(",") if m.strip()]
    e = a.exp
    jobs = []
    if e in COUNTED and e != "E6":
        tech = EXPERIMENTS[e]["technique"]
        for m in models:
            jobs.append({"job": MODELS[m]["folder"], "model": m, "technique": tech})
    elif e == "E6":
        t = a.teacher_model
        jobs.append({"job": f"teacher_{MODELS[t]['folder']}", "model": t,
                     "technique": "full_ft", "is_teacher": True})
        for m in models:
            jobs.append({"job": f"student_{MODELS[m]['folder']}", "model": m, "technique": "kd",
                         "student": True, "depends": [f"teacher_{MODELS[t]['folder']}"]})
    elif e == "E7":
        am = a.ablation_model
        fixed = {"epochs": a.e7_fixed_epochs, "frac": a.e7_fixed_frac}
        for spec in [s.strip() for s in a.e7_jobs.split(",") if s.strip()]:
            f = MODELS[am]["folder"]
            j = {"job": f"{spec}_{f}", "model": am, "technique": a.ablation_technique,
                 "fixed_plan": fixed}
            if spec in ("ia3", "prompt", "prefix", "qlora", "lora", "lorma", "full_ft",
                        "partial_ft"):
                j = {"job": f"{spec}_{f}", "model": am, "technique": spec}
            elif spec == "zeroshot":
                j["fixed_plan"] = {"epochs": 0}
            elif spec.startswith("pool_"):
                j["pooling"] = spec[5:]
            elif spec.startswith("attn_"):
                j["attn_mode"] = spec[5:]
            elif spec.startswith("repr_"):
                j["doc_repr"] = spec[5:]
            elif spec.startswith("hn_"):
                j["hard_negs"] = int(spec[3:])
            else:
                continue
            jobs.append(j)
    for j in jobs:
        j.setdefault("depends", [])
        j["state"] = "pending"
        j["attempt"] = 0
        j["overrides"] = {}
    return jobs


def planned_row(a, j: Dict) -> Dict:
    return {"experiment": a.exp, "technique": j["technique"] + (" (teacher)" if j.get(
        "is_teacher") else ""), "model_id": j["model"], "arch": MODELS[j["model"]]["arch"],
            "job": j["job"], "status": "PENDING", "doc_repr": j.get("doc_repr", a.doc_repr),
            "pooling": j.get("pooling", a.pooling), "attn_mode": j.get("attn_mode", a.attn_mode),
            "seed": a.seed}


# =============================================================================================
# INDEX.md
# =============================================================================================
def write_index(a, out_root, exp_dir, jobs, extra: Optional[List[str]] = None) -> None:
    from aggregate import write_exp_csvs
    rows = write_exp_csvs(exp_dir, [planned_row(a, j) for j in jobs]) if jobs else []
    lines = [f"# Phase 7 (prior case retrieval) -- {a.exp}: {EXPERIMENTS[a.exp]['title']}", "",
             f"Updated {now_str()}. Deadline {SPEC['deadline']}.", "",
             "| job | model | technique | status | Micro-F1@7 | dense only | BM25 only | dense w "
             "| epochs | train frac | minutes | reason |",
             "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for r in rows:
        f = lambda k, n=4: (f"{float(r[k]):.{n}f}" if r.get(k) not in (None, "") else "")  # noqa
        lines.append(f"| {r.get('job')} | {r.get('model_id')} | {r.get('technique')} | "
                     f"{r.get('status')} | {f('dev_micro_f1')} | {f('dev_dense_f1')} | "
                     f"{f('dev_bm25_f1')} | {f('dense_weight', 1)} | {f('epochs_run', 2)} | "
                     f"{f('train_fraction', 2)} | {f('train_minutes', 1)} | "
                     f"{str(r.get('fail_reason') or '').replace('|', '/')[:120]} |")
    todo = [r for r in rows if r.get("status") in ("FAILED", "SKIPPED_TIME", "PENDING")]
    if todo:
        ms = sorted({r["model_id"] for r in todo if r.get("model_id")})
        lines += ["", "## Next command for the unfinished models (here or on another account)",
                  "", "```", f"!cd /kaggle/working/phase7 && python run_experiment.py --exp "
                             f"{a.exp} --models {','.join(ms)} --resume auto --time-budget-h "
                             f"{a.time_budget_h}", "```",
                  "Attach this notebook's output as an input so finished models are skipped."]
    if extra:
        lines += [""] + extra
    p = os.path.join(out_root, "INDEX.md")
    os.makedirs(out_root, exist_ok=True)
    with open(p + ".tmp", "w") as f:
        f.write("\n".join(lines) + "\n")
    os.replace(p + ".tmp", p)


# =============================================================================================
# data
# =============================================================================================
def prepare_data(a, out_root: str, bpe_dir: str) -> Dict:
    import numpy as np
    from ner_common import file_md5
    from pcr_data import build_cache, cache_key, chunk_stats, find_cache, load_all, load_cache
    roots = default_input_roots()
    data, schemas, info = load_all(roots, log)
    write_json(os.path.join(out_root, "PROVENANCE.json"),
               {"source": info["source"], "files": {k: {"path": v["path"], "md5": v["md5"],
                                                        "n": v["n"]} for k, v in schemas.items()},
                "rule": "each split's queries are ranked against that split's pool only; dev "
                        "labels are used for model selection, never for training"})
    key = cache_key({s: schemas[s]["md5"] for s in schemas},
                    file_md5(os.path.join(bpe_dir, "bpe_model.json")), a.chunk_tokens,
                    a.max_chunks)
    path = find_cache(key, [a.cache_dir] + roots)
    if path:
        log(f"   [cache] reusing {path}")
    else:
        t0 = time.time()
        log(f"   [cache] tokenising every case into <= {a.max_chunks} chunks of "
            f"{a.chunk_tokens} tokens ({a.n_proc} processes) + BM25 ...")
        path = build_cache(data, bpe_dir, a.cache_dir, key, a.chunk_tokens, a.max_chunks,
                           a.n_proc, log)
        log(f"   [cache] built in {time.time() - t0:.0f}s")
    cache = load_cache(path)
    sp, bm = cache["splits"], cache["bm25"]
    write_json(os.path.join(out_root, "test_ids.json"), [x["id"] for x in sp["test"]["q"]])
    write_json(os.path.join(out_root, "test_pool_ids.json"), [x["id"] for x in sp["test"]["pool"]])
    np.savez_compressed(os.path.join(out_root, "eval_data.npz"),
                        bm25_dev=bm["dev"].astype(np.float16), bm25_test=bm["test"].astype(np.float16),
                        dev_q=np.array([x["id"] for x in sp["dev"]["q"]]),
                        dev_pool=np.array([x["id"] for x in sp["dev"]["pool"]]),
                        test_q=np.array([x["id"] for x in sp["test"]["q"]]),
                        test_pool=np.array([x["id"] for x in sp["test"]["pool"]]),
                        dev_gold=np.array([json.dumps(x.get("rel") or []) for x in sp["dev"]["q"]]),
                        dev_chars=np.array([x.get("n_chars") or 0 for x in sp["dev"]["q"]]))
    audit = {**info["audit"], **chunk_stats(cache), "bm25": {k: v for k, v in bm.items()
                                                             if k in ("params", "dev_micro_f1",
                                                                      "dev_micro_p", "dev_micro_r",
                                                                      "dev_curve")}}
    write_json(os.path.join(out_root, "data_audit.json"), audit)
    log(f"   [audit] chunks per case (mean, cap {a.max_chunks}): train pool "
        f"{audit['train_pool_chunks_mean']:.1f}, dev pool {audit['dev_pool_chunks_mean']:.1f}, "
        f"test pool {audit['test_pool_chunks_mean']:.1f}; {100 * audit['test_pool_chunks_full_frac']:.0f}% "
        f"of test candidates longer than {a.max_chunks} chunks (sampled evenly)")
    log(f"   [bm25] baseline (lexical channel, same for every job): dev Micro-F1@7 "
        f"{bm['dev_micro_f1']:.4f} with {bm['params']}")
    return {"cache_path": path, "cache_key": key,
            "n": {s: len(sp[s]["q"]) for s in sp}, "pool": {s: len(sp[s]["pool"]) for s in sp},
            "data_md5": {s: schemas[s]["md5"] for s in schemas}}


# =============================================================================================
# scheduler
# =============================================================================================
def n_gpus() -> int:
    try:
        import torch
        return torch.cuda.device_count()
    except Exception:
        return 0


def job_config(a, j: Dict, p1: Dict[str, str], data: Dict, exp_dir: str, slot: int,
               budget_s: float, attn: str, bpe_dir: str, jobs_by_name: Dict) -> Dict:
    tech = j["technique"]
    lr = DEFAULT_LR.get(tech, 5e-4) * a.lr_mult * j["overrides"].get("lr_mult", 1.0)
    cfg = {
        "experiment": a.exp, "exp_folder": EXPERIMENTS[a.exp]["folder"], "job": j["job"],
        "model": j["model"], "model_folder": MODELS[j["model"]]["folder"],
        "arch": MODELS[j["model"]]["arch"], "phase1_dir": p1[j["model"]], "bpe_dir": bpe_dir,
        "technique": tech, "doc_repr": j.get("doc_repr", a.doc_repr),
        "pooling": j.get("pooling", a.pooling), "attn_mode": j.get("attn_mode", a.attn_mode),
        "chunk_tokens": a.chunk_tokens, "k_train": a.k_train, "k_infer": a.k_infer,
        "batch_pairs": a.batch_pairs, "hard_negs": int(j.get("hard_negs", a.hard_negs)),
        "min_dense_weight": a.min_dense_weight, "lr": lr, "head_lr": max(lr, 1e-4),
        "weight_decay": 0.01, "grad_clip": 1.0, "warmup_frac": 0.06, "epochs_max": a.epochs_max,
        "min_epochs": a.min_epochs, "evals_per_epoch": a.evals_per_epoch,
        "patience": a.patience, "effective_batch": a.batch_pairs,
        "micro_tokens": int(j["overrides"].get("micro_tokens", a.micro_tokens)),
        "n_virtual": a.n_virtual, "seed": a.seed,
        "budget_s": budget_s, "cache_path": data["cache_path"], "cache_key": data["cache_key"],
        "data_md5": data["data_md5"], "n_train": data["n"]["train"], "n_dev": data["n"]["dev"],
        "n_test": data["n"]["test"], "skip_job_checks": bool(
            a.skip_job_checks or j["overrides"].get("skip_job_checks")),
        "is_teacher": bool(j.get("is_teacher")), "fixed_plan": j.get("fixed_plan"),
        "gpu_slot": slot, "attempt": j["attempt"], "started_at": now_str(),
        "leak_guard": "passed", "ckpt_every_min": 30,
        "min_train_fraction": 0.1, "kd_alpha": a.kd_alpha, "kd_T": a.kd_T,
        "lorma_r": 16, "lorma_alpha": 32, "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05,
        "partial_top_k": 6, "layer_decay": 0.85, "module_md5": module_md5s(),
    }
    if j.get("student"):
        cfg["layer_ids"] = student_layers(j["model"], p1[j["model"]])
        tdir = os.path.join(exp_dir, j["depends"][0])
        cfg["teacher_dir"] = tdir
        if not (os.path.exists(os.path.join(tdir, "DONE.json")) and
                os.path.exists(os.path.join(tdir, "checkpoints", "best.pt"))):
            # teacher failed: train the pruned student on the gold labels only (CSV says so)
            cfg["technique"] = "student_hard"
            cfg["teacher_failed"] = True
    return cfg


def copy_from_inputs(a, exp_dir: str, job: str) -> Optional[str]:
    """--resume auto: a finished (or resumable) copy of this job in an attached input. The E6
    teacher (full FT of M4) is also taken from a finished E2_full_ft output when attached."""
    folder = EXPERIMENTS[a.exp]["folder"]
    from cjpe_common import iter_out_dirs
    cands = [(folder, job)]
    if a.exp == "E6" and job.startswith("teacher_"):
        cands.append((EXPERIMENTS["E2"]["folder"], job[len("teacher_"):]))
    for od in iter_out_dirs(None, default_input_roots()):
      for fo, jb in cands:
        src = os.path.join(od, fo, jb)
        if fo != folder and not os.path.exists(os.path.join(src, "DONE.json")):
            continue
        if os.path.exists(os.path.join(src, "DONE.json")) or os.path.exists(
                os.path.join(src, "checkpoints", "last.pt")):
            dst = os.path.join(exp_dir, job)
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            if fo != folder:                                  # E2 run reused as the E6 teacher
                rp = os.path.join(dst, "logs", "run_config.json")
                rc = read_json(rp, {}) or {}
                rc.update({"experiment": a.exp, "exp_folder": folder, "job": job,
                           "is_teacher": True, "reused_from": src})
                write_json(rp, rc)
            return src
    return None


def archive_attempt(run_dir: str, attempt: int) -> None:
    lg = os.path.join(run_dir, "logs")
    for f in ("console.log", "job_checks.json", "plan.json", "train_log.jsonl",
              "progress.json"):
        p = os.path.join(lg, f)
        if os.path.exists(p):
            os.replace(p, os.path.join(lg, f"attempt{attempt}_{f}"))
    for f in ("checkpoints/last.pt", "checkpoints/best.pt", "metrics.json", "DONE.json"):
        p = os.path.join(run_dir, f)
        if os.path.exists(p):
            os.remove(p)


def next_attempt(a, j: Dict, code: int, reducible: bool) -> Optional[str]:
    """The retry ladder (rule R5). Returns a description, or None when giving up."""
    if code == 2:
        return None
    ladder = []
    if code in (4, 5):
        ladder.append(("lr_mult", 0.3, "learning rate x0.3"))
    if code == 7 or code == 1:
        ladder.append(("micro_tokens", max(1024, a.micro_tokens // 2), "micro-batch halved"))
        ladder.append(("micro_tokens", max(512, a.micro_tokens // 4), "micro-batch / 4"))
    if code == 6:
        ladder.append(("skip_job_checks", True, "self-checks skipped"))
    used = j.setdefault("ladder_used", [])
    for k, v, desc in ladder:
        if desc not in used:
            used.append(desc)
            j["overrides"][k] = v
            return desc
    return None


def schedule(a, jobs: List[Dict], p1, data, exp_dir, attn, bpe_dir, t_end, out_root) -> None:
    slots = max(1, n_gpus())
    log(f"   [queue] {len(jobs)} jobs on {slots} slot(s) "
        f"({'GPU' if n_gpus() else 'CPU'}); notebook budget ends at "
        f"{time.strftime('%H:%M', time.localtime(t_end))}")
    by = {j["job"]: j for j in jobs}
    running: Dict[int, Dict] = {}
    last_refresh = 0.0
    reserve = 60 * a.reserve_min
    while True:
        # ---- launch ---------------------------------------------------------------------
        for slot in range(slots):
            while slot not in running:
                ready = [j for j in jobs if j["state"] == "pending" and all(
                    by[d]["state"] in ("done", "failed", "skipped") for d in j["depends"]
                    if d in by)]
                if not ready:
                    break
                j = ready[0]
                remaining = t_end - time.time() - reserve
                n_pending = sum(1 for x in jobs if x["state"] == "pending")
                if j.get("is_teacher") and any(x.get("student") for x in jobs):
                    budget = a.teacher_frac * remaining
                else:
                    budget = remaining * slots / max(slots, n_pending + len(running))
                    budget = min(budget, remaining)
                run_dir = os.path.join(exp_dir, j["job"])
                if budget < 8 * 60:
                    j["state"] = "skipped"
                    os.makedirs(os.path.join(run_dir, "logs"), exist_ok=True)
                    cfg = job_config(a, j, p1, data, exp_dir, slot, budget, attn, bpe_dir, by)
                    write_json(os.path.join(run_dir, "logs", "run_config.json"), cfg)
                    write_json(os.path.join(run_dir, "status.json"),
                               {"state": "skipped_time", "time": now_str(), "exit_code": 2,
                                "reason": f"only {budget / 60:.0f} min left in the notebook budget"})
                    log(f"   [queue] SKIP {j['job']}: only {budget / 60:.0f} min left")
                    continue
                j["attempt"] += 1
                cfg = job_config(a, j, p1, data, exp_dir, slot, budget, attn, bpe_dir, by)
                os.makedirs(os.path.join(run_dir, "logs"), exist_ok=True)
                write_json(os.path.join(run_dir, "logs", "run_config.json"), cfg)
                st = read_json(os.path.join(run_dir, "status.json"), {}) or {}
                write_json(os.path.join(run_dir, "status.json"),
                           {**st, "state": "preflight", "time": now_str(), "reason": "",
                            "attempt": j["attempt"]})
                mode = "train"
                env = dict(os.environ, PYTHONPATH=CODE_DIR, PYTHONUNBUFFERED="1")
                if n_gpus():
                    env["CUDA_VISIBLE_DEVICES"] = str(slot)
                cmd = [sys.executable, os.path.join(CODE_DIR, "pcr_train.py"), "--run-dir", run_dir,
                       "--mode", mode]
                log(f"   [queue] START {j['job']} ({cfg['technique']}, attempt {j['attempt']}) on "
                    f"slot {slot}, budget {budget / 60:.0f} min")
                j["proc"] = subprocess.Popen(cmd, env=env, cwd=CODE_DIR)
                j["state"] = "running"
                j["slot"] = slot
                running[slot] = j
        if not running:
            break
        # ---- poll -----------------------------------------------------------------------
        time.sleep(5)
        for slot, j in list(running.items()):
            code = j["proc"].poll()
            if code is None:
                continue
            del running[slot]
            run_dir = os.path.join(exp_dir, j["job"])
            st = read_json(os.path.join(run_dir, "status.json"), {}) or {}
            if code == 0 and st.get("state") in ("done", "degenerate", "smoke_ok"):
                j["state"] = "done"
                log(f"   [queue] END {j['job']}: {st.get('state')} ({st.get('reason', '')})")
                continue
            reason = f"exit {code} ({JOB_EXIT.get(code, '?')}): {st.get('reason', '')}"
            log(f"   [queue] FAIL {j['job']}: {reason[:300]}")
            reducible = False
            remaining = t_end - time.time() - reserve
            desc = None
            if j["attempt"] <= a.max_retries and remaining > 15 * 60:
                desc = next_attempt(a, j, code, reducible)
            if desc:
                archive_attempt(run_dir, j["attempt"])
                write_json(os.path.join(run_dir, "status.json"),
                           {"state": "pending", "time": now_str(),
                            "reason": f"retry after {reason[:300]} -> {desc}"})
                j["state"] = "pending"
                log(f"   [queue] RETRY {j['job']} with {desc}")
            else:
                j["state"] = "failed"
                write_json(os.path.join(run_dir, "status.json"),
                           {**st, "state": "skipped_time" if code == 2 else "failed",
                            "exit_code": code, "time": now_str(), "reason": reason[:2000]})
        if time.time() - last_refresh > 60:
            try:
                write_index(a, out_root, exp_dir, jobs)
            except Exception as e:
                log(f"   [index] refresh failed: {e}")
            last_refresh = time.time()


# =============================================================================================
# main
# =============================================================================================
def finalize(a, out_root, exp_dir, jobs, extra=None) -> int:
    from aggregate import aggregate, error_analysis, ensemble
    from select_best import plan_uploads, select_exp
    rows = []
    try:
        from aggregate import write_exp_csvs
        rows = write_exp_csvs(exp_dir, [planned_row(a, j) for j in jobs])
        if a.exp in COUNTED or a.exp == "E7":
            select_exp(exp_dir, rows, log)
    except Exception as e:
        log(f"   [final] BEST selection failed: {e}\n{traceback.format_exc()[-1500:]}")
    try:
        allrows = aggregate(out_root, default_input_roots(), log)
        log(f"   [final] ALL_RESULTS.csv: {len(allrows)} rows")
        plan = plan_uploads(allrows, os.path.join(out_root, "UPLOAD_PLAN.csv"), log)
        if a.exp == "E8":
            ensemble(allrows, out_root, log)
            error_analysis(allrows, out_root, None, log)
    except Exception as e:
        log(f"   [final] aggregation failed: {e}\n{traceback.format_exc()[-1500:]}")
    write_index(a, out_root, exp_dir, jobs, extra)
    done = [r for r in rows if r.get("status") == "DONE"]
    write_json(os.path.join(exp_dir, "summary.json"),
               {"experiment": a.exp, "finished": now_str(), "rows": rows,
                "output_gb": dir_size_gb(out_root)})
    log(f"   [final] {len(done)}/{len(rows)} jobs DONE; output {dir_size_gb(out_root):.2f} GB")
    return 0 if (done or a.exp == "E8") else 1


def main() -> int:
    global LOG
    a = build_parser().parse_args()
    t_start = time.time()
    t_end = t_start + 3600 * a.time_budget_h
    out_root = a.out_root
    exp_dir = os.path.join(out_root, EXPERIMENTS[a.exp]["folder"])
    os.makedirs(exp_dir, exist_ok=True)
    LOG = os.path.join(exp_dir, "orchestrator.log")
    log(f"=== Phase 7 {a.exp}: {EXPERIMENTS[a.exp]['title']} | {now_str()} | budget "
        f"{a.time_budget_h} h | {n_gpus()} GPU(s) | free disk {free_gb(out_root):.0f} GB ===")
    if a.exp == "E8":
        jobs: List[Dict] = []
        write_index(a, out_root, exp_dir, jobs)
        return finalize(a, out_root, exp_dir, jobs)

    # ---- 1. Phase-1 models + job list + PENDING rows ----------------------------------------
    p1 = find_phase1_models(default_input_roots())
    for m in MODELS:
        log(f"   [phase1] {m} {MODELS[m]['arch']:<16} -> {p1.get(m, 'NOT FOUND')}")
    jobs = make_jobs(a, p1)
    need = sorted({j["model"] for j in jobs})
    missing = [m for m in need if m not in p1]
    write_index(a, out_root, exp_dir, jobs)
    if missing:
        msg = (f"Phase-1 weights missing for {missing} ({[MODELS[m]['arch'] for m in missing]}). "
               "Attach the four Phase-1 datasets (final/model.pt + config.json + bpe_model.json).")
        log("!" * 90 + f"\n   {msg}\n" + "!" * 90)
        jobs = [j for j in jobs if j["model"] not in missing]
        write_json(os.path.join(out_root, "BLOCKED.json"), {"reason": msg, "time": now_str()})
        if not jobs:
            write_index(a, out_root, exp_dir, [], [f"**BLOCKED:** {msg}"])
            return 2
    # bpe_model.json: next to a Phase-1 model.pt, or anywhere in any attached dataset
    # (e.g. a separate "bpe_model" dataset).
    from ner_common import find_files
    bpe_dir = None
    for d in list(p1.values()):
        if os.path.exists(os.path.join(d, "bpe_model.json")):
            bpe_dir = d
            break
    if bpe_dir is None:
        hits = find_files("bpe_model.json", default_input_roots(), 50)
        hits = [h for h in hits if os.path.isfile(h)]
        if hits:
            bpe_dir = os.path.dirname(hits[0])
            log(f"   [bpe] using {hits[0]} ({len(hits)} found)")
    if bpe_dir is None:
        msg = ("bpe_model.json not found in any attached dataset (searched "
               f"{default_input_roots()})")
        write_json(os.path.join(out_root, "BLOCKED.json"), {"reason": msg})
        write_index(a, out_root, exp_dir, jobs, [f"**BLOCKED:** {msg}"])
        return 2

    attn = "causal"
    try:
        data = prepare_data(a, out_root, bpe_dir)
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        log("!" * 90 + f"\n   DATA STEP FAILED: {msg}\n" + "!" * 90)
        log(traceback.format_exc()[-2000:])
        write_json(os.path.join(out_root, "BLOCKED.json"), {"reason": msg, "time": now_str()})
        for j in jobs:
            rd = os.path.join(exp_dir, j["job"])
            os.makedirs(os.path.join(rd, "logs"), exist_ok=True)
            write_json(os.path.join(rd, "status.json"), {"state": "failed", "reason": msg,
                                                         "time": now_str()})
        write_index(a, out_root, exp_dir, jobs, [f"**BLOCKED (data):** {msg}"])
        return 2

    # ---- 4. resume / queue ---------------------------------------------------------------
    for j in jobs:
        rd = os.path.join(exp_dir, j["job"])
        if a.resume == "auto":
            if os.path.exists(os.path.join(rd, "DONE.json")):
                j["state"] = "done"
                log(f"   [resume] {j['job']} already DONE -> skipped")
                continue
            src = None if os.path.exists(os.path.join(rd, "checkpoints", "last.pt")) else \
                copy_from_inputs(a, exp_dir, j["job"])
            if src:
                if os.path.exists(os.path.join(rd, "DONE.json")):
                    j["state"] = "done"
                    log(f"   [resume] {j['job']} finished in an earlier session ({src}) -> copied")
                else:
                    log(f"   [resume] {j['job']} resumes from {src}")
        os.makedirs(os.path.join(rd, "logs"), exist_ok=True)
        if j["state"] == "pending" and not os.path.exists(os.path.join(rd, "status.json")):
            write_json(os.path.join(rd, "status.json"), {"state": "pending", "time": now_str()})
    write_index(a, out_root, exp_dir, jobs)
    try:
        schedule(a, jobs, p1, data, exp_dir, attn, bpe_dir, t_end, out_root)
    except Exception as e:
        log(f"   [queue] scheduler error: {e}\n{traceback.format_exc()[-2000:]}")
        for j in jobs:
            if j.get("proc") is not None and j["proc"].poll() is None:
                j["proc"].terminate()
    return finalize(a, out_root, exp_dir, jobs)


if __name__ == "__main__":
    try:
        code = main()
    except SystemExit:
        raise
    except Exception as e:
        print(f"run_experiment.py crashed: {e}\n{traceback.format_exc()}", flush=True)
        code = 1
    sys.exit(code)


In [ ]:
%%writefile /kaggle/working/phase7/selftest.py
"""
selftest.py -- Phase-7 CPU tests with tiny random-initialised models of the four Phase-1
architectures and a tiny BPE trained on synthetic legal-ish text (no real weights or data).

    python selftest.py [--out SELFTEST.json]

  1 metrics     Micro-P/R/F1@7 vs hand values (gold sets longer / shorter than 7), top-k order and
                ties, BM25 ranks an identical case first, fusion keeps masked entries masked,
                submission validator (good + 6 kinds of bad zips)
  2 data        loader (list / dict-keyed pools, zero-padded ids), relevant ids mapped into the
                split's OWN pool, self-retrieval masked, chunk cache bounds, BM25 tuned on dev
  3 model       for all 4 architectures x 8 techniques: trainable set, identity at init
                (LoRMA / LoRA / IA3 / full / partial exact), GradCache gradient == exact gradient,
                gradients reach the trainable tensors, padding invariance, virtual positions
                excluded from pooling, NF4 dequantisation error small
  4 overfit     4 pairs: contrastive loss falls below 30% of its start (subprocess)
  5 resume      train -> kill (137) at step 3 -> resume at step 3 -> DONE + valid submission
"""
import argparse
import copy
import json
import os
import random
import subprocess
import sys
import tempfile
import time
import traceback
from typing import Dict

import numpy as np
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

from cjpe_common import MODELS, write_json  # noqa: E402

TINY = {
    "M1": ("GPT2StyleLM", {"vocab_size": 600, "n_positions": 1024, "n_embd": 32, "n_layer": 4,
                           "n_head": 4, "mlp_ratio": 4}),
    "M2": ("LlamaStyleLM", {"vocab_size": 600, "max_position_embeddings": 1024,
                            "hidden_size": 32, "intermediate_size": 64, "num_hidden_layers": 4,
                            "num_attention_heads": 4, "num_key_value_heads": 2,
                            "sliding_window": None}),
    "M3": ("LlamaStyleLM", {"vocab_size": 600, "max_position_embeddings": 1024,
                            "hidden_size": 32, "intermediate_size": 64, "num_hidden_layers": 4,
                            "num_attention_heads": 4, "num_key_value_heads": 2,
                            "sliding_window": 8}),
    "M4": ("BloomStyleLM", {"vocab_size": 600, "hidden_size": 32, "n_layer": 4, "n_head": 4,
                            "mlp_ratio": 4}),
}
WORDS = ("the court held that appellant respondent section act order petition dismissed "
         "allowed appeal judgment high supreme evidence witness learned counsel state "
         "accused conviction sentence bail facts issue reasons therefore accordingly").split()


# =============================================================================================
# fixtures
# =============================================================================================
def synth_text(rng: random.Random, n_words: int, label) -> str:
    """label: list of statute ids (Phase 5); an int is treated as [int]."""
    labs = label if isinstance(label, (list, tuple)) else [int(label)]
    words = []
    for k in range(n_words):
        w = rng.choice(WORDS)
        if rng.random() < 0.08:
            w = w.upper()
        words.append(w)
        if rng.random() < 0.1:
            words.append(rng.choice([",", ".", ";", "(1)", "12/2003", "\n", "  "]))
    cue = {1: "murder penal", 2: "cheating fraud", 3: "dowry cruelty", 4: "bail custody",
           5: "property land", 6: "contract breach"}
    tail = "".join(f" the accused was charged under section {x} for {cue.get(x, 'offence')} ."
                   for x in labs)
    return " ".join(words) + "\n\nORDER\n" + tail * 3


def write_tokenizer(d: str, seed: int = 0) -> str:
    from bpe_tokenizer import BPETokenizer, count_pretokens, train_bpe
    rng = random.Random(seed)
    texts = [synth_text(rng, 200, [i % 6 + 1]) for i in range(40)]
    merges, _ = train_bpe(count_pretokens(texts, log=lambda *a: None), 3 + 256 + 300,
                          log=lambda *a: None)
    tok = BPETokenizer(merges)
    tok.save_pretrained(d)
    return d


def write_phase1(root: str, mid: str, seed: int = 0, tok_dir: str = None) -> str:
    from phase1_models import MODEL_CLASSES
    cls_name, conf = TINY[mid]
    d = os.path.join(root, f"p1_{mid}", "final")
    os.makedirs(d, exist_ok=True)
    torch.manual_seed(seed)
    cls, ccls = MODEL_CLASSES[cls_name]
    torch.save(cls(ccls(**conf)).state_dict(), os.path.join(d, "model.pt"))
    with open(os.path.join(d, "config.json"), "w") as f:
        json.dump({"model_class": cls_name, "arch": MODELS[mid]["arch"], "config": conf}, f)
    if tok_dir:
        import shutil
        for fn in os.listdir(tok_dir):
            shutil.copy2(os.path.join(tok_dir, fn), os.path.join(d, fn))
    return d


def write_dataset(root: str, n_q=(30, 10, 10), seed=0, shape="list") -> str:
    """Phase_7_PCR/{Train,Dev,Test}: topic-structured cases; relevant = same topic."""
    rng = random.Random(seed)
    base = os.path.join(root, "Phase_7_PCR")
    nid = [1000]

    def new_id():
        nid[0] += rng.randint(1, 9)
        return f"{nid[0]:010d}"

    def case(topic):
        sents = []
        for _ in range(rng.choice([3, 8, 20])):
            sents.append(synth_text(rng, rng.randint(8, 25), [topic % 6 + 1]).split("\n")[0]
                         + f" the doctrine of topic{topic} and rule {100 + topic} applied .")
        return sents
    for split, nq in zip(("train", "dev", "test"), n_q):
        os.makedirs(os.path.join(base, split.capitalize()), exist_ok=True)
        pool, queries = [], []
        for k in range(nq):
            t = rng.randint(0, 11)
            rel = [new_id() for _ in range(rng.randint(1, 4))]
            pool += [{"id": r, "text": case(t)} for r in rel]
            q = {"id": new_id(), "text": case(t)}
            if split != "test":
                q["relevant_candidates"] = rel if k % 5 else [r.lstrip("0") for r in rel]
            queries.append(q)
        pool += [{"id": new_id(), "text": case(rng.randint(12, 20))} for _ in range(2 * nq)]
        if split == "dev":                                    # a query inside its own pool
            pool.append({"id": queries[0]["id"], "text": queries[0]["text"]})
        rng.shuffle(pool)
        if shape == "dict":
            pool = {p["id"]: p["text"] for p in pool}
        for name, obj in ((f"{split}.json", queries), (f"{split}_candidates.json", pool)):
            with open(os.path.join(base, split.capitalize(), name), "w") as f:
                json.dump(obj, f)
    return base


def _data(root, tok_dir, name, **kw):
    from pcr_data import build_cache, load_all, load_cache
    base = write_dataset(os.path.join(root, name), **kw)
    data, sch, info = load_all([os.path.dirname(base)], log=lambda *a: None)
    p = build_cache(data, tok_dir, os.path.join(root, name + "_cache"), name, 48, 4, 1,
                    log=lambda *a: None)
    return load_cache(p), p, data, info


# =============================================================================================
def t1_metrics(tok_dir: str) -> Dict:
    import zipfile
    from pcr_ml import BM25, fuse, mask_self, micro_prf_at_k, topk
    from validate_submission import validate
    m = micro_prf_at_k([[1, 2, 3, 4, 5, 6, 7], [9, 8, 7, 6, 5, 4, 3]],
                       [{1, 2, 99}, set(range(3, 13))])
    # tp = 2 + 7 = 9; pred = 14; gold = 3 + 10 = 13
    assert m["tp"] == 9 and abs(m["micro_p"] - 9 / 14) < 1e-12 and abs(m["micro_r"] - 9 / 13) < 1e-12
    S = np.array([[0.1, 0.5, 0.5, 0.9, -np.inf]], np.float32)
    assert topk(S, 3).tolist() == [[3, 1, 2]]
    docs = ["the appellant was convicted under section 302 for murder",
            "contract breach damages awarded to plaintiff", "bail granted to accused under 437"]
    bm = BM25(docs, 1.2, 0.75, 2)
    assert bm.score(["contract breach damages awarded"]).argmax() == 1
    Sm = mask_self(np.ones((1, 3), np.float32), ["b"], ["a", "b", "c"])
    F = fuse(Sm, np.arange(3, dtype=np.float32)[None], 0.5)
    assert not np.isfinite(F[0, 1]) and np.isfinite(F[0, [0, 2]]).all()
    tmp = tempfile.mkdtemp()
    ids, pool = ["q1", "q2"], [f"c{i}" for i in range(10)]

    def mk(name, obj, entry="prediction.json"):
        p = os.path.join(tmp, name)
        with zipfile.ZipFile(p, "w") as z:
            z.writestr(entry, json.dumps(obj))
        return validate(p, ids, pool)
    good = {"q1": pool[:7], "q2": pool[1:8]}
    assert mk("good.zip", good)["ok"]
    bad = {"short": {"q1": pool[:6], "q2": pool[1:8]},
           "dup": {"q1": pool[:6] + [pool[0]], "q2": pool[1:8]},
           "foreign": {"q1": pool[:6] + ["zz"], "q2": pool[1:8]},
           "missing": {"q1": pool[:7]},
           "list_format": [{"id": "q1", "candidates": pool[:7]}]}
    for k, v in bad.items():
        assert not mk(f"{k}.zip", v)["ok"], k
    assert not mk("entry.zip", good, "pred.json")["ok"]
    return {"micro": m, "validator_bad_cases": len(bad) + 1}


def t2_data(root: str, tok_dir: str) -> Dict:
    cache, path, data, info = _data(root, tok_dir, "ds_dict", shape="dict")
    a = info["audit"]
    for s in ("train", "dev", "test"):
        assert a[s]["relevant_missing_from_pool"] == 0, a[s]
    assert a["dev"]["queries_in_own_pool"] == 1
    sp = cache["splits"]
    for s in ("train", "dev"):
        for q in sp[s]["q"]:
            assert q["rel"] and all(0 <= j < len(sp[s]["pool"]) for j in q["rel"])
            assert all(sp[s]["pool"][j]["id"] != q["id"] for j in q["rel"])
    for s in sp:
        for x in sp[s]["q"] + sp[s]["pool"]:
            assert 1 <= len(x["chunks"]) <= 4 and all(1 <= len(c) <= 48 and c[-1] == 0
                                                     for c in x["chunks"])
    bm = cache["bm25"]
    qid, pool = sp["dev"]["q"][0]["id"], [x["id"] for x in sp["dev"]["pool"]]
    assert not np.isfinite(bm["dev"][0, pool.index(qid)]), "self not masked"
    assert bm["dev_micro_f1"] > 0.2, bm["dev_micro_f1"]
    assert bm["train_top"].shape[0] == len(sp["train"]["q"])
    return {"bm25_dev_f1": bm["dev_micro_f1"], "params": bm["params"]}


def t3_model(root: str) -> Dict:
    from cjpe_train import make_optimizer, make_scaler
    from pcr_model import NEW_TECHNIQUES, apply_pcr_technique, build_pcr_model, encode_chunks
    from pcr_train import ZERO_INIT, Stepper, build_data, job_checks
    tok = os.path.join(root, "tok")
    cache, _p, _d, _i = _data(root, tok, "ds_model")
    out = {}
    nullctx = lambda: torch.autocast("cpu", enabled=False)  # noqa: E731
    for mid in MODELS:
        p1 = write_phase1(root, mid, seed=3)
        for tech in ("full_ft", "partial_ft", "lorma", "lora", "qlora", "prefix", "prompt", "ia3"):
            torch.manual_seed(0)
            model, _ = build_pcr_model(p1)
            base = copy.deepcopy(model)
            cfg = {"lr": 1e-3, "n_virtual": 3, "partial_top_k": 2, "seed": 1, "batch_pairs": 4,
                   "hard_negs": 2, "k_train": 2, "micro_tokens": 4096, "chunk_tokens": 48}
            groups, techo = apply_pcr_technique(model, tech, cfg, lambda *a: None, [5, 6, 7])
            if tech in ("qlora",):                                   # NF4 error small
                a_ = encode_chunks(base, cache["splits"]["train"]["q"][0]["chunks"], "cpu",
                                   nullctx, 10 ** 6)
                b_ = encode_chunks(model, cache["splits"]["train"]["q"][0]["chunks"], "cpu",
                                   nullctx, 10 ** 6)
                assert float((a_ - b_).abs().max()) < 0.25, f"{mid}/qlora NF4 error"
            ctxo = type("C", (), {"device": torch.device("cpu"), "log": print})()
            d = build_data(cfg, cache)
            st = Stepper(ctxo, model, make_optimizer(groups), None, make_scaler(False), techo,
                         nullctx, cfg, cache, d)
            st.ctx_cache = cache
            res = job_checks(base if tech in ZERO_INIT else None, model, st, cfg, d,
                             torch.device("cpu"), nullctx)
            assert not res["critical_failed"], (mid, tech, res)
            bad_adv = [c for c in res["checks"] if c["verdict"] == "fail"]
            assert not bad_adv, (mid, tech, bad_adv)
            if tech in ("prefix", "prompt"):
                assert model.n_virtual == 3
            out[f"{mid}/{tech}"] = {c["name"]: round(c.get("value", 0) or 0, 6) for c in res["checks"]}
    return {"n": len(out), "sample": dict(list(out.items())[:3])}


def _job_cfg(root, tok_dir, cache_path, tech, run, **kw):
    p1 = write_phase1(root, "M2", tok_dir=tok_dir)
    cfg = {"experiment": "T", "job": os.path.basename(run), "model": "M2", "phase1_dir": p1,
           "bpe_dir": tok_dir, "technique": tech, "doc_repr": "hier", "pooling": "mean",
           "attn_mode": "causal", "chunk_tokens": 48, "k_train": 2, "k_infer": 4,
           "batch_pairs": 8, "hard_negs": 2, "lr": 3e-3, "head_lr": 3e-3, "micro_tokens": 4096,
           "seed": 1, "budget_s": 900, "cache_path": cache_path, "skip_job_checks": False,
           "n_virtual": 4, "epochs_max": 2, "n_dev": 10, "n_test": 10}
    cfg.update(kw)
    write_json(os.path.join(run, "logs", "run_config.json"), cfg)
    return cfg


def t4_overfit(root: str, tok_dir: str) -> Dict:
    _c, path, _d, _i = _data(root, tok_dir, "ds_job")
    run = os.path.join(root, "run_overfit")
    _job_cfg(root, tok_dir, path, "full_ft", run)
    env = dict(os.environ, PYTHONPATH=HERE, CUDA_VISIBLE_DEVICES="")
    p = subprocess.run([sys.executable, os.path.join(HERE, "pcr_train.py"), "--run-dir", run,
                        "--mode", "overfit"], env=env, capture_output=True, text=True, timeout=900)
    f = os.path.join(run, "logs", "overfit.json")
    assert os.path.exists(f), (p.stdout + p.stderr)[-2500:]
    ov = json.load(open(f))
    assert ov["passed"], ov
    return ov


def t5_resume(root: str, tok_dir: str) -> Dict:
    from pcr_data import load_cache
    from validate_submission import validate
    path = json.load(open(os.path.join(root, "run_overfit", "logs", "run_config.json")))["cache_path"]
    run = os.path.join(root, "run_resume")
    cfg = _job_cfg(root, tok_dir, path, "lorma", run, kill_at_step=3, lr=1e-3,
                   batch_pairs=4)
    env = dict(os.environ, PYTHONPATH=HERE, CUDA_VISIBLE_DEVICES="")
    cmd = [sys.executable, os.path.join(HERE, "pcr_train.py"), "--run-dir", run]
    p = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=900)
    assert p.returncode == 137, (p.returncode, (p.stdout + p.stderr)[-2000:])
    assert os.path.exists(os.path.join(run, "checkpoints", "last.pt"))
    cfg["kill_at_step"] = -1
    write_json(os.path.join(run, "logs", "run_config.json"), cfg)
    p = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=900)
    log = open(os.path.join(run, "logs", "console.log")).read()
    assert p.returncode == 0, log[-2500:]
    assert "[resume] from step 3" in log
    assert os.path.exists(os.path.join(run, "DONE.json"))
    sp = load_cache(path)["splits"]["test"]
    v = validate(os.path.join(run, "submission", "submission.zip"), [x["id"] for x in sp["q"]],
                 [x["id"] for x in sp["pool"]])
    assert v["ok"], v
    met = json.load(open(os.path.join(run, "metrics.json")))
    assert met["dev_micro_f1"] is not None and met["dev_bm25_f1"] is not None
    for f in ("dev/dev_predictions.jsonl", "dev/fusion_curve.csv", "dev/dev_report.json",
              "dev/scores.npz"):
        assert os.path.exists(os.path.join(run, f)), f
    return {"resumed_at_step": 3, "dev_micro_f1": met["dev_micro_f1"],
            "dev_dense_f1": met["dev_dense_f1"], "dev_bm25_f1": met["dev_bm25_f1"]}


def run_all(log=print) -> Dict:
    res = {"tests": {}, "failed": [], "started": time.strftime("%Y-%m-%d %H:%M:%S")}
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
    with tempfile.TemporaryDirectory() as root:
        tok_dir = write_tokenizer(os.path.join(root, "tok"))
        tests = [("1_metrics", lambda: t1_metrics(tok_dir)),
                 ("2_data_pools_cache_bm25", lambda: t2_data(root, tok_dir)),
                 ("3_model_techniques_gradcache", lambda: t3_model(root)),
                 ("4_overfit", lambda: t4_overfit(root, tok_dir)),
                 ("5_kill_resume", lambda: t5_resume(root, tok_dir))]
        only = os.environ.get("SELFTEST_ONLY")
        for name, fn in tests:
            if only and not any(name.startswith(o) for o in only.split(",")):
                continue
            t0 = time.time()
            try:
                res["tests"][name] = {"result": fn(), "seconds": round(time.time() - t0, 1)}
                log(f"   PASS {name} ({time.time() - t0:.1f}s)")
            except Exception as e:
                res["failed"].append(name)
                res["tests"][name] = {"error": str(e)[:2000],
                                      "trace": traceback.format_exc()[-3000:]}
                log(f"   FAIL {name}: {str(e)[:500]}\n{traceback.format_exc()[-1500:]}")
    res["ok"] = not res["failed"]
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=None)
    a = ap.parse_args()
    res = run_all()
    if a.out:
        write_json(a.out, res)
    print(json.dumps({"ok": res["ok"], "failed": res["failed"]}))
    sys.exit(0 if res["ok"] else 1)


if __name__ == "__main__":
    main()


In [ ]:
# every module must be byte-identical to the generated version (same code in all notebooks)
try:
    bad = [rel for rel, h in EXPECTED_MD5.items()
           if hashlib.md5(open(os.path.join(CODE, rel), 'rb').read()).hexdigest() != h]
    if bad:
        print('!' * 80 + f'\n!!! modules differ from the generated version: {bad}\n' + '!' * 80)
    else:
        print(f'all {len(EXPECTED_MD5)} modules verified')
except Exception as e:
    print('module check failed:', e)


In [ ]:
# what is attached (never raises)
try:
    sys.path.insert(0, CODE)
    from cjpe_common import default_input_roots, find_files, find_phase1_models
    from cjpe_data import get_hf_token
    print('Phase-1 weights:', json.dumps(find_phase1_models(default_input_roots()), indent=1))
    for n in ('train.json', 'train_candidates.json', 'dev.json', 'dev_candidates.json', 'test.json', 'test_candidates.json'):
        print(n, '->', find_files(n, default_input_roots())[:4])
    t = get_hf_token()
    print('HF secret found:', 'yes (length %d)' % len(t) if t else 'no')
    print('earlier phase7_out inputs:', glob.glob('/kaggle/input/**/phase7_out', recursive=True)[:6])
except Exception as e:
    print('discovery failed:', e); traceback.print_exc()


In [ ]:
# the run (streams its log; a non-zero exit code is PRINTED, never raised -- rule R1)
# to split the four models over two accounts: add e.g. '--models M4,M2' here and
# '--models M3,M1' on the other account (then attach both outputs to E8).
ARGS = "--exp E3 --models M4,M2,M3,M1 --gpu-mode jobs --resume auto --time-budget-h 3.5"
RC = None
try:
    p = subprocess.Popen([sys.executable, '-u', os.path.join(CODE, 'run_experiment.py')] + ARGS.split(),
                         cwd=CODE, env=dict(os.environ, PYTHONPATH=CODE, PYTHONUNBUFFERED='1'),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    RC = p.wait()
except Exception as e:
    print('launch failed:', e); traceback.print_exc()
print(f'\n===== run_experiment.py exit code: {RC} '
      '(0 ok, 1 nothing finished, 2 data / weights missing) =====')


In [ ]:
# R7: final verification + aggregation (never raises)
FOLDER = "E3_partial_ft"
try:
    sys.path.insert(0, CODE)
    from cjpe_common import read_csv
    from validate_submission import validate
    ld = lambda f: json.load(open(f'{OUT}/{f}')) if os.path.exists(f'{OUT}/{f}') else None
    test_ids, pool = ld('test_ids.json'), ld('test_pool_ids.json')
    rows = read_csv(f'{OUT}/{FOLDER}/results_{FOLDER}.csv')
    print(f"{'job':<34}{'status':<13}{'F1@7':>8}{'dense':>8}{'BM25':>8}{'w':>5}{'frac':>6}{'min':>7}  check")
    for r in rows:
        z = f"{OUT}/{FOLDER}/{r['job']}/submission/submission.zip"
        verdict = ''
        if r['status'] in ('DONE', 'DEGENERATE'):
            v = validate(z, test_ids, pool) if (test_ids and os.path.exists(z)) else {'ok': False, 'errors': ['zip missing']}
            files_ok = all(os.path.exists(f"{OUT}/{FOLDER}/{r['job']}/{f}")
                           for f in ('dev/dev_predictions.jsonl', 'dev/fusion_curve.csv', 'dev/dev_report.json', 'dev/scores.npz'))
            verdict = 'PASS' if (v['ok'] and files_ok and r.get('dev_score')) else f"FAIL {v.get('errors')}"
        g = lambda k: f'{float(r[k]):.4f}' if r.get(k) else ''
        print(f"{r['job']:<34}{r['status']:<13}{g('dev_micro_f1'):>8}{g('dev_dense_f1'):>8}{g('dev_bm25_f1'):>8}{(r.get('dense_weight') or '')[:4]:>5}"
              f"{(r.get('train_fraction') or '')[:4]:>6}{(r.get('train_minutes') or '')[:5]:>7}  {verdict} {r.get('fail_reason', '')[:80]}")
    best = f'{OUT}/{FOLDER}/BEST/submission.zip'
    if os.path.exists(best):
        print('\nBEST:', json.dumps({k: v for k, v in json.load(open(f'{OUT}/{FOLDER}/BEST/SELECTION.json')).items() if k != 'ranking'}))
        print('UPLOAD THIS:', best)
    elif FOLDER != 'E8_report':
        print('!' * 80 + '\n!!! NO MODEL FINISHED in this notebook. Read the fail_reason column above and\n'
              f'!!! {OUT}/INDEX.md / BLOCKED.json; the logs are in {OUT}/{FOLDER}/<job>/logs/console.log\n' + '!' * 80)
    for f in ('INDEX.md', 'UPLOAD_PLAN.csv', 'BLOCKED.json'):
        if os.path.exists(f'{OUT}/{f}'):
            print(f'\n----- {f} -----'); print(open(f'{OUT}/{f}').read()[:6000])
    tot = 0
    for dp, dn, fn in sorted(os.walk(OUT)):
        depth = dp.count(os.sep) - OUT.count(os.sep)
        size = sum(os.path.getsize(os.path.join(dp, f)) for f in fn)
        tot += size
        if depth <= 2:
            print('  ' * depth + os.path.basename(dp) + f'/  ({size / 1e6:.1f} MB in files here)')
    print(f'total output {tot / 1e9:.2f} GB')
except Exception as e:
    print('final check failed:', e); traceback.print_exc()
print('\nREMINDER: Save Version -> Save & Run All (Commit) keeps /kaggle/working/phase7_out as this '
      'notebook\'s output. Download BEST/submission.zip and results_*.csv from the Output tab.')


Upload `phase7_out/E3_partial_ft/BEST/submission.zip` to Codabench (or any `<job>/submission/submission.zip`). After E1-E6 are done, run `E8_report` with all outputs attached: it writes `UPLOAD_PLAN.csv` (six distinct uploads, all four Phase-1 models).